In [ ]:
%%writefile /kaggle/working/run_lexical.py
"""
Canonical Configuration, Constants & Path Resolution
====================================================
Master canonical label ordering, provider families, simplex helpers,
and Kaggle filesystem asset discovery.
"""

import hashlib
import json
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np

# Canonical 18-Class Labels (Strict order from NTL two-stage LoRA part two)
LABELS = [
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "claude-fable-5",
    "qwen3.8-max",
    "glm-5.3",
    "glm-5.3-flash",
    "kimi-k3",
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

# Master list of highest-error and most confused classes requiring loss upweighting:
# 1: gpt-5.6-luna (60.0% error, easily confused with astra/sol)
# 2: gpt-5.6-sol (75.0% error, heavily confused with astra)
# 3: gpt-5.6-terra (70.0% error, confused with luna/sol)
# 6: gemini-3.6-flash (45.0% error, confused with 3.1-pro)
# 7: gemini-3.7-flash (80.0% error, confused with 3.8-flash & 3.6-flash)
# 8: gemini-3.8-flash (65.0% error, confused with 3.7-flash & 3.6-flash)
# 13: claude-fable-5 (45.0% error, cross-family confusion)
# 14: qwen3.8-max (40.0% error, lowest pool count 4,209)
# 15: glm-5.3 (80.0% error, confused with glm-5.3-flash)
# 16: glm-5.3-flash (80.0% error, confused with glm-5.3)
# 17: kimi-k3 (80.0% error, straight-quote & Chinese frontier confusion)
HARD_CLASS_INDICES = [1, 2, 3, 6, 7, 8, 13, 14, 15, 16, 17]

# Extended specialist groups: includes the 4 provider families PLUS the cross-family
# Straight-Quote / Chinese Frontier confusion cluster (opus-4-8, fable-5, qwen3.8-max, glm-5.3, glm-5.3-flash, kimi-k3)
# and high-confusion sibling sub-clusters!
SPECIALIST_GROUPS = {
    "gpt": [0, 1, 2, 3, 4],
    "gpt_56_triad": [1, 2, 3],  # gpt-5.6-luna vs gpt-5.6-sol vs gpt-5.6-terra
    "gemini": [5, 6, 7, 8],
    "gemini_flash_triad": [6, 7, 8],  # gemini-3.6-flash vs 3.7-flash vs 3.8-flash
    "gemini_37_38_pair": [7, 8],  # Dedicated 1-vs-1 binary tie-breaker for gemini-3.7-flash vs gemini-3.8-flash!
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],  # Dedicated 1-vs-1 binary tie-breaker for glm-5.3 vs glm-5.3-flash!
    "glm_kimi_triad": [15, 16, 17],  # Resolves glm-5.3 <-> glm-5.3-flash <-> kimi-k3 confusion!
    "frontier_and_fable": [10, 13, 14, 15, 16, 17],  # resolves claude-fable-5/opus-4-8 <-> glm-5.3/flash <-> kimi-k3 <-> qwen3.8-max!
    "kimi_gpt55_qwen": [0, 14, 17],  # resolves kimi-k3 <-> gpt-5.5 <-> qwen3.8-max confusion!
}

PRIOR_SCALES = {
    "qwen3_14b_p2": 1.00,
    "qwen3_14b_inhouse": 1.00,
    "qwen25_14b_p2": 0.85,
    "qwen25_14b_inhouse": 0.85,
    "deberta_p2": 0.80,
    "deberta_inhouse": 0.80,
    "qwen3_14b_p1": 0.50,
    "instyle": 0.45,
    "qwen3_8b_claude": 0.30,
    "qwen3_8b_inhouse": 0.30,
}


def align_cols(arr: np.ndarray, src_labels: list) -> np.ndarray:
    """Aligns columns of array to canonical 18-class order."""
    src_list = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    idx = [src_list.index(l) for l in LABELS]
    if arr.ndim == 1:
        return arr[idx]
    elif arr.ndim == 2:
        if arr.shape[0] == len(src_list) and arr.shape[1] != len(src_list):
            return arr[idx, :]
        elif arr.shape[1] == len(src_list):
            return arr[:, idx]
    return arr[..., idx]


def calibrate_model_lp(
    lp: np.ndarray,
    model_name: str = "",
    min_std: float = 1.6,
    max_std: float = 4.2,
) -> np.ndarray:
    """
    1. Normalizes row-wise log-probability std into `[min_std, max_std]` so neither
       over-confident neural heads (LP_std=13.1) nor flat lexical heads (LP_std=0.40)
       distort ensemble vote shares or collapse under Sinkhorn balancing.
    2. Idempotency guard: if `lp` was already calibrated and clipped (e.g. loaded from
       local_eval_cache.npz), returns it without double-scaling.
    """
    if lp is None or len(lp) == 0:
        return lp
    out = np.asarray(lp, dtype=np.float64).copy()

    # Check if `out` is already a calibrated log-softmax matrix from cache
    row_max = np.max(out, axis=1)
    row_min = float(np.min(out))
    row_std = float(out.std(axis=1).mean())
    if np.all(row_max <= 1e-4) and row_min >= -16.5 and (1.4 <= row_std <= max_std + 0.1):
        return log_softmax(out, axis=1)

    if row_std > max_std and row_std > 1e-6:
        out = out * (max_std / row_std)
    elif row_std < min_std and row_std > 1e-6:
        out = out * (min_std / row_std)
    out = log_softmax(out, axis=1)
    out = np.maximum(out, -16.0)
    return log_softmax(out, axis=1)


def _text_hash(text: str) -> str:
    """
    Computes a canonical SHA-1 hash of normalized text for zero-leakage deduplication.

    Why Strict Normalization Matters:
    --------------------------------
    Subtle variations (different Unicode quotes, leading/trailing markdown asterisks,
    differing whitespace) could cause a training passage to evade exact string matching.
    We apply:
    1. Unicode NFKC normalization (replaces typographic quotes, fullwidth chars).
    2. Stripping enclosing markdown bold (**text** -> text).
    3. Collapsing all multi-whitespace and newlines into single spaces.
    This guarantees 0% exact or near-exact leakage between training and holdout!
    """
    s = unicodedata.normalize("NFKC", str(text or "")).strip()
    s = re.sub(r"^\*\*(.*)\*\*$", r"\1", s).strip()
    norm = " ".join(s.split())
    return hashlib.sha1(norm.encode("utf-8", errors="ignore")).hexdigest()


def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)


def resolve_dataset_paths(input_dir: Path | None = None) -> tuple[Path, Path, Path | None]:
    """Resolves train.csv, test.csv, and pool_v4.parquet across local and Kaggle environments."""
    if input_dir is None:
        input_dir = Path("/kaggle/input")
        if not input_dir.exists():
            input_dir = Path(".")

    candidates = [
        input_dir,
        input_dir / "competitions" / "name-that-llm",
        input_dir / "name-that-llm",
        Path("local_data"),
        Path("."),
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / "train.csv").exists() and train_path is None:
            train_path = c / "train.csv"
        if (c / "test.csv").exists() and test_path is None:
            test_path = c / "test.csv"

    if train_path is None:
        train_path = Path("local_data") / "train.csv"
    if test_path is None:
        test_path = Path("local_data") / "test.csv"

    pool_files = resolve_all_pool_files(input_dir)
    pool_file = pool_files[0] if pool_files else None

    return train_path, test_path, pool_file


def resolve_all_pool_files(input_dir: Path | None = None) -> list[Path]:
    """Discovers all pretraining/unlabeled pool parquet files in priority order.

    If the consolidated pool_120k.parquet exists, returns it directly as it already
    subsumes all individual pool sources (nlm_matched_clean, pool_v4, nlm_pool_v5, nlm_pool_v2).
    """
    if input_dir is None:
        input_dir = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")

    # If the consolidated 120k pool is present, use it directly to eliminate redundant I/O
    for p in input_dir.rglob("pool_120k.parquet"):
        return [p]

    pool_files: list[Path] = []
    pool_patterns = [
        "nlm_matched_clean.parquet",
        "pool_v4.parquet",
        "nlm_pool_v5.parquet",
        "nlm_pool_v2.parquet",
    ]
    for pat in pool_patterns:
        for p in input_dir.rglob(pat):
            if p not in pool_files:
                pool_files.append(p)
    return pool_files


def find_base_model(input_dir: Path, model_family: str, hidden_sz: int, num_layers: int) -> Path | None:
    """Discovers base model directories from Hugging Face config.json inspection."""
    for cfg_p in input_dir.rglob("config.json"):
        try:
            with open(cfg_p, "r", encoding="utf-8") as f:
                cfg = json.load(f)
            m_type = str(cfg.get("model_type", "")).lower()
            h_dim = cfg.get("hidden_size", 0)
            n_lay = cfg.get("num_hidden_layers", 0)
            if model_family.lower() in m_type and h_dim == hidden_sz and n_lay == num_layers:
                return cfg_p.parent
        except Exception:
            continue
    return None


def _inspect_checkpoint_dim_and_keep(ckpt_dir: Path) -> tuple[int, int, str]:
    """Returns (head_feature_dim, keep_layers, backbone) for checkpoint directory without full model loading."""
    dim = 0
    keep = 0
    backbone = ""
    meta_p = ckpt_dir / "meta.json"
    if meta_p.exists():
        try:
            with open(meta_p, "r", encoding="utf-8") as f:
                m = json.load(f)
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", 0))
            backbone = str(m.get("backbone", cfg.get("backbone", ""))).lower()
        except Exception:
            pass

    heads_p = ckpt_dir / "heads.safetensors"
    if heads_p.exists():
        try:
            from safetensors import safe_open

            with safe_open(str(heads_p), framework="pt", device="cpu") as f:
                for k in f.keys():
                    shape = f.get_tensor(k).shape
                    if len(shape) == 2 and (shape[0] in (18, 5, 4, 2) or shape[1] in (18, 5, 4, 2)):
                        dim = int(shape[1] if shape[0] in (18, 5, 4, 2) else shape[0])
                        break
        except Exception:
            pass
    elif (ckpt_dir / "heads.pt").exists():
        try:
            import torch

            sd = torch.load(ckpt_dir / "heads.pt", map_location="cpu", weights_only=False)
            if isinstance(sd, dict):
                for k, v in sd.items():
                    if hasattr(v, "shape") and len(v.shape) == 2 and (v.shape[0] in (18, 5, 4, 2) or v.shape[1] in (18, 5, 4, 2)):
                        dim = int(v.shape[1] if v.shape[0] in (18, 5, 4, 2) else v.shape[0])
                        break
        except Exception:
            pass
    return dim, keep, backbone


def find_checkpoints(input_dir: Path) -> dict[str, Path]:
    """Finds all fine-tuned LoRA and head checkpoints across safetensors and PyTorch formats."""
    ckpts = {}

    # 1. Check for heads.safetensors and heads.pt files
    head_candidates = list(input_dir.rglob("heads.safetensors")) + list(input_dir.rglob("heads.pt"))
    for p in head_candidates:
        p_str = str(p).lower().replace("\\", "/")
        is_fold = "_f0" in p_str or "_f1" in p_str or "_f2" in p_str or "_f3" in p_str or "_f4" in p_str

        target_family = ""
        meta_p = p.parent / "meta.json"
        if meta_p.exists():
            try:
                with open(meta_p, "r", encoding="utf-8") as f:
                    m_data = json.load(f)
                target_family = str(m_data.get("target_family", "")).lower()
            except Exception:
                pass

        # Critical routing guard: check family specialists before general inhouse / lora-peft-training
        if target_family == "gpt" or any(t in p_str for t in ("export_gpt", "gpt_specialist", "gpt_spec", "qwen3_14b_gpt")):
            ckpts["qwen3_14b_gpt"] = p.parent
            continue
        elif target_family == "gemini" or any(
            t in p_str for t in ("export_gemini", "gemini_specialist", "gemini_spec", "qwen3_14b_gemini")
        ):
            ckpts["qwen3_14b_gemini"] = p.parent
            continue
        elif target_family == "glm" or any(t in p_str for t in ("export_glm", "glm_specialist", "glm_spec", "qwen3_14b_glm")):
            ckpts["qwen3_14b_glm"] = p.parent
            continue

        if "lora_qwen3_14b" in p_str or "qwen3_14b_part2" in p_str or ("14b" in p_str and "part2" in p_str and "qwen3" in p_str):
            ckpts["qwen3_14b_p2"] = p.parent
        elif "lora_qwen25_14b" in p_str or "qwen25_14b_part2" in p_str or ("14b" in p_str and "part2" in p_str and "qwen2" in p_str):
            ckpts["qwen25_14b_p2"] = p.parent
        elif "q14v5" in p_str:
            if "qwen3_14b_v5" not in ckpts or (not is_fold):
                ckpts["qwen3_14b_v5"] = p.parent
        elif "ts_q3b14" in p_str:
            if "qwen3_14b_ts" not in ckpts or (not is_fold):
                ckpts["qwen3_14b_ts"] = p.parent
        elif "lora14b_v1" in p_str:
            ckpts["qwen3_14b_p1"] = p.parent
        elif any(
            token in p_str
            for token in ("inhouse", "ntl-inhouse", "export_lora24", "export_champion", "lora-peft-training", "peft_training")
        ):
            dim, keep, backbone = _inspect_checkpoint_dim_and_keep(p.parent)
            if "2.5" in backbone or "qwen25" in backbone or (dim == 40960 and (keep >= 30 or "25" in p_str or "qwen25" in p_str)):
                ckpts["qwen25_14b_inhouse"] = p.parent
            elif dim == 40960 or "14b" in p_str or "14b" in backbone:
                ckpts["qwen3_14b_inhouse"] = p.parent
            elif dim == 8192 or "deberta" in p_str or "deberta" in backbone:
                ckpts["deberta_inhouse"] = p.parent
            else:
                ckpts["qwen3_8b_inhouse"] = p.parent
        elif "q8v5" in p_str:
            if "qwen3_8b_v5" not in ckpts or (not is_fold):
                ckpts["qwen3_8b_v5"] = p.parent
        elif "ts_q3b8" in p_str or "ts8_s7" in p_str or "l24x_q3b8" in p_str:
            if "qwen3_8b_ts" not in ckpts or (not is_fold):
                ckpts["qwen3_8b_ts"] = p.parent
        elif "ll8v5" in p_str:
            if "llama3_8b_v5" not in ckpts or (not is_fold):
                ckpts["llama3_8b_v5"] = p.parent
        elif "ts_llama8" in p_str:
            if "llama3_8b_ts" not in ckpts or (not is_fold):
                ckpts["llama3_8b_ts"] = p.parent
        elif "lora8b_v1" in p_str:
            ckpts["qwen3_8b_p1"] = p.parent
        elif "ts_q25_7b" in p_str or "l24x_q25_7b" in p_str:
            if "qwen25_7b_ts" not in ckpts or (not is_fold):
                ckpts["qwen25_7b_ts"] = p.parent
        elif "instyle" in p_str:
            ckpts["instyle"] = p.parent

    # 2. Check for Claude specialist head
    for p in input_dir.rglob("claude_head.safetensors"):
        ckpts["qwen3_8b_claude"] = p.parent
        break
    if "qwen3_8b_claude" not in ckpts:
        for p in input_dir.rglob("readouts.npz"):
            if "claude" in str(p).lower():
                ckpts["qwen3_8b_claude"] = p.parent
                break

    # 3. Check for DeBERTa sequence classification head
    for p in list(input_dir.rglob("head.safetensors")) + list(input_dir.rglob("heads.safetensors")):
        if "deberta" in str(p).lower():
            ckpts["deberta_p2"] = p.parent
            break

    # 4. Fallback search for instyle
    if "instyle" not in ckpts:
        for p in input_dir.rglob("*"):
            p_s = str(p).lower().replace("\\", "/")
            if "instyle" in p_s and (
                (p / "adapter_model.safetensors").exists()
                or (p / "adapter" / "adapter_model.safetensors").exists()
                or (p / "heads.safetensors").exists()
                or (p / "heads.pt").exists()
            ):
                ckpts["instyle"] = p
                break

    # 5. Fallback search for q14v5 / ts_q3b14 adapter
    if "qwen3_14b_ts" not in ckpts:
        for p in input_dir.rglob("adapter_model.safetensors"):
            p_s = str(p).lower().replace("\\", "/")
            if "q14v5" in p_s or "ts_q3b14" in p_s:
                ckpts["qwen3_14b_ts"] = p.parent
                break

    return ckpts


def find_fold_adapters(input_dir: Path | None, prefix: str) -> list[Path]:
    """Scans input_dir for all adapter/head directories matching prefix, prefix_f*, prefix_fold*, prefix-f*."""
    if input_dir is None:
        input_dir = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")

    matches: set[Path] = set()
    prefix_lower = prefix.lower()

    candidate_files = (
        list(input_dir.rglob("heads.safetensors")) + list(input_dir.rglob("heads.pt")) + list(input_dir.rglob("adapter_model.safetensors"))
    )
    for f in candidate_files:
        p = f.parent
        p_name = p.name.lower()
        if p_name in ("adapter", "checkpoint"):
            p = p.parent
            p_name = p.name.lower()

        p_str = str(p).lower().replace("\\", "/")
        if prefix_lower in p_name or prefix_lower in p_str:
            # Check pattern: exact match or prefix followed by _ or - or fold
            tokens = p_name.replace("-", "_").split("_")
            matched = False
            for t in tokens:
                if t == prefix_lower or t.startswith(prefix_lower):
                    matched = True
                    break
            if not matched and prefix_lower in p_str:
                matched = True

            if matched:
                matches.add(p)

    return sorted(list(matches), key=lambda x: str(x).lower())


def validate_production_assets(
    input_dir: Path,
    train_path: Path,
    test_path: Path,
    pool_files: list[Path],
    ckpts: dict[str, Path],
    cfg: dict[str, Any] | None = None,
) -> dict[str, Path]:
    """
    Fail-closed pre-flight validation for production inference runs (when NTL_SMOKE != '1').
    Raises FileNotFoundError immediately at startup if any required dataset, pool parquet,
    base model trunk, or checkpoint is missing.
    Returns a dict of verified base_model paths keyed by name.
    """
    if cfg is None:
        cfg = {}
    req = cfg.get("required_assets", {})
    missing: list[str] = []

    # 1. Verify competition CSVs
    if not train_path or not Path(train_path).exists():
        missing.append(f"Competition train.csv not found at: {train_path}")
    if not test_path or not Path(test_path).exists():
        missing.append(f"Competition test.csv not found at: {test_path}")

    # 2. Verify public pool parquet(s)
    require_pool = req.get("require_pool_parquet", True)
    valid_pools = [p for p in pool_files if p is not None and Path(p).exists()]
    if require_pool and len(valid_pools) == 0:
        missing.append(f"No public pool parquet (pool_v4.parquet / nlm_pool_v5.parquet) found under {input_dir}")

    # 3. Verify all required base model trunks
    default_bases = [
        {"name": "qwen3_14b", "family": "qwen3", "hidden_size": 5120, "num_layers": 40},
        {"name": "qwen3_8b", "family": "qwen3", "hidden_size": 4096, "num_layers": 36},
        {"name": "deberta_v3_large", "family": "deberta", "hidden_size": 1024, "num_layers": 24},
    ]
    base_specs = req.get("required_base_models", default_bases)
    verified_bases: dict[str, Path] = {}
    for b_spec in base_specs:
        b_path = find_base_model(
            input_dir,
            b_spec["family"],
            int(b_spec["hidden_size"]),
            int(b_spec["num_layers"]),
        )
        if b_path is None or not Path(b_path).exists():
            missing.append(
                f"Required base model '{b_spec['name']}' (family={b_spec['family']}, "
                f"hidden={b_spec['hidden_size']}, layers={b_spec['num_layers']}) not found under {input_dir}"
            )
        else:
            verified_bases[b_spec["name"]] = b_path

    # 4. Verify all required fine-tuned checkpoints
    default_req_ckpts = ["qwen3_14b_p2", "deberta_p2", "qwen3_8b_claude"]
    req_ckpts = req.get("required_checkpoints", default_req_ckpts)
    for ck_name in req_ckpts:
        if ck_name not in ckpts or not Path(ckpts[ck_name]).exists():
            missing.append(f"Required checkpoint '{ck_name}' not found under {input_dir} (discovered: {list(ckpts.keys())})")

    if missing:
        error_msg = "[PRE-FLIGHT ASSET AUDIT FAILED] Missing required inference assets:\n  - " + "\n  - ".join(missing)
        raise FileNotFoundError(error_msg)

    print(
        f"[Pre-Flight Audit PASSED] Verified {len(valid_pools)} pool file(s), "
        f"{len(verified_bases)} base trunks, and {len(ckpts)} checkpoints: {list(ckpts.keys())}"
    )
    return verified_bases


"""
Subprocess Dual-Output Lexical Engine
====================================
Subprocess memory isolation running 78,482-row 4-View LinearSVC (Output 1)
and 5-View BPE + SGD + Style Twin + Specialist Complementary Ensemble (Output 2).
"""

import collections
import gc
import os
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC



def _norm_text_key(value: Any) -> str:
    """Normalized NFKC string key for exact deduplication."""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(value or ""))).strip()


FUNCTION_WORDS = set(
    [
        "a",
        "an",
        "the",
        "and",
        "or",
        "but",
        "if",
        "while",
        "although",
        "because",
        "as",
        "so",
        "than",
        "that",
        "which",
        "who",
        "whom",
        "whose",
        "this",
        "these",
        "those",
        "it",
        "its",
        "they",
        "them",
        "their",
        "theirs",
        "he",
        "him",
        "his",
        "she",
        "her",
        "hers",
        "we",
        "us",
        "our",
        "ours",
        "you",
        "your",
        "yours",
        "i",
        "me",
        "my",
        "mine",
        "in",
        "on",
        "at",
        "to",
        "from",
        "with",
        "without",
        "for",
        "of",
        "by",
        "over",
        "under",
        "between",
        "through",
        "into",
        "about",
        "is",
        "are",
        "was",
        "were",
        "be",
        "been",
        "being",
        "have",
        "has",
        "had",
        "do",
        "does",
        "did",
        "can",
        "could",
        "will",
        "would",
        "shall",
        "should",
        "may",
        "might",
        "must",
        "not",
        "no",
        "nor",
        "all",
        "any",
        "some",
        "each",
        "every",
        "both",
        "either",
        "neither",
        "other",
        "another",
        "such",
        "only",
        "also",
        "just",
        "very",
        "too",
        "more",
        "most",
        "less",
        "least",
        "here",
        "there",
        "when",
        "where",
        "why",
        "how",
        "what",
        "whether",
        "then",
        "now",
        "still",
        "yet",
        "even",
        "much",
        "many",
        "few",
        "little",
        "am",
        "having",
        "s",
        "t",
        "re",
        "ve",
        "ll",
        "d",
        "m",
        "don",
        "doesn",
        "didn",
        "isn",
        "aren",
        "wasn",
        "weren",
        "hasn",
        "haven",
        "hadn",
        "won",
        "wouldn",
        "couldn",
        "shouldn",
    ]
)
STOP_WORDS = FUNCTION_WORDS


def shape_text(t: str) -> str:
    """
    Transforms text into structural casing patterns (shape view).
    Replaces capitalized words with 'X' and lowercase words with 'x'.
    Captures formatting capitalization rhythm independent of lexical vocabulary.
    """
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m.group(0)[0].isupper() else "x", str(t))


def function_text(t: str) -> str:
    """
    Transforms text into a pure function-word and syntactic skeleton (function view).
    Masks open-class content words (nouns, main verbs, adjectives) to 'CONTENT' while
    preserving closed-class function words (prepositions, conjunctions, pronouns, auxiliaries)
    and punctuation. Stylometric research demonstrates that function-word transition rates
    form one of the strongest, topic-invariant signatures of LLM generator identity.
    """
    tokens = re.findall(r"(?u)\b\w+\b|[^\w\s]", str(t))
    return " ".join([w.lower() if (w.lower() in STOP_WORDS or not w.isalnum()) else "CONTENT" for w in tokens])


_RE_CODE_BLOCK = re.compile(r"```[\s\S]*?```|`[^`\n]+`")
_RE_STRAIGHT_CONTR = re.compile(r"[A-Za-z]'[A-Za-z]")
_RE_CURLY_CONTR = re.compile(r"[A-Za-z]\u2019[A-Za-z]")


def extract_handwriting_and_orthographic_features(texts: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """
    Single-pass extractor returning:
    - feats: (N, 26) float32 continuous punctuation, contraction, and structural densities
    - rules: (N, 4) float32 binary indicators for the 4 discrete orthographic invariants:
        col 0: any curly quote (\u2018, \u2019, \u201c, \u201d)
        col 1: >= 2 em-dashes (\u2014) in non-code prose
        col 2: >= 2 straight contractions [A-Za-z]'[A-Za-z] in non-code prose AND 0 curly \u2019
        col 3: >= 1 curly contraction [A-Za-z]\u2019[A-Za-z] in non-code prose
    """
    N = len(texts)
    feats = np.zeros((N, 26), dtype=np.float32)
    rules = np.zeros((N, 4), dtype=np.float32)

    for i, raw_t in enumerate(texts):
        t = str(raw_t)
        prose = _RE_CODE_BLOCK.sub(" ", t)
        t_len = max(len(t), 1)
        words = t.split()
        n_words = max(len(words), 1)

        raw_curly_sq = t.count("\u2018") + t.count("\u2019")
        raw_curly_dq = t.count("\u201c") + t.count("\u201d")
        prose_em_dash = prose.count("\u2014")
        n_str_c = len(_RE_STRAIGHT_CONTR.findall(prose))
        n_cur_c = len(_RE_CURLY_CONTR.findall(prose))

        # 4 Discrete Structural Rule Flags
        rules[i, 0] = 1.0 if (raw_curly_sq + raw_curly_dq) > 0 else 0.0
        rules[i, 1] = 1.0 if prose_em_dash >= 2 else 0.0
        rules[i, 2] = 1.0 if (n_str_c >= 2 and raw_curly_sq == 0) else 0.0
        rules[i, 3] = 1.0 if n_cur_c >= 1 else 0.0

        # 26 Continuous Density Features
        c_colon = t.count(":") / n_words
        c_semi = t.count(";") / n_words
        c_em_dash = t.count("\u2014") / n_words
        c_double_hyph = t.count("--") / n_words
        c_curly_sq = raw_curly_sq / n_words
        c_straight_sq = t.count("'") / n_words
        c_curly_dq = raw_curly_dq / n_words
        c_straight_dq = t.count('"') / n_words
        c_paren = (t.count("(") + t.count(")")) / n_words
        c_comma = t.count(",") / n_words
        c_excl = t.count("!") / n_words
        c_ques = t.count("?") / n_words
        c_ellipsis = (t.count("...") + t.count("\u2026")) / n_words
        c_hyphen = t.count("-") / n_words
        n_straight_contr = n_str_c / n_words
        n_curly_contr = n_cur_c / n_words

        c_newline = t.count("\n") / n_words
        c_bullet = (t.count("\n- ") + t.count("\n* ")) / n_words
        c_upper = sum(1 for ch in t if ch.isupper()) / t_len
        c_digits = sum(1 for ch in t if ch.isdigit()) / t_len

        w_lens = [len(w) for w in words]
        mean_w_len = float(np.mean(w_lens)) if w_lens else 0.0
        std_w_len = float(np.std(w_lens)) if len(w_lens) > 1 else 0.0

        sents = [s.strip() for s in t.replace("!", ".").replace("?", ".").split(".") if s.strip()]
        s_lens = [len(s.split()) for s in sents]
        mean_s_len = float(np.mean(s_lens)) if s_lens else 0.0
        std_s_len = float(np.std(s_lens)) if len(s_lens) > 1 else 0.0
        burstiness = float(std_s_len / max(mean_s_len, 1e-5))
        ttr = float(len(set(words)) / n_words)

        feats[i] = [
            c_colon,
            c_semi,
            c_em_dash,
            c_double_hyph,
            c_curly_sq,
            c_straight_sq,
            c_curly_dq,
            c_straight_dq,
            c_paren,
            c_comma,
            c_excl,
            c_ques,
            c_ellipsis,
            c_hyphen,
            n_straight_contr,
            n_curly_contr,
            c_newline,
            c_bullet,
            c_upper,
            c_digits,
            mean_w_len,
            std_w_len,
            mean_s_len,
            std_s_len,
            burstiness,
            ttr,
        ]
    return feats, rules


def extract_handwriting_features(texts: list[str]) -> np.ndarray:
    """Backward-compatible wrapper returning only the (N, 26) continuous matrix."""
    feats, _ = extract_handwriting_and_orthographic_features(texts)
    return feats


def fit_handwriting_model(
    hw_tr: np.ndarray,
    fit_y: np.ndarray,
    hw_te: np.ndarray,
    df_tr: pd.DataFrame,
    groups_085: np.ndarray,
    twin_rows_085: np.ndarray,
    len_tr_raw: int,
) -> np.ndarray:
    """
    Fits dedicated LogisticRegression plain (C=0.25) and prompt-centered (C=0.15)
    classifiers strictly on the 26 continuous handwriting features.
    Returns log_p_handwriting (N_test, 18).
    """
    clf_hw_plain = LogisticRegression(C=0.25, max_iter=1000, random_state=42, class_weight="balanced")
    clf_hw_plain.fit(hw_tr, fit_y)
    lp_hw = log_softmax(clf_hw_plain.decision_function(hw_te))

    if twin_rows_085.any() and len_tr_raw >= 18:
        hw_tr_c = hw_tr[:len_tr_raw].copy()
        for p in np.unique(df_tr["prompt_id"].values):
            m_p = df_tr["prompt_id"].values == p
            hw_tr_c[m_p] -= hw_tr_c[m_p].mean(axis=0, keepdims=True)
        clf_hw_twin = LogisticRegression(C=0.15, max_iter=1000, random_state=42, class_weight="balanced")
        clf_hw_twin.fit(hw_tr_c, fit_y[:len_tr_raw])

        hw_te_c = hw_te.copy()
        for g in np.unique(groups_085[twin_rows_085]):
            m_g = groups_085 == g
            hw_te_c[m_g] -= hw_te_c[m_g].mean(axis=0, keepdims=True)
        lp_hw_twin = log_softmax(clf_hw_twin.decision_function(hw_te_c[twin_rows_085]))
        lp_hw[twin_rows_085] = log_softmax(0.70 * lp_hw[twin_rows_085] + 0.30 * lp_hw_twin)

    return lp_hw


def run_lexical_engine(
    train_path: Path, test_path: Path, pool_file: Path | None, qwen_tok_dir: str, working_dir: Path
) -> dict[str, np.ndarray]:
    """
    Runs the complete dual-output lexical engine in-process or writes standalone script.
    Saves and returns:
    - log_p_svc_main.npy (Output 1 for LOGIT_MAIN)
    - p_lex_comp.npy (Output 2 for COMP_LIST)
    - D_tfidf.npy
    - groups_085.npy
    """
    df_tr = pd.read_csv(train_path)
    df_te = pd.read_csv(test_path)

    test_texts = df_te["text"].fillna("").astype(str).tolist()
    fit_texts = df_tr["text"].fillna("").astype(str).tolist()
    fit_y = np.array([LAB2I[m] for m in df_tr["model"]])
    fit_sw = np.ones(len(df_tr), dtype=np.float32)  # Official train rows get 1.00
    seen_texts = {_norm_text_key(t) for t in fit_texts}
    len_tr_raw = len(df_tr)

    # 1. Compute stopword TF-IDF distance matrix and topic clusters (t=0.85) on test
    vec_kwargs = dict(
        stop_words="english",
        sublinear_tf=True,
        min_df=2 if len(test_texts) >= 20 else 1,
        max_df=0.25 if len(test_texts) >= 100 else 1.0,
        max_features=100000,
        dtype=np.float32,
    )
    try:
        Xt_tfidf = TfidfVectorizer(**vec_kwargs).fit_transform(test_texts)
    except ValueError:
        vec_kwargs["min_df"] = 1
        vec_kwargs["max_df"] = 1.0
        Xt_tfidf = TfidfVectorizer(**vec_kwargs).fit_transform(test_texts)
    if len(test_texts) >= 2:
        D = np.clip(1.0 - (Xt_tfidf @ Xt_tfidf.T).toarray(), 0.0, None)
        np.fill_diagonal(D, 0.0)
        cond_d = squareform(D, checks=False)
        groups_085 = fcluster(linkage(cond_d, method="average"), 0.85, criterion="distance")
    else:
        D = np.zeros((1, 1), dtype=np.float32)
        groups_085 = np.ones(1, dtype=int)

    sz_085 = np.bincount(groups_085)[groups_085]
    twin_rows_085 = (sz_085 >= 2) & (sz_085 <= 18)

    # 1.5. Check for quarantined unseen holdout dataset
    working_dir = Path(working_dir)
    unseen_parquet = working_dir / "unseen_holdout.parquet"
    unseen_texts: list[str] = []
    unseen_hashes = set()
    if unseen_parquet.exists():
        try:
            df_unseen_holdout = pd.read_parquet(unseen_parquet)
            if len(df_unseen_holdout) > 0 and "text" in df_unseen_holdout.columns:
                unseen_texts = df_unseen_holdout["text"].fillna("").astype(str).tolist()
                if "_hash" in df_unseen_holdout.columns:
                    unseen_hashes = set(df_unseen_holdout["_hash"].tolist())
                else:
                    unseen_hashes = set(_text_hash(t) for t in unseen_texts)
                print(f"[Lexical] Found unseen holdout set: {len(unseen_texts)} rows quarantined from training.")
        except Exception as e:
            print(f"[Lexical] Note: Could not read {unseen_parquet}: {e}")

    # 2. Load pool parquet files if present with source weights & stratified two-tier caps
    core_texts = list(fit_texts)
    core_y = fit_y.copy()
    core_sw = fit_sw.copy()
    extra_fit_texts: list[str] = []
    extra_fit_y: list[int] = []
    extra_fit_sw: list[float] = []

    MAX_CORE_POOL_PER_CLASS = 700
    MAX_TOTAL_POOL_PER_CLASS = 2400
    core_class_counts = np.zeros(K, dtype=int)
    total_class_counts = np.zeros(K, dtype=int)
    rng_pool = np.random.RandomState(42)

    pool_candidates = [pool_file] if isinstance(pool_file, (str, Path)) else (pool_file if isinstance(pool_file, list) else [])
    for pf in pool_candidates:
        if pf and os.path.exists(pf):
            try:
                p_df = pd.read_parquet(pf, columns=["text", "model"])
                p_df = p_df[p_df["model"].isin(LABELS)].dropna(subset=["text", "model"]).copy()
                pf_filename = Path(pf).name.lower()
                is_matched_clean = "matched" in pf_filename or "clean" in pf_filename
                src_w = 0.50 if is_matched_clean else 0.35

                # Filter to competition passage length (25..360 words) when pool is large
                if len(p_df) > 5000:
                    w_counts = p_df["text"].astype(str).str.split().str.len()
                    p_df = p_df[w_counts.between(25, 360)].copy()

                # Shuffle deterministically before stratified cap
                p_df = p_df.sample(frac=1.0, random_state=rng_pool).reset_index(drop=True)

                fit_add_t, fit_add_y, fit_add_sw = [], [], []
                core_add_t, core_add_y, core_add_sw = [], [], []

                for t_val, m_val in zip(p_df["text"].values, p_df["model"].values):
                    c_idx = LAB2I[m_val]
                    max_tot_c = int(1.4 * MAX_TOTAL_POOL_PER_CLASS) if c_idx in HARD_CLASS_INDICES else MAX_TOTAL_POOL_PER_CLASS
                    if total_class_counts[c_idx] >= max_tot_c:
                        continue
                    k = _norm_text_key(t_val)
                    if not k or k in seen_texts:
                        continue
                    if unseen_hashes and _text_hash(t_val) in unseen_hashes:
                        continue
                    seen_texts.add(k)
                    t_str = str(t_val)
                    total_class_counts[c_idx] += 1

                    max_core_c = int(1.5 * MAX_CORE_POOL_PER_CLASS) if c_idx in HARD_CLASS_INDICES else MAX_CORE_POOL_PER_CLASS
                    if core_class_counts[c_idx] < max_core_c:
                        core_class_counts[c_idx] += 1
                        core_add_t.append(t_str)
                        core_add_y.append(c_idx)
                        core_add_sw.append(src_w)
                    else:
                        fit_add_t.append(t_str)
                        fit_add_y.append(c_idx)
                        fit_add_sw.append(src_w)

                if core_add_t:
                    core_texts.extend(core_add_t)
                    core_y = np.r_[core_y, np.array(core_add_y, dtype=int)]
                    core_sw = np.r_[core_sw, np.array(core_add_sw, dtype=np.float32)]
                if fit_add_t:
                    extra_fit_texts.extend(fit_add_t)
                    extra_fit_y.extend(fit_add_y)
                    extra_fit_sw.extend(fit_add_sw)
                print(
                    f"[Lexical] Added {len(core_add_t) + len(fit_add_t)} rows (core={len(core_add_t)}) from {Path(pf).name} with source weight {src_w}"
                )
            except Exception as e:
                print(f"[Lexical] Note: Could not read pool {pf}: {e}")

    fit_texts = core_texts + extra_fit_texts
    fit_y = np.r_[core_y, np.array(extra_fit_y, dtype=int)] if extra_fit_y else core_y
    fit_sw = np.r_[core_sw, np.array(extra_fit_sw, dtype=np.float32)] if extra_fit_sw else core_sw

    # 3. Canonical 4 Views (from NTL two-stage LoRA part two)
    VIEWS_4V = [
        ("char", 3, 5, 250000, 1.0),
        ("word", 1, 3, 180000, 0.7),
        ("shape", 2, 6, 60000, 0.5),
        ("function", 1, 3, 50000, 0.5),
    ]

    fit_sh = [shape_text(t) for t in fit_texts]
    test_sh = [shape_text(t) for t in test_texts]
    unseen_sh = [shape_text(t) for t in unseen_texts] if unseen_texts else []
    fit_fn = [function_text(t) for t in fit_texts]
    test_fn = [function_text(t) for t in test_texts]
    unseen_fn = [function_text(t) for t in unseen_texts] if unseen_texts else []

    tr_mats_4v = []
    te_mats_4v = []
    unseen_mats_4v = []
    for mode, mn, mx, mf, sc in VIEWS_4V:
        analyzer = "char" if mode == "shape" else ("word" if mode == "function" else mode)
        opts = dict(
            analyzer=analyzer,
            ngram_range=(mn, mx),
            lowercase=False,
            sublinear_tf=True,
            min_df=2 if len(fit_texts) >= 50 else 1,
            max_features=mf,
            dtype=np.float32,
        )
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"

        v = TfidfVectorizer(**opts)
        if mode == "shape":
            tr_m = v.fit_transform(fit_sh) * np.float32(sc)
            te_m = v.transform(test_sh) * np.float32(sc)
            u_m = v.transform(unseen_sh) * np.float32(sc) if unseen_texts else None
        elif mode == "function":
            tr_m = v.fit_transform(fit_fn) * np.float32(sc)
            te_m = v.transform(test_fn) * np.float32(sc)
            u_m = v.transform(unseen_fn) * np.float32(sc) if unseen_texts else None
        else:
            tr_m = v.fit_transform(fit_texts) * np.float32(sc)
            te_m = v.transform(test_texts) * np.float32(sc)
            u_m = v.transform(unseen_texts) * np.float32(sc) if unseen_texts else None
        tr_mats_4v.append(tr_m)
        te_mats_4v.append(te_m)
        if u_m is not None:
            unseen_mats_4v.append(u_m)

    X_tr_4v = sparse.hstack(tr_mats_4v, format="csr")
    X_te_4v = sparse.hstack(te_mats_4v, format="csr")
    X_unseen_4v = sparse.hstack(unseen_mats_4v, format="csr") if unseen_mats_4v else None
    X_core_4v = X_tr_4v[: len(core_texts)]

    # OUTPUT 1: Pure 4-View LinearSVC (C=0.5, class_weight='balanced', source sample weights)
    svc_main = LinearSVC(C=0.5, dual=True, random_state=42, max_iter=2000, tol=1e-3, class_weight="balanced")
    svc_main.fit(X_tr_4v, fit_y, sample_weight=fit_sw)
    logits_svc_main = svc_main.decision_function(X_te_4v)
    log_p_svc_main = log_softmax(logits_svc_main)

    if X_unseen_4v is not None and X_unseen_4v.shape[0] > 0:
        unseen_log_p_svc_main = log_softmax(svc_main.decision_function(X_unseen_4v))
    else:
        unseen_log_p_svc_main = np.zeros((0, K), dtype=np.float32)

    # OUTPUT 2: Stamatatos Text Distortion Model (trained strictly on core_texts with feature caps)
    wre = re.compile(r"[A-Za-z’']+")
    cnt = collections.Counter(w.lower() for t in core_texts for w in wre.findall(t))
    distort_keep = {w for w, _ in cnt.most_common(100)}

    def dist_fn(arr):
        return [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in distort_keep else "*", t) for t in arr]

    D_tr = dist_fn(core_texts)
    D_te = dist_fn(test_texts)
    D_unseen = dist_fn(unseen_texts) if unseen_texts else []

    vs_dist = [
        TfidfVectorizer(
            sublinear_tf=True,
            lowercase=False,
            analyzer="char",
            ngram_range=(2, 6),
            max_features=100000,
            dtype=np.float32,
            min_df=2 if len(core_texts) >= 50 else 1,
        ),
        TfidfVectorizer(
            sublinear_tf=True,
            lowercase=True,
            token_pattern=r"[^\s]+",
            ngram_range=(1, 3),
            max_features=60000,
            dtype=np.float32,
            min_df=2 if len(core_texts) >= 50 else 1,
        ),
    ]
    Xa_dist = sparse.hstack([v_.fit_transform(D_tr) for v_ in vs_dist], format="csr")
    Xb_dist = sparse.hstack([v_.transform(D_te) for v_ in vs_dist], format="csr")
    Xu_dist = sparse.hstack([v_.transform(D_unseen) for v_ in vs_dist], format="csr") if unseen_texts else None

    lr_distort = LogisticRegression(C=10.0, max_iter=500, random_state=42)
    lr_distort.fit(Xa_dist, core_y, sample_weight=core_sw)
    P_distort = lr_distort.predict_proba(Xb_dist)
    log_p_distort = lr_distort.predict_log_proba(Xb_dist)
    if Xu_dist is not None and Xu_dist.shape[0] > 0:
        P_unseen_distort = lr_distort.predict_proba(Xu_dist)
        log_p_unseen_distort = lr_distort.predict_log_proba(Xu_dist)
    else:
        P_unseen_distort = None
        log_p_unseen_distort = None

    # OUTPUT 3: 5-View BPE + SGD + Style Twin + Specialist Complementary Ensemble
    # Continuous Handwriting & Punctuation Density Features + Dedicated Classifier (fit strictly on df_tr train.csv)
    hw_core_raw, _ = extract_handwriting_and_orthographic_features(core_texts)
    hw_te_raw, rule_flags_te = extract_handwriting_and_orthographic_features(test_texts)
    hw_unseen_raw, _ = extract_handwriting_and_orthographic_features(unseen_texts) if unseen_texts else (None, None)
    scaler = StandardScaler()
    scaler.fit(hw_core_raw[:len_tr_raw])
    hw_core = np.clip(scaler.transform(hw_core_raw), -5.0, 5.0).astype(np.float32)
    hw_te = np.clip(scaler.transform(hw_te_raw), -5.0, 5.0).astype(np.float32)
    hw_unseen = np.clip(scaler.transform(hw_unseen_raw), -5.0, 5.0).astype(np.float32) if hw_unseen_raw is not None else None
    hw_tr = hw_core[:len_tr_raw]

    lp_hw = fit_handwriting_model(hw_tr, fit_y[:len_tr_raw], hw_te, df_tr, groups_085, twin_rows_085, len_tr_raw)

    tok = None
    if qwen_tok_dir and os.path.exists(qwen_tok_dir):
        try:
            from transformers import AutoTokenizer

            tok = AutoTokenizer.from_pretrained(qwen_tok_dir, use_fast=True)
        except Exception:
            tok = None

    if tok is not None:

        def batch_bpe_strings(texts: list[str], batch_sz: int = 4000) -> list[str]:
            out_strs = []
            for s in range(0, len(texts), batch_sz):
                batch_ids = tok(
                    texts[s : s + batch_sz],
                    add_special_tokens=False,
                    truncation=True,
                    max_length=512,
                )["input_ids"]
                out_strs.extend(" ".join(map(str, ids)) for ids in batch_ids)
            return out_strs

        fit_bpe_strs = batch_bpe_strings(core_texts)
        test_bpe_strs = batch_bpe_strings(test_texts)
        unseen_bpe_strs = batch_bpe_strings(unseen_texts) if unseen_texts else []
        v_bpe = TfidfVectorizer(
            analyzer="word",
            token_pattern=r"\S+",
            lowercase=False,
            ngram_range=(1, 3),
            max_features=75000,
            sublinear_tf=True,
            dtype=np.float32,
        )
        tr_bpe = v_bpe.fit_transform(fit_bpe_strs) * 0.8
        te_bpe = v_bpe.transform(test_bpe_strs) * 0.8
        unseen_bpe = v_bpe.transform(unseen_bpe_strs) * 0.8 if unseen_texts else None
        del fit_bpe_strs, test_bpe_strs, unseen_bpe_strs, v_bpe
        gc.collect()
        X_core_5v = sparse.hstack([X_core_4v, tr_bpe], format="csr")
        X_te_5v = sparse.hstack([X_te_4v, te_bpe], format="csr")
        X_unseen_5v = sparse.hstack([X_unseen_4v, unseen_bpe], format="csr") if unseen_bpe is not None else None
    else:
        v_cw = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=75000, sublinear_tf=True, dtype=np.float32)
        tr_cw = v_cw.fit_transform(core_texts) * 0.8
        te_cw = v_cw.transform(test_texts) * 0.8
        unseen_cw = v_cw.transform(unseen_texts) * 0.8 if unseen_texts else None
        X_core_5v = sparse.hstack([X_core_4v, tr_cw], format="csr")
        X_te_5v = sparse.hstack([X_te_4v, te_cw], format="csr")
        X_unseen_5v = sparse.hstack([X_unseen_4v, unseen_cw], format="csr") if unseen_cw is not None else None

    comp_core_sw = core_sw.copy()
    # Boost non-Gemini hard classes, keeping 7 and 8 strictly symmetric
    hard_mask = np.isin(core_y, [c for c in HARD_CLASS_INDICES if c not in (7, 8)])
    comp_core_sw[hard_mask] *= 1.20

    svc_comp = LinearSVC(C=0.35, random_state=42, max_iter=3000, tol=1e-3, class_weight="balanced")
    svc_comp.fit(X_core_5v, core_y, sample_weight=comp_core_sw)
    P_svc_comp = softmax(svc_comp.decision_function(X_te_5v), axis=1)

    sgd_comp = SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=1500, random_state=42)
    sgd_comp.fit(X_core_5v, core_y, sample_weight=comp_core_sw)
    P_sgd_comp = sgd_comp.predict_proba(X_te_5v)

    # Compute baseline probabilities
    P_unseen_svc = (
        softmax(svc_comp.decision_function(X_unseen_5v), axis=1) if (X_unseen_5v is not None and X_unseen_5v.shape[0] > 0) else None
    )
    P_unseen_sgd = sgd_comp.predict_proba(X_unseen_5v) if (X_unseen_5v is not None and X_unseen_5v.shape[0] > 0) else None

    # Style Twin on groups_085
    P_twin_comp = P_svc_comp.copy()
    P_unseen_twin = P_unseen_svc.copy() if P_unseen_svc is not None else None
    if twin_rows_085.any() and len_tr_raw >= 18:
        X_train_only = X_core_5v[:len_tr_raw]
        u_tr, inv_tr = np.unique(df_tr["prompt_id"].values, return_inverse=True)
        counts_tr = np.bincount(inv_tr)
        vals_tr = (1.0 / counts_tr[inv_tr]).astype(np.float32)
        G = sparse.csr_matrix((vals_tr, (np.arange(len(inv_tr)), inv_tr)), shape=(len(inv_tr), len(u_tr)), dtype=np.float32)
        X_train_c = X_train_only - G @ (G.T @ X_train_only)

        svc_twin = LinearSVC(C=0.30, random_state=42, max_iter=2500)
        svc_twin.fit(X_train_c, core_y[:len_tr_raw])
        del X_train_only, G, X_train_c
        gc.collect()

        X_te_twin = X_te_5v[twin_rows_085]
        u_t, inv_t = np.unique(groups_085[twin_rows_085], return_inverse=True)
        counts_t = np.bincount(inv_t)
        vals_t = (1.0 / counts_t[inv_t]).astype(np.float32)
        G_t = sparse.csr_matrix((vals_t, (np.arange(len(inv_t)), inv_t)), shape=(len(inv_t), len(u_t)), dtype=np.float32)
        X_te_twin_c = X_te_twin - G_t @ (G_t.T @ X_te_twin)
        P_twin_comp[twin_rows_085] = softmax(svc_twin.decision_function(X_te_twin_c), axis=1)

        if X_unseen_5v is not None and X_unseen_5v.shape[0] > 0:
            if (
                "df_unseen_holdout" in locals()
                and "prompt_id" in df_unseen_holdout.columns
                and len(df_unseen_holdout) == X_unseen_5v.shape[0]
            ):
                u_u, inv_u = np.unique(df_unseen_holdout["prompt_id"].values, return_inverse=True)
                counts_u = np.bincount(inv_u)
                vals_u = (1.0 / counts_u[inv_u]).astype(np.float32)
                G_u = sparse.csr_matrix((vals_u, (np.arange(len(inv_u)), inv_u)), shape=(len(inv_u), len(u_u)), dtype=np.float32)
                X_unseen_twin_c = X_unseen_5v - G_u @ (G_u.T @ X_unseen_5v)
                P_unseen_twin = softmax(svc_twin.decision_function(X_unseen_twin_c), axis=1)
                del G_u, X_unseen_twin_c
            else:
                P_unseen_twin = softmax(svc_twin.decision_function(X_unseen_5v), axis=1)

        del svc_twin, X_te_twin, G_t, X_te_twin_c
        gc.collect()

    if X_unseen_5v is not None and X_unseen_5v.shape[0] > 0:
        if P_unseen_distort is not None:
            P_unseen_comp = 0.40 * P_unseen_svc + 0.25 * P_unseen_sgd + 0.15 * P_unseen_twin + 0.20 * P_unseen_distort
        else:
            P_unseen_comp = 0.55 * P_unseen_svc + 0.45 * P_unseen_sgd
        log_p_unseen_comp = np.log(np.clip(P_unseen_comp, 1e-9, 1.0))
    else:
        log_p_unseen_comp = None

    P_lex_comp = 0.40 * P_svc_comp + 0.25 * P_sgd_comp + 0.15 * P_twin_comp + 0.20 * P_distort
    P_lex_comp_final = P_lex_comp.copy()
    unseen_p_lex_comp_final = (
        P_unseen_comp.copy() if (X_unseen_5v is not None and X_unseen_5v.shape[0] > 0) else np.zeros((0, K), dtype=np.float32)
    )

    hw_c_sp = sparse.csr_matrix(hw_core * 0.25)
    hw_te_sp = sparse.csr_matrix(hw_te * 0.25)
    hw_u_sp = sparse.csr_matrix(hw_unseen * 0.25) if hw_unseen is not None else None

    # Disjoint Mass-Conserving Provider Family Refiners (prevents overlapping triple-counting)
    for grp_name, grp_cols in FAMILIES.items():
        grp_mask = np.isin(core_y, grp_cols)
        if grp_mask.sum() > 30 and len(np.unique(core_y[grp_mask])) == len(grp_cols):
            sub_y = core_y[grp_mask]
            sub_lab2i = {c: i for i, c in enumerate(grp_cols)}
            sub_y_mapped = np.array([sub_lab2i[y] for y in sub_y], dtype=int)

            c_val = 0.35 if grp_name == "glm" else 0.40
            X_tr_grp = sparse.hstack([X_core_5v[grp_mask], hw_c_sp[grp_mask]], format="csr")
            X_te_grp = sparse.hstack([X_te_5v, hw_te_sp], format="csr")
            X_u_grp = sparse.hstack([X_unseen_5v, hw_u_sp], format="csr") if X_unseen_5v is not None else None

            spec_svc = LinearSVC(C=c_val, random_state=42, max_iter=2000, tol=1e-3, class_weight="balanced")
            spec_svc.fit(X_tr_grp, sub_y_mapped, sample_weight=comp_core_sw[grp_mask])
            df_spec = spec_svc.decision_function(X_te_grp)
            logits_spec = np.column_stack([-df_spec, df_spec]) if df_spec.ndim == 1 else df_spec
            P_spec_plain = softmax(logits_spec, axis=1)

            train_mask = grp_mask[:len_tr_raw]
            has_centered_pair = train_mask.sum() >= 6 * len(grp_cols) and len(np.unique(core_y[:len_tr_raw][train_mask])) == len(grp_cols)
            if has_centered_pair:
                X_tr_pair = sparse.hstack([X_core_5v[:len_tr_raw][train_mask], hw_c_sp[:len_tr_raw][train_mask]], format="csr")
                y_tr_pair = np.array([sub_lab2i[y] for y in core_y[:len_tr_raw][train_mask]], dtype=int)
                prompts_pair = df_tr["prompt_id"].values[train_mask]
                u_p, inv_p = np.unique(prompts_pair, return_inverse=True)
                counts_p = np.bincount(inv_p)
                vals_p = (1.0 / np.maximum(counts_p[inv_p], 1)).astype(np.float32)
                G_p = sparse.csr_matrix((vals_p, (np.arange(len(inv_p)), inv_p)), shape=(len(inv_p), len(u_p)), dtype=np.float32)
                X_tr_pair_c = X_tr_pair - G_p @ (G_p.T @ X_tr_pair)

                clf_centered = LogisticRegression(C=0.25, class_weight="balanced", random_state=42, max_iter=1000)
                clf_centered.fit(X_tr_pair_c, y_tr_pair)

                P_spec = P_spec_plain.copy()
                if twin_rows_085.any():
                    X_te_sub = X_te_grp[twin_rows_085]
                    u_t, inv_t = np.unique(groups_085[twin_rows_085], return_inverse=True)
                    counts_t = np.bincount(inv_t)
                    vals_t = (1.0 / np.maximum(counts_t[inv_t], 1)).astype(np.float32)
                    G_t = sparse.csr_matrix((vals_t, (np.arange(len(inv_t)), inv_t)), shape=(len(inv_t), len(u_t)), dtype=np.float32)
                    X_te_pair_c = X_te_sub - G_t @ (G_t.T @ X_te_sub)
                    P_spec_centered = clf_centered.predict_proba(X_te_pair_c)
                    w_plain_s = 0.50 if grp_name == "glm" else 0.65
                    P_spec[twin_rows_085] = w_plain_s * P_spec_plain[twin_rows_085] + (1.0 - w_plain_s) * P_spec_centered
            else:
                P_spec = P_spec_plain

            beta_grp = 0.25
            P_prior = P_lex_comp_final[:, grp_cols]
            fam_mass = P_prior.sum(axis=1, keepdims=True)
            P_prior_norm = P_prior / (fam_mass + 1e-12)
            P_refined = (1.0 - beta_grp) * P_prior_norm + beta_grp * P_spec
            P_lex_comp_final[:, grp_cols] = fam_mass * P_refined

            if len(unseen_p_lex_comp_final) > 0 and X_u_grp is not None and X_u_grp.shape[0] > 0:
                df_spec_u = spec_svc.decision_function(X_u_grp)
                logits_spec_u = np.column_stack([-df_spec_u, df_spec_u]) if df_spec_u.ndim == 1 else df_spec_u
                P_spec_u_plain = softmax(logits_spec_u, axis=1)
                P_spec_u = P_spec_u_plain.copy()
                if (
                    has_centered_pair
                    and "df_unseen_holdout" in locals()
                    and "prompt_id" in df_unseen_holdout.columns
                    and len(df_unseen_holdout) == X_u_grp.shape[0]
                ):
                    u_u, inv_u = np.unique(df_unseen_holdout["prompt_id"].values, return_inverse=True)
                    counts_u = np.bincount(inv_u)
                    sz_u = counts_u[inv_u]
                    twin_u_mask = (sz_u >= 2) & (sz_u <= 18)
                    if twin_u_mask.any():
                        X_u_sub = X_u_grp[twin_u_mask]
                        u_t, inv_t = np.unique(df_unseen_holdout["prompt_id"].values[twin_u_mask], return_inverse=True)
                        counts_t = np.bincount(inv_t)
                        vals_t = (1.0 / np.maximum(counts_t[inv_t], 1)).astype(np.float32)
                        G_t = sparse.csr_matrix((vals_t, (np.arange(len(inv_t)), inv_t)), shape=(len(inv_t), len(u_t)), dtype=np.float32)
                        X_u_pair_c = X_u_sub - G_t @ (G_t.T @ X_u_sub)
                        P_spec_u_centered = clf_centered.predict_proba(X_u_pair_c)
                        w_plain_s = 0.50 if grp_name == "glm" else 0.65
                        P_spec_u[twin_u_mask] = w_plain_s * P_spec_u_plain[twin_u_mask] + (1.0 - w_plain_s) * P_spec_u_centered

                P_prior_u = unseen_p_lex_comp_final[:, grp_cols]
                fam_mass_u = P_prior_u.sum(axis=1, keepdims=True)
                P_prior_u_norm = P_prior_u / (fam_mass_u + 1e-12)
                P_refined_u = (1.0 - beta_grp) * P_prior_u_norm + beta_grp * P_spec_u
                unseen_p_lex_comp_final[:, grp_cols] = fam_mass_u * P_refined_u

    P_lex_comp_final /= P_lex_comp_final.sum(axis=1, keepdims=True).clip(min=1e-12)
    if len(unseen_p_lex_comp_final) > 0:
        unseen_p_lex_comp_final /= unseen_p_lex_comp_final.sum(axis=1, keepdims=True).clip(min=1e-12)

    # Save to disk
    os.makedirs(working_dir, exist_ok=True)
    np.save(working_dir / "hw_train.npy", hw_core[:len_tr_raw])
    np.save(working_dir / "hw_test.npy", hw_te)
    if hw_unseen is not None:
        np.save(working_dir / "hw_unseen.npy", hw_unseen)
    np.save(working_dir / "log_p_svc_main.npy", log_p_svc_main)
    np.save(working_dir / "p_lex_comp.npy", P_lex_comp_final)
    np.save(working_dir / "D_tfidf.npy", D)
    np.save(working_dir / "groups_085.npy", groups_085)
    np.save(working_dir / "log_p_handwriting.npy", lp_hw)
    np.save(working_dir / "rule_flags_test.npy", rule_flags_te)
    np.save(working_dir / "unseen_log_p_svc_main.npy", unseen_log_p_svc_main)
    np.save(working_dir / "unseen_p_lex_comp.npy", unseen_p_lex_comp_final)
    if log_p_distort is not None:
        np.save(working_dir / "log_p_distort.npy", log_p_distort)
    if log_p_unseen_distort is not None:
        np.save(working_dir / "unseen_log_p_distort.npy", log_p_unseen_distort)

    return {
        "log_p_svc_main": log_p_svc_main,
        "p_lex_comp": P_lex_comp_final,
        "D_tfidf": D,
        "groups_085": groups_085,
        "log_p_handwriting": lp_hw,
        "rule_flags_test": rule_flags_te,
        "unseen_log_p_svc_main": unseen_log_p_svc_main,
        "unseen_p_lex_comp": unseen_p_lex_comp_final,
        "hw_train": hw_core[:len_tr_raw],
        "hw_test": hw_te,
        "hw_unseen": hw_unseen,
        "log_p_distort": log_p_distort,
        "unseen_log_p_distort": log_p_unseen_distort,
    }


if __name__ == "__main__":
    working_dir = Path("/kaggle/working")
    if not working_dir.exists():
        working_dir = Path("local_data")
    working_dir.mkdir(parents=True, exist_ok=True)

    input_dir = Path("/kaggle/input")
    if not input_dir.exists():
        input_dir = Path(".")

    train_path, test_path, _ = resolve_dataset_paths(input_dir)
    pool_files = resolve_all_pool_files(input_dir)

    tok_paths = list(input_dir.rglob("vocab.json"))
    qwen_tok_dir = str(tok_paths[0].parent) if tok_paths else ""

    print(f"[Module 1 CLI] Running lexical engine: train={train_path}, test={test_path}, pools={[p.name for p in pool_files]}")
    run_lexical_engine(train_path, test_path, pool_files, qwen_tok_dir, working_dir)
    print("[Module 1 CLI] Lexical engine finished successfully.")


In [ ]:
# === MODULE: src/namethatllm/config.py ===
ENSEMBLE_CONFIG = {'version': '1.0.0', 'name': 'production_ensemble', 'description': 'Pure Solo In-House 14B LoRA [qwen3_14b_inhouse (1.00 plain)] - No Specialists, No Balancing, No Rules', 'probe_gate_min_bacc': 0.35, 'enable_ablation_probe': True, 'ablation_samples_per_class': 20, 'lock_prior_weights': True, 'wpca_max_prompts': 60, 'labels': ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'], 'families': {'gpt': [0, 1, 2, 3, 4], 'gemini': [5, 6, 7, 8], 'claude': [9, 10, 11, 12, 13], 'glm': [15, 16]}, 'required_assets': {'require_pool_parquet': True, 'required_base_models': [{'name': 'qwen3_14b', 'family': 'qwen3', 'hidden_size': 5120, 'num_layers': 40}], 'required_checkpoints': ['qwen3_14b_inhouse'], 'optional_checkpoints': []}, 'model_weights': {'qwen3_14b_inhouse': {'plain_weight': 1.0, 'twin_weight': 0.0}, 'qwen3_14b_p2': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'svc_4view': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'spectral_bank': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'deberta_p2': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'lexical_5v': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'wpca_dp': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'qwen3_8b_claude': {'plain_weight': 0.0, 'twin_weight': 0.0}}, 'main_stack': {'linear_svc_weight': 0.0, 'members': {'lora_qwen3_14b': {'plain_weight': 0.0, 'twin_weight': 0.0, 'twin_clustering': 'groups_085'}, 'deberta_v3_large': {'plain_weight': 0.0, 'twin_weight': 0.0}}}, 'complementary_stack': {'stream_weights': {'lora_and_twins': 1.0, 'lexical': 0.0, 'dp_posteriors': 0.0, 'spectral_bank': 0.0}, 'lora_members': {'qwen3_14b_inhouse': {'plain_weight': 1.0, 'twin_weight': 0.0}, 'lora_qwen3_14b': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'ntl_claude_head': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'deberta_v3_large': {'plain_weight': 0.0, 'twin_weight': 0.0}}, 'twin_resolutions': {'t85': 0.5, 't72': 0.3, 't60': 0.2}, 'dp_weights': {'q3_14b': 0.0, 'q25_14b': 0.0, 'deberta': 0.0, 'claude_8b': 0.0}, 'se_floors': {'qwen3_14b_p2': 0.1, 'qwen25_14b_p2': 0.1, 'deberta_p2': 0.1, 'qwen3_8b_claude': 0.35}, 'prior_scales': {'qwen3_14b_inhouse': 1.0, 'qwen3_14b_p2': 1.0, 'qwen25_14b_p2': 0.85, 'deberta_p2': 0.8, 'qwen3_8b_claude': 0.3}, 'spectral_bank': {'plain_C': 0.04, 'center_C': 0.03, 'center_blend': 0.35}}, 'fusion': {'main_ratio': 0.0, 'comp_ratio': 1.0}, 'postprocess': {'enable_within_family_confidence_gates': False, 'family_gate_strength': 0.0, 'claude_specialist_weight': 0.0, 'gpt_specialist_weight': 0.0, 'gemini_specialist_weight': 0.0, 'glm_specialist_weight': 0.0, 'curly_quote_multiplier': 1.0, 'multi_em_dash_log_penalty': 0.0, 'straight_contr_log_penalty': 0.0, 'curly_contr_log_penalty': 0.0, 'continuous_hw_weight': 0.0, 'cluster_repulsion_strength': 0.0, 'sinkhorn_iterations': 0, 'sinkhorn_strength': 0.0, 'sinkhorn_target_mix': 0.0, 'enable_typobayes': False, 'typobayes_kappa': 24.0, 'typobayes_evidence_scale': 1.2, 'typobayes_evidence_cap': 0.3, 'max_cluster_size': 14}}

"""
Canonical Configuration, Constants & Path Resolution
====================================================
Master canonical label ordering, provider families, simplex helpers,
and Kaggle filesystem asset discovery.
"""

import hashlib
import json
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np

# Canonical 18-Class Labels (Strict order from NTL two-stage LoRA part two)
LABELS = [
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "claude-fable-5",
    "qwen3.8-max",
    "glm-5.3",
    "glm-5.3-flash",
    "kimi-k3",
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

# Master list of highest-error and most confused classes requiring loss upweighting:
# 1: gpt-5.6-luna (60.0% error, easily confused with astra/sol)
# 2: gpt-5.6-sol (75.0% error, heavily confused with astra)
# 3: gpt-5.6-terra (70.0% error, confused with luna/sol)
# 6: gemini-3.6-flash (45.0% error, confused with 3.1-pro)
# 7: gemini-3.7-flash (80.0% error, confused with 3.8-flash & 3.6-flash)
# 8: gemini-3.8-flash (65.0% error, confused with 3.7-flash & 3.6-flash)
# 13: claude-fable-5 (45.0% error, cross-family confusion)
# 14: qwen3.8-max (40.0% error, lowest pool count 4,209)
# 15: glm-5.3 (80.0% error, confused with glm-5.3-flash)
# 16: glm-5.3-flash (80.0% error, confused with glm-5.3)
# 17: kimi-k3 (80.0% error, straight-quote & Chinese frontier confusion)
HARD_CLASS_INDICES = [1, 2, 3, 6, 7, 8, 13, 14, 15, 16, 17]

# Extended specialist groups: includes the 4 provider families PLUS the cross-family
# Straight-Quote / Chinese Frontier confusion cluster (opus-4-8, fable-5, qwen3.8-max, glm-5.3, glm-5.3-flash, kimi-k3)
# and high-confusion sibling sub-clusters!
SPECIALIST_GROUPS = {
    "gpt": [0, 1, 2, 3, 4],
    "gpt_56_triad": [1, 2, 3],  # gpt-5.6-luna vs gpt-5.6-sol vs gpt-5.6-terra
    "gemini": [5, 6, 7, 8],
    "gemini_flash_triad": [6, 7, 8],  # gemini-3.6-flash vs 3.7-flash vs 3.8-flash
    "gemini_37_38_pair": [7, 8],  # Dedicated 1-vs-1 binary tie-breaker for gemini-3.7-flash vs gemini-3.8-flash!
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],  # Dedicated 1-vs-1 binary tie-breaker for glm-5.3 vs glm-5.3-flash!
    "glm_kimi_triad": [15, 16, 17],  # Resolves glm-5.3 <-> glm-5.3-flash <-> kimi-k3 confusion!
    "frontier_and_fable": [10, 13, 14, 15, 16, 17],  # resolves claude-fable-5/opus-4-8 <-> glm-5.3/flash <-> kimi-k3 <-> qwen3.8-max!
    "kimi_gpt55_qwen": [0, 14, 17],  # resolves kimi-k3 <-> gpt-5.5 <-> qwen3.8-max confusion!
}

PRIOR_SCALES = {
    "qwen3_14b_p2": 1.00,
    "qwen3_14b_inhouse": 1.00,
    "qwen25_14b_p2": 0.85,
    "qwen25_14b_inhouse": 0.85,
    "deberta_p2": 0.80,
    "deberta_inhouse": 0.80,
    "qwen3_14b_p1": 0.50,
    "instyle": 0.45,
    "qwen3_8b_claude": 0.30,
    "qwen3_8b_inhouse": 0.30,
}


def align_cols(arr: np.ndarray, src_labels: list) -> np.ndarray:
    """Aligns columns of array to canonical 18-class order."""
    src_list = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    idx = [src_list.index(l) for l in LABELS]
    if arr.ndim == 1:
        return arr[idx]
    elif arr.ndim == 2:
        if arr.shape[0] == len(src_list) and arr.shape[1] != len(src_list):
            return arr[idx, :]
        elif arr.shape[1] == len(src_list):
            return arr[:, idx]
    return arr[..., idx]


def calibrate_model_lp(
    lp: np.ndarray,
    model_name: str = "",
    min_std: float = 1.6,
    max_std: float = 4.2,
) -> np.ndarray:
    """
    1. Normalizes row-wise log-probability std into `[min_std, max_std]` so neither
       over-confident neural heads (LP_std=13.1) nor flat lexical heads (LP_std=0.40)
       distort ensemble vote shares or collapse under Sinkhorn balancing.
    2. Idempotency guard: if `lp` was already calibrated and clipped (e.g. loaded from
       local_eval_cache.npz), returns it without double-scaling.
    """
    if lp is None or len(lp) == 0:
        return lp
    out = np.asarray(lp, dtype=np.float64).copy()

    # Check if `out` is already a calibrated log-softmax matrix from cache
    row_max = np.max(out, axis=1)
    row_min = float(np.min(out))
    row_std = float(out.std(axis=1).mean())
    if np.all(row_max <= 1e-4) and row_min >= -16.5 and (1.4 <= row_std <= max_std + 0.1):
        return log_softmax(out, axis=1)

    if row_std > max_std and row_std > 1e-6:
        out = out * (max_std / row_std)
    elif row_std < min_std and row_std > 1e-6:
        out = out * (min_std / row_std)
    out = log_softmax(out, axis=1)
    out = np.maximum(out, -16.0)
    return log_softmax(out, axis=1)


def _text_hash(text: str) -> str:
    """
    Computes a canonical SHA-1 hash of normalized text for zero-leakage deduplication.

    Why Strict Normalization Matters:
    --------------------------------
    Subtle variations (different Unicode quotes, leading/trailing markdown asterisks,
    differing whitespace) could cause a training passage to evade exact string matching.
    We apply:
    1. Unicode NFKC normalization (replaces typographic quotes, fullwidth chars).
    2. Stripping enclosing markdown bold (**text** -> text).
    3. Collapsing all multi-whitespace and newlines into single spaces.
    This guarantees 0% exact or near-exact leakage between training and holdout!
    """
    s = unicodedata.normalize("NFKC", str(text or "")).strip()
    s = re.sub(r"^\*\*(.*)\*\*$", r"\1", s).strip()
    norm = " ".join(s.split())
    return hashlib.sha1(norm.encode("utf-8", errors="ignore")).hexdigest()


def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)


def resolve_dataset_paths(input_dir: Path | None = None) -> tuple[Path, Path, Path | None]:
    """Resolves train.csv, test.csv, and pool_v4.parquet across local and Kaggle environments."""
    if input_dir is None:
        input_dir = Path("/kaggle/input")
        if not input_dir.exists():
            input_dir = Path(".")

    candidates = [
        input_dir,
        input_dir / "competitions" / "name-that-llm",
        input_dir / "name-that-llm",
        Path("local_data"),
        Path("."),
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / "train.csv").exists() and train_path is None:
            train_path = c / "train.csv"
        if (c / "test.csv").exists() and test_path is None:
            test_path = c / "test.csv"

    if train_path is None:
        train_path = Path("local_data") / "train.csv"
    if test_path is None:
        test_path = Path("local_data") / "test.csv"

    pool_files = resolve_all_pool_files(input_dir)
    pool_file = pool_files[0] if pool_files else None

    return train_path, test_path, pool_file


def resolve_all_pool_files(input_dir: Path | None = None) -> list[Path]:
    """Discovers all pretraining/unlabeled pool parquet files in priority order.

    If the consolidated pool_120k.parquet exists, returns it directly as it already
    subsumes all individual pool sources (nlm_matched_clean, pool_v4, nlm_pool_v5, nlm_pool_v2).
    """
    if input_dir is None:
        input_dir = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")

    # If the consolidated 120k pool is present, use it directly to eliminate redundant I/O
    for p in input_dir.rglob("pool_120k.parquet"):
        return [p]

    pool_files: list[Path] = []
    pool_patterns = [
        "nlm_matched_clean.parquet",
        "pool_v4.parquet",
        "nlm_pool_v5.parquet",
        "nlm_pool_v2.parquet",
    ]
    for pat in pool_patterns:
        for p in input_dir.rglob(pat):
            if p not in pool_files:
                pool_files.append(p)
    return pool_files


def find_base_model(input_dir: Path, model_family: str, hidden_sz: int, num_layers: int) -> Path | None:
    """Discovers base model directories from Hugging Face config.json inspection."""
    for cfg_p in input_dir.rglob("config.json"):
        try:
            with open(cfg_p, "r", encoding="utf-8") as f:
                cfg = json.load(f)
            m_type = str(cfg.get("model_type", "")).lower()
            h_dim = cfg.get("hidden_size", 0)
            n_lay = cfg.get("num_hidden_layers", 0)
            if model_family.lower() in m_type and h_dim == hidden_sz and n_lay == num_layers:
                return cfg_p.parent
        except Exception:
            continue
    return None


def _inspect_checkpoint_dim_and_keep(ckpt_dir: Path) -> tuple[int, int, str]:
    """Returns (head_feature_dim, keep_layers, backbone) for checkpoint directory without full model loading."""
    dim = 0
    keep = 0
    backbone = ""
    meta_p = ckpt_dir / "meta.json"
    if meta_p.exists():
        try:
            with open(meta_p, "r", encoding="utf-8") as f:
                m = json.load(f)
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", 0))
            backbone = str(m.get("backbone", cfg.get("backbone", ""))).lower()
        except Exception:
            pass

    heads_p = ckpt_dir / "heads.safetensors"
    if heads_p.exists():
        try:
            from safetensors import safe_open

            with safe_open(str(heads_p), framework="pt", device="cpu") as f:
                for k in f.keys():
                    shape = f.get_tensor(k).shape
                    if len(shape) == 2 and (shape[0] in (18, 5, 4, 2) or shape[1] in (18, 5, 4, 2)):
                        dim = int(shape[1] if shape[0] in (18, 5, 4, 2) else shape[0])
                        break
        except Exception:
            pass
    elif (ckpt_dir / "heads.pt").exists():
        try:
            import torch

            sd = torch.load(ckpt_dir / "heads.pt", map_location="cpu", weights_only=False)
            if isinstance(sd, dict):
                for k, v in sd.items():
                    if hasattr(v, "shape") and len(v.shape) == 2 and (v.shape[0] in (18, 5, 4, 2) or v.shape[1] in (18, 5, 4, 2)):
                        dim = int(v.shape[1] if v.shape[0] in (18, 5, 4, 2) else v.shape[0])
                        break
        except Exception:
            pass
    return dim, keep, backbone


def find_checkpoints(input_dir: Path) -> dict[str, Path]:
    """Finds all fine-tuned LoRA and head checkpoints across safetensors and PyTorch formats."""
    ckpts = {}

    # 1. Check for heads.safetensors and heads.pt files
    head_candidates = list(input_dir.rglob("heads.safetensors")) + list(input_dir.rglob("heads.pt"))
    for p in head_candidates:
        p_str = str(p).lower().replace("\\", "/")
        is_fold = "_f0" in p_str or "_f1" in p_str or "_f2" in p_str or "_f3" in p_str or "_f4" in p_str

        target_family = ""
        meta_p = p.parent / "meta.json"
        if meta_p.exists():
            try:
                with open(meta_p, "r", encoding="utf-8") as f:
                    m_data = json.load(f)
                target_family = str(m_data.get("target_family", "")).lower()
            except Exception:
                pass

        # Critical routing guard: check family specialists before general inhouse / lora-peft-training
        if target_family == "gpt" or any(t in p_str for t in ("export_gpt", "gpt_specialist", "gpt_spec", "qwen3_14b_gpt")):
            ckpts["qwen3_14b_gpt"] = p.parent
            continue
        elif target_family == "gemini" or any(
            t in p_str for t in ("export_gemini", "gemini_specialist", "gemini_spec", "qwen3_14b_gemini")
        ):
            ckpts["qwen3_14b_gemini"] = p.parent
            continue
        elif target_family == "glm" or any(t in p_str for t in ("export_glm", "glm_specialist", "glm_spec", "qwen3_14b_glm")):
            ckpts["qwen3_14b_glm"] = p.parent
            continue

        if "lora_qwen3_14b" in p_str or "qwen3_14b_part2" in p_str or ("14b" in p_str and "part2" in p_str and "qwen3" in p_str):
            ckpts["qwen3_14b_p2"] = p.parent
        elif "lora_qwen25_14b" in p_str or "qwen25_14b_part2" in p_str or ("14b" in p_str and "part2" in p_str and "qwen2" in p_str):
            ckpts["qwen25_14b_p2"] = p.parent
        elif "q14v5" in p_str:
            if "qwen3_14b_v5" not in ckpts or (not is_fold):
                ckpts["qwen3_14b_v5"] = p.parent
        elif "ts_q3b14" in p_str:
            if "qwen3_14b_ts" not in ckpts or (not is_fold):
                ckpts["qwen3_14b_ts"] = p.parent
        elif "lora14b_v1" in p_str:
            ckpts["qwen3_14b_p1"] = p.parent
        elif any(
            token in p_str
            for token in ("inhouse", "ntl-inhouse", "export_lora24", "export_champion", "lora-peft-training", "peft_training")
        ):
            dim, keep, backbone = _inspect_checkpoint_dim_and_keep(p.parent)
            if "2.5" in backbone or "qwen25" in backbone or (dim == 40960 and (keep >= 30 or "25" in p_str or "qwen25" in p_str)):
                ckpts["qwen25_14b_inhouse"] = p.parent
            elif dim == 40960 or "14b" in p_str or "14b" in backbone:
                ckpts["qwen3_14b_inhouse"] = p.parent
            elif dim == 8192 or "deberta" in p_str or "deberta" in backbone:
                ckpts["deberta_inhouse"] = p.parent
            else:
                ckpts["qwen3_8b_inhouse"] = p.parent
        elif "q8v5" in p_str:
            if "qwen3_8b_v5" not in ckpts or (not is_fold):
                ckpts["qwen3_8b_v5"] = p.parent
        elif "ts_q3b8" in p_str or "ts8_s7" in p_str or "l24x_q3b8" in p_str:
            if "qwen3_8b_ts" not in ckpts or (not is_fold):
                ckpts["qwen3_8b_ts"] = p.parent
        elif "ll8v5" in p_str:
            if "llama3_8b_v5" not in ckpts or (not is_fold):
                ckpts["llama3_8b_v5"] = p.parent
        elif "ts_llama8" in p_str:
            if "llama3_8b_ts" not in ckpts or (not is_fold):
                ckpts["llama3_8b_ts"] = p.parent
        elif "lora8b_v1" in p_str:
            ckpts["qwen3_8b_p1"] = p.parent
        elif "ts_q25_7b" in p_str or "l24x_q25_7b" in p_str:
            if "qwen25_7b_ts" not in ckpts or (not is_fold):
                ckpts["qwen25_7b_ts"] = p.parent
        elif "instyle" in p_str:
            ckpts["instyle"] = p.parent

    # 2. Check for Claude specialist head
    for p in input_dir.rglob("claude_head.safetensors"):
        ckpts["qwen3_8b_claude"] = p.parent
        break
    if "qwen3_8b_claude" not in ckpts:
        for p in input_dir.rglob("readouts.npz"):
            if "claude" in str(p).lower():
                ckpts["qwen3_8b_claude"] = p.parent
                break

    # 3. Check for DeBERTa sequence classification head
    for p in list(input_dir.rglob("head.safetensors")) + list(input_dir.rglob("heads.safetensors")):
        if "deberta" in str(p).lower():
            ckpts["deberta_p2"] = p.parent
            break

    # 4. Fallback search for instyle
    if "instyle" not in ckpts:
        for p in input_dir.rglob("*"):
            p_s = str(p).lower().replace("\\", "/")
            if "instyle" in p_s and (
                (p / "adapter_model.safetensors").exists()
                or (p / "adapter" / "adapter_model.safetensors").exists()
                or (p / "heads.safetensors").exists()
                or (p / "heads.pt").exists()
            ):
                ckpts["instyle"] = p
                break

    # 5. Fallback search for q14v5 / ts_q3b14 adapter
    if "qwen3_14b_ts" not in ckpts:
        for p in input_dir.rglob("adapter_model.safetensors"):
            p_s = str(p).lower().replace("\\", "/")
            if "q14v5" in p_s or "ts_q3b14" in p_s:
                ckpts["qwen3_14b_ts"] = p.parent
                break

    return ckpts


def find_fold_adapters(input_dir: Path | None, prefix: str) -> list[Path]:
    """Scans input_dir for all adapter/head directories matching prefix, prefix_f*, prefix_fold*, prefix-f*."""
    if input_dir is None:
        input_dir = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")

    matches: set[Path] = set()
    prefix_lower = prefix.lower()

    candidate_files = (
        list(input_dir.rglob("heads.safetensors")) + list(input_dir.rglob("heads.pt")) + list(input_dir.rglob("adapter_model.safetensors"))
    )
    for f in candidate_files:
        p = f.parent
        p_name = p.name.lower()
        if p_name in ("adapter", "checkpoint"):
            p = p.parent
            p_name = p.name.lower()

        p_str = str(p).lower().replace("\\", "/")
        if prefix_lower in p_name or prefix_lower in p_str:
            # Check pattern: exact match or prefix followed by _ or - or fold
            tokens = p_name.replace("-", "_").split("_")
            matched = False
            for t in tokens:
                if t == prefix_lower or t.startswith(prefix_lower):
                    matched = True
                    break
            if not matched and prefix_lower in p_str:
                matched = True

            if matched:
                matches.add(p)

    return sorted(list(matches), key=lambda x: str(x).lower())


def validate_production_assets(
    input_dir: Path,
    train_path: Path,
    test_path: Path,
    pool_files: list[Path],
    ckpts: dict[str, Path],
    cfg: dict[str, Any] | None = None,
) -> dict[str, Path]:
    """
    Fail-closed pre-flight validation for production inference runs (when NTL_SMOKE != '1').
    Raises FileNotFoundError immediately at startup if any required dataset, pool parquet,
    base model trunk, or checkpoint is missing.
    Returns a dict of verified base_model paths keyed by name.
    """
    if cfg is None:
        cfg = {}
    req = cfg.get("required_assets", {})
    missing: list[str] = []

    # 1. Verify competition CSVs
    if not train_path or not Path(train_path).exists():
        missing.append(f"Competition train.csv not found at: {train_path}")
    if not test_path or not Path(test_path).exists():
        missing.append(f"Competition test.csv not found at: {test_path}")

    # 2. Verify public pool parquet(s)
    require_pool = req.get("require_pool_parquet", True)
    valid_pools = [p for p in pool_files if p is not None and Path(p).exists()]
    if require_pool and len(valid_pools) == 0:
        missing.append(f"No public pool parquet (pool_v4.parquet / nlm_pool_v5.parquet) found under {input_dir}")

    # 3. Verify all required base model trunks
    default_bases = [
        {"name": "qwen3_14b", "family": "qwen3", "hidden_size": 5120, "num_layers": 40},
        {"name": "qwen3_8b", "family": "qwen3", "hidden_size": 4096, "num_layers": 36},
        {"name": "deberta_v3_large", "family": "deberta", "hidden_size": 1024, "num_layers": 24},
    ]
    base_specs = req.get("required_base_models", default_bases)
    verified_bases: dict[str, Path] = {}
    for b_spec in base_specs:
        b_path = find_base_model(
            input_dir,
            b_spec["family"],
            int(b_spec["hidden_size"]),
            int(b_spec["num_layers"]),
        )
        if b_path is None or not Path(b_path).exists():
            missing.append(
                f"Required base model '{b_spec['name']}' (family={b_spec['family']}, "
                f"hidden={b_spec['hidden_size']}, layers={b_spec['num_layers']}) not found under {input_dir}"
            )
        else:
            verified_bases[b_spec["name"]] = b_path

    # 4. Verify all required fine-tuned checkpoints
    default_req_ckpts = ["qwen3_14b_p2", "deberta_p2", "qwen3_8b_claude"]
    req_ckpts = req.get("required_checkpoints", default_req_ckpts)
    for ck_name in req_ckpts:
        if ck_name not in ckpts or not Path(ckpts[ck_name]).exists():
            missing.append(f"Required checkpoint '{ck_name}' not found under {input_dir} (discovered: {list(ckpts.keys())})")

    if missing:
        error_msg = "[PRE-FLIGHT ASSET AUDIT FAILED] Missing required inference assets:\n  - " + "\n  - ".join(missing)
        raise FileNotFoundError(error_msg)

    print(
        f"[Pre-Flight Audit PASSED] Verified {len(valid_pools)} pool file(s), "
        f"{len(verified_bases)} base trunks, and {len(ckpts)} checkpoints: {list(ckpts.keys())}"
    )
    return verified_bases


In [ ]:
# === MODULE: src/namethatllm/wpca_gaussian.py ===
"""
Full-Rank wPCA & ExactAuthorPosterior (2^18 Bitmask DP)
======================================================
Two-way variance decomposition, Ledoit-Wolf shrinkage floors,
crash-proof cluster splitting (max_size <= 14), and exact forward-backward
bitmask DP author posteriors under the unique-authorship constraint.
"""

from functools import lru_cache

import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from scipy.special import logsumexp
from sklearn.covariance import ledoit_wolf_shrinkage


# =============================================================================
# 1. TWO-WAY ANOVA DECOMPOSITION & COVARIANCE SHRINKAGE
# =============================================================================


def _two_way(
    Z: np.ndarray, y: np.ndarray, pinv: np.ndarray, G: int, K_classes: int = 18, iters: int = 4
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Iterative Two-Way ANOVA variance decomposition.

    Mathematical Formulation:
    -------------------------
    Each sample embedding Z_{i} is modeled as:
        Z_{i} = mu_{y_i} + P_{g_i} + epsilon_{i}
    where:
    - mu_{k} is the fixed effect of LLM author k (the pure writing style centroid)
    - P_{g} is the random effect of prompt topic g (the prompt-specific content shift)
    - epsilon_{i} is residual noise

    By alternating:
    1. Estimate mu_{k} by averaging residuals (Z - P_{g}) for each class k
    2. Estimate P_{g} by averaging residuals (Z - mu_{k}) for each prompt group g
    for 4 iterations, we completely decouple the author's stylistic signature
    from the topic/subject of the prompt!
    """
    N, d = Z.shape
    mu = np.zeros((K_classes, d), dtype=np.float64)
    P = np.zeros((G, d), dtype=np.float64)
    ng = np.bincount(pinv, minlength=G).clip(min=1)

    for _ in range(iters):
        # Step 1: Subtract current prompt effects, update author centroids mu
        R = Z - P[pinv]
        for k in range(K_classes):
            idx = y == k
            if idx.any():
                mu[k] = R[idx].mean(axis=0)

        # Step 2: Subtract updated author centroids, update prompt effects P
        R = Z - mu[y]
        P[:] = 0.0
        np.add.at(P, pinv, R)
        P /= ng[:, None]

    return mu, P, ng


def _shrunk(S: np.ndarray, alpha: float) -> np.ndarray:
    """
    Shrinks covariance matrix S towards a spherical target:
        S_shrunk = (1 - alpha) * S + alpha * (trace(S)/d) * I
    Prevents degenerate zero-variance dimensions in high dimensions.
    """
    d = S.shape[0]
    tr = max(float(np.trace(S)) / d, 1e-6)
    return (1.0 - alpha) * S + alpha * tr * np.eye(d)


def fit_shared_gaussian_lowram(
    X_tr_f32: np.ndarray, y: np.ndarray, prompts: np.ndarray, d_target: int = 256, se_floor: float = 0.10
) -> dict:
    """
    Fits shared Gaussian whitener with within-prompt centering and Ledoit-Wolf shrinkage floor.
    Returns unsupervised top-64 within-prompt principal components V_top64 (zero label leakage!).
    """
    N, D = X_tr_f32.shape
    pu, pinv = np.unique(prompts, return_inverse=True)
    G = len(pu)

    mx = X_tr_f32.mean(axis=0, dtype=np.float64).astype(np.float32)
    sx = X_tr_f32.std(axis=0, dtype=np.float64).astype(np.float32)
    sx = np.where(sx > 1e-6, sx, 1.0).astype(np.float32)

    # Standardize and subtract prompt means in-place (Within-Prompt Whitened PCA!)
    R = (X_tr_f32 - mx) / sx
    Pm = np.zeros((G, D), dtype=np.float32)
    np.add.at(Pm, pinv, R)
    Pm /= np.bincount(pinv, minlength=G)[:, None].clip(min=1).astype(np.float32)
    R -= Pm[pinv]
    del Pm

    d = int(min(d_target, max(N - G - 1, 1), D))
    Gram = (R @ R.T).astype(np.float64)
    lam_r, U = np.linalg.eigh(Gram)
    idx_top = np.argsort(lam_r)[::-1][:d]
    lam_top = np.clip(lam_r[idx_top], 1e-10, None)
    U_f32 = U[:, idx_top].astype(np.float32)
    V_f32 = (R.T @ U_f32) / np.sqrt(lam_top).astype(np.float32)[None, :]
    del R, Gram, U, U_f32

    # Unsupervised within-prompt top-64 principal components (zero label leakage)
    V_top64 = V_f32[:, : min(64, d)].copy()

    Z = (((X_tr_f32 - mx) / sx) @ V_f32).astype(np.float64)

    mu, P, ng = _two_way(Z, y, pinv, G, K_classes=K, iters=4)
    E = Z - mu[y] - P[pinv]
    dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = (E.T @ E) / dof
    ae = max(float(ledoit_wolf_shrinkage(E, assume_centered=True)), se_floor)
    Se = _shrunk(Se_raw, ae)

    Pc = P - P.mean(0)
    Sp_raw = (Pc.T @ Pc) / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = float(ledoit_wolf_shrinkage(Pc, assume_centered=True))
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se))
    Sp = (1.0 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)

    L = None
    for jitter in (0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2):
        try:
            target_matrix = Se if jitter == 0.0 else Se + jitter * np.eye(d)
            L = np.linalg.cholesky(target_matrix)
            break
        except np.linalg.LinAlgError:
            continue
    if L is None:
        eigvals, eigvecs = np.linalg.eigh(Se)
        eigvals = np.clip(eigvals, 1e-6, None)
        L = eigvecs @ np.diag(np.sqrt(eigvals))
    Li = np.linalg.inv(L)
    M = 0.5 * ((Li @ Sp @ Li.T) + (Li @ Sp @ Li.T).T)
    lam, Q = np.linalg.eigh(M)
    lam = np.clip(lam, 1e-4, None)
    A = Q.T @ Li
    W_f32 = (V_f32 @ A.T.astype(np.float32)).astype(np.float32)
    m_whitened = mu @ A.T

    return {"mx": mx, "sx": sx, "W": W_f32, "m": m_whitened, "lam": lam, "V_top64": V_top64, "T1": 2.40, "Tg": 2.35, "ae": ae}


def transform_lowram(X_f32: np.ndarray, prm: dict, batch_rows: int = 1000) -> np.ndarray:
    """Projects representations into whitened space."""
    N = len(X_f32)
    d = prm["W"].shape[1]
    U_out = np.empty((N, d), dtype=np.float64)
    mx, sx, W = prm["mx"], prm["sx"], prm["W"]
    for s in range(0, N, batch_rows):
        chunk = ((X_f32[s : s + batch_rows] - mx) / sx) @ W
        U_out[s : s + batch_rows] = chunk.astype(np.float64)
    return U_out


def split_oversized_clusters_dist(D: np.ndarray | None, groups: np.ndarray, max_cluster_size: int = 14) -> np.ndarray:
    """Recursively splits clusters with size > max_cluster_size to prevent bitmask DP overflow."""
    if len(groups) < 2:
        return groups.copy()
    new_groups = np.zeros(len(groups), dtype=np.int64)
    next_id = 1
    for g in np.unique(groups):
        rows = np.flatnonzero(groups == g)
        if len(rows) <= max_cluster_size:
            new_groups[rows] = next_id
            next_id += 1
        else:
            if D is None or D.shape != (len(groups), len(groups)):
                for start in range(0, len(rows), max_cluster_size):
                    new_groups[rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
                continue
            n_sub = int(np.ceil(len(rows) / max_cluster_size))
            D_sub = D[np.ix_(rows, rows)].copy()
            np.fill_diagonal(D_sub, 0.0)
            D_sub = np.clip(0.5 * (D_sub + D_sub.T), 0.0, None)
            d_condensed = squareform(D_sub, checks=False)
            sub_labels = fcluster(linkage(d_condensed, method="average"), t=n_sub, criterion="maxclust")
            for sg in np.unique(sub_labels):
                s_rows = rows[sub_labels == sg]
                for start in range(0, len(s_rows), max_cluster_size):
                    new_groups[s_rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
    return new_groups


@lru_cache(maxsize=4)
def _bitmask_levels(k: int = 18):
    """Precomputes bitmasks partitioned by population count (number of set bits)."""
    masks = np.arange(1 << k, dtype=np.int64)
    counts = np.fromiter((int(x).bit_count() for x in masks), dtype=np.uint8, count=len(masks))
    return tuple(masks[counts == n] for n in range(k + 1))


class ExactAuthorPosterior:
    """
    Exact Gaussian Posteriors under the Unique-Authorship Constraint via 2^18 Bitmask DP.

    Domain Insight & Problem Setup:
    -------------------------------
    When n completions (e.g. n = 4) belong to the same prompt cluster, each completion
    must come from a DIFFERENT LLM author. This is an injective bipartite matching
    between the n observed completions and the K = 18 potential LLM authors.
    Evaluating all possible injective assignments naively requires:
        18 * 17 * ... * (18 - n + 1) assignments (up to 1.76 * 10^11 permutations for n=14!)

    Dynamic Programming Solution (2^18 Bitmask DP):
    -----------------------------------------------
    By representing the set of assigned authors as a binary bitmask of length K = 18
    (total 2^18 = 262,144 states), we can compute the EXACT marginal posterior using
    an efficient forward-backward message passing algorithm:
    1. Forward Pass: Computes forward[mask], the cumulative likelihood of matching
       the first `count` passages to the subset of authors represented by `mask`.
    2. Terminal Prompt Correction: Evaluates the Gaussian prompt coupling effect:
       residual = sum(U) - sum(mu_authors), adding the quadratic coupling term.
    3. Backward Pass: Propagates backward messages from the terminal level down to the base.
    4. Exact Marginalization: Combines forward, unary passage likelihood A[row, author],
       and backward terms to yield the exact marginal probability P(author | all n texts in cluster).
    Runs in milliseconds!
    """

    def __init__(self, m: np.ndarray, lam: np.ndarray):
        self.m = np.asarray(m, dtype=np.float64)
        self.lam = np.asarray(lam, dtype=np.float64)
        self.K, self.d = self.m.shape
        self.levels = _bitmask_levels(self.K)
        self._subset_means = {}

    def single(self, u: np.ndarray, temperature: float = 2.40) -> np.ndarray:
        diff = u[None, :] - self.m
        dist = np.sum((diff * diff) / (1.0 + self.lam)[None, :], axis=1)
        log_p = -0.5 * dist / max(float(temperature), 1e-4)
        return log_p - logsumexp(log_p)

    def group(self, U: np.ndarray, temperature: float = 2.35) -> np.ndarray:
        U = np.asarray(U, dtype=np.float64)
        n = len(U)
        if n <= 1 or n > self.K:
            return np.vstack([self.single(U[i], temperature=temperature) for i in range(n)])

        if n not in self._subset_means:
            if len(self._subset_means) >= 3:
                self._subset_means.clear()
            masks = self.levels[n]
            membership = ((masks[:, None] >> np.arange(self.K)) & 1).astype(np.float32)
            self._subset_means[n] = membership @ self.m.astype(np.float32)

        terminal = self.levels[n]
        gamma = self.lam / (1.0 + n * self.lam)
        residual = U.sum(axis=0, keepdims=True) - self._subset_means[n].astype(np.float64)
        correction = 0.5 * np.einsum("nd,d,nd->n", residual, gamma, residual) / max(float(temperature), 1e-4)
        correction -= correction.max()

        # Passage-specific unary log-likelihoods A[i, author]
        A = -0.5 * np.sum((U[:, None, :] - self.m[None, :, :]) ** 2, axis=2) / max(float(temperature), 1e-4)
        A -= A.max(axis=1, keepdims=True)

        forward = np.full(1 << self.K, -np.inf, dtype=np.float64)
        forward[0] = 0.0
        for count in range(1, n + 1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) != 0]
                forward[chosen] = np.logaddexp(forward[chosen], forward[chosen ^ bit] + A[count - 1, author])

        backward = np.full_like(forward, -np.inf)
        backward[terminal] = correction
        for count in range(n - 1, -1, -1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                backward[chosen] = np.logaddexp(backward[chosen], A[count, author] + backward[chosen | bit])

        result = np.empty((n, self.K), dtype=np.float64)
        for row in range(n):
            masks = self.levels[row]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                result[row, author] = logsumexp(forward[chosen] + A[row, author] + backward[chosen | bit]) - backward[0]

        result -= logsumexp(result, axis=1, keepdims=True)
        return result


def scores_exact_dp(U_test: np.ndarray, groups: np.ndarray, prm: dict, D_dist: np.ndarray | None = None) -> np.ndarray:
    """Wraps ExactAuthorPosterior with oversized cluster splitting guardrail."""
    N = len(U_test)
    if N == 0:
        return np.zeros((0, K), dtype=np.float64)
    if N == 1:
        solver = ExactAuthorPosterior(prm["m"], prm["lam"])
        return solver.single(U_test[0], temperature=prm.get("T1", 2.40))[None, :]

    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_input_groups = inv_g + 1

    if D_dist is None or D_dist.shape != (N, N):
        U_c = U_test - U_test.mean(axis=0, keepdims=True)
        U_n = U_c / np.linalg.norm(U_c, axis=1, keepdims=True).clip(min=1e-6)
        D_dist = np.clip(1.0 - (U_n @ U_n.T), 0.0, None)
        np.fill_diagonal(D_dist, 0.0)

    safe_groups = split_oversized_clusters_dist(D_dist, safe_input_groups, max_cluster_size=14)
    solver = ExactAuthorPosterior(prm["m"], prm["lam"])
    T1 = prm.get("T1", 2.40)
    Tg = prm.get("Tg", 2.35)

    lp_dp = np.zeros((N, K), dtype=np.float64)
    for g in np.unique(safe_groups):
        idx_g = np.where(safe_groups == g)[0]
        if len(idx_g) == 1:
            lp_dp[idx_g] = solver.single(U_test[idx_g[0]], temperature=T1)
        else:
            lp_dp[idx_g] = solver.group(U_test[idx_g], temperature=Tg)
    return lp_dp


In [ ]:
# === MODULE: src/namethatllm/spectral_bank.py ===
"""
Leak-Free Prior-Scaled 384-Dim Joint Spectral Bank
=================================================
Constructs full-rank cross-backbone spectral representations from unsupervised
prompt-centered principal components V_top64, scaled by out-of-sample prior scales.
"""

import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold



def make_spectral_blocks(Z_tr: np.ndarray, Z_te: np.ndarray, res: dict, prior_scale: float) -> tuple[np.ndarray, np.ndarray]:
    """
    Constructs unit-normalized spectral feature blocks projected onto unsupervised principal components.

    Why This Works (Zero Label Leakage):
    -----------------------------------
    V_top64 is computed purely via within-prompt covariance decomposition (no model labels are used).
    Projecting representations onto V_top64 isolates the 64 highest-variance directions of stylistic
    variation across prompts, regularized by the reader's out-of-sample prior scale.
    """
    mx, sx, V64 = res["mx"], res["sx"], res["V_top64"]
    S_tr = (((Z_tr - mx) / sx) @ V64).astype(np.float32)
    S_te = (((Z_te - mx) / sx) @ V64).astype(np.float32)
    S_tr = (S_tr / np.linalg.norm(S_tr, axis=1, keepdims=True).clip(min=1e-6)) * float(prior_scale)
    S_te = (S_te / np.linalg.norm(S_te, axis=1, keepdims=True).clip(min=1e-6)) * float(prior_scale)
    return S_tr, S_te


def _resolve_spectral_groups(
    X_te: np.ndarray,
    D_tfidf: np.ndarray | None,
    D_neural_sum: np.ndarray | None,
    D_neural_weight_total: float,
    max_cluster_size: int = 14,
    groups_override: np.ndarray | None = None,
) -> np.ndarray:
    """
    Identifies test prompt clusters by blending lexical and neural cosine distances:
        D_hybrid = 0.55 * D_tfidf + 0.45 * D_neural
    Clusters at average-linkage distance threshold 0.65, splitting clusters > 14.
    """
    N_te = len(X_te)
    if N_te < 2:
        return np.ones(N_te, dtype=int)
    if groups_override is not None and len(groups_override) == N_te:
        _, inv = np.unique(np.asarray(groups_override, dtype=str), return_inverse=True)
        return split_oversized_clusters_dist(None, inv + 1, max_cluster_size=max_cluster_size)
    if D_tfidf is not None and D_tfidf.shape == (N_te, N_te):
        if D_neural_sum is not None and D_neural_weight_total > 0 and D_neural_sum.shape == (N_te, N_te):
            D_neural_final = D_neural_sum / max(1e-6, D_neural_weight_total)
            D_hybrid_final = 0.55 * D_tfidf + 0.45 * D_neural_final
        else:
            D_hybrid_final = D_tfidf.copy()
    else:
        X_c = X_te - X_te.mean(axis=0, keepdims=True)
        X_n = X_c / np.linalg.norm(X_c, axis=1, keepdims=True).clip(min=1e-6)
        D_hybrid_final = np.clip(1.0 - (X_n @ X_n.T), 0.0, None)
        np.fill_diagonal(D_hybrid_final, 0.0)
    g65 = fcluster(linkage(squareform(D_hybrid_final, checks=False), method="average"), t=0.65, criterion="distance")
    return split_oversized_clusters_dist(D_hybrid_final, g65, max_cluster_size=max_cluster_size)


def fit_joint_spectral_bank(
    spectral_tr_blocks: list[np.ndarray],
    spectral_te_blocks: list[np.ndarray],
    y_sub: np.ndarray,
    prompts_sub: np.ndarray,
    D_tfidf: np.ndarray | None,
    D_neural_sum: np.ndarray | None,
    D_neural_weight_total: float,
    N_test: int,
    plain_C: float = 0.04,
    center_C: float = 0.03,
    center_blend: float = 0.35,
    max_cluster_size: int = 14,
    groups_override: np.ndarray | None = None,
    compute_oof: bool = True,
) -> tuple[np.ndarray, float]:
    """
    Fits plain and prompt-centered logistic regressions on stacked unsupervised spectral features.
    Returns:
    - P_spectral: (N_test, K) predicted probabilities
    - oof_bacc: 5-fold GroupKFold balanced accuracy
    """
    if len(spectral_tr_blocks) == 0:
        return np.ones((N_test, K), dtype=np.float64) / float(K), 0.0

    X_spec_tr = np.hstack(spectral_tr_blocks)
    X_spec_te = np.hstack(spectral_te_blocks)

    oof_bacc = 0.0
    if compute_oof:
        # 5-fold GroupKFold OOF Balanced Accuracy (zero label leakage)
        gkf = GroupKFold(n_splits=5)
        oof_preds = np.zeros(len(y_sub), dtype=int)
        for tr_idx, val_idx in gkf.split(X_spec_tr, y_sub, groups=prompts_sub):
            clf_fold = LogisticRegression(C=plain_C, max_iter=2000, random_state=42)
            clf_fold.fit(X_spec_tr[tr_idx], y_sub[tr_idx])
            oof_preds[val_idx] = clf_fold.predict(X_spec_tr[val_idx])
        oof_bacc = float(balanced_accuracy_score(y_sub, oof_preds))

    # Plain Logistic Regression
    clf_plain = LogisticRegression(C=plain_C, max_iter=2000, random_state=42)
    clf_plain.fit(X_spec_tr, y_sub)
    lp_spec_plain = clf_plain.predict_log_proba(X_spec_te)
    if hasattr(clf_plain, "classes_"):
        lp_spec_plain = align_cols(lp_spec_plain, [LABELS[c] for c in clf_plain.classes_])

    # Prompt-Centered Logistic Regression
    X_spec_tr_c = X_spec_tr.copy()
    for p in np.unique(prompts_sub):
        mask = prompts_sub == p
        X_spec_tr_c[mask] -= X_spec_tr_c[mask].mean(axis=0, keepdims=True)
    clf_center = LogisticRegression(C=center_C, max_iter=2000, random_state=42)
    clf_center.fit(X_spec_tr_c, y_sub)

    # Prompt-centered test features
    X_spec_te_c = X_spec_te.copy()
    safe_groups = _resolve_spectral_groups(
        X_spec_te, D_tfidf, D_neural_sum, D_neural_weight_total, max_cluster_size=max_cluster_size, groups_override=groups_override
    )

    if N_test >= 2 and len(np.unique(safe_groups)) > 0:
        for g in np.unique(safe_groups):
            mask = safe_groups == g
            if mask.sum() > 1:
                X_spec_te_c[mask] -= X_spec_te_c[mask].mean(axis=0, keepdims=True)
            else:
                X_spec_te_c[mask] = 0.0
    else:
        X_spec_te_c = np.zeros_like(X_spec_te)

    lp_spec_center = clf_center.predict_log_proba(X_spec_te_c)
    if hasattr(clf_center, "classes_"):
        lp_spec_center = align_cols(lp_spec_center, [LABELS[c] for c in clf_center.classes_])

    lp_spec_joint = lp_spec_plain + center_blend * lp_spec_center
    P_spectral = softmax(lp_spec_joint, axis=1)

    return P_spectral, oof_bacc


def fit_within_family_spectral_gates(
    spectral_tr_blocks: list[np.ndarray],
    spectral_te_blocks: list[np.ndarray],
    y_sub: np.ndarray,
    prompts_sub: np.ndarray,
    D_tfidf: np.ndarray | None,
    D_neural_sum: np.ndarray | None,
    D_neural_weight_total: float,
    N_test: int,
    max_cluster_size: int = 14,
    hw_tr: np.ndarray | None = None,
    hw_te: np.ndarray | None = None,
    groups_override: np.ndarray | None = None,
) -> dict[str, np.ndarray]:
    """
    Fits plain + prompt-centered LogisticRegression sibling specialist gates
    on the stacked cross-backbone spectral features for each provider family
    ('gpt', 'gemini', 'claude', 'glm').
    Returns dict mapping family_name -> (N_test, len(fam_cols)) log-probabilities.
    """
    if len(spectral_tr_blocks) == 0 or N_test == 0:
        return {}
    X_tr = np.hstack(spectral_tr_blocks)
    X_te = np.hstack(spectral_te_blocks)

    if hw_tr is not None and hw_te is not None and len(hw_tr) == len(X_tr) and len(hw_te) == len(X_te):
        hw_tr_n = (hw_tr / np.linalg.norm(hw_tr, axis=1, keepdims=True).clip(min=1e-6)) * 0.35
        hw_te_n = (hw_te / np.linalg.norm(hw_te, axis=1, keepdims=True).clip(min=1e-6)) * 0.35
        X_tr = np.hstack([X_tr, hw_tr_n])
        X_te = np.hstack([X_te, hw_te_n])

    X_tr_c = X_tr.copy()
    for p in np.unique(prompts_sub):
        m = prompts_sub == p
        if m.sum() > 1:
            X_tr_c[m] -= X_tr_c[m].mean(axis=0, keepdims=True)

    X_te_c = X_te.copy()
    safe_groups = _resolve_spectral_groups(
        X_te, D_tfidf, D_neural_sum, D_neural_weight_total, max_cluster_size=max_cluster_size, groups_override=groups_override
    )
    if N_test >= 2:
        for g in np.unique(safe_groups):
            m = safe_groups == g
            if m.sum() > 1:
                X_te_c[m] -= X_te_c[m].mean(axis=0, keepdims=True)
            else:
                X_te_c[m] = 0.0
    else:
        X_te_c = np.zeros_like(X_te)

    groups_dict = SPECIALIST_GROUPS if SPECIALIST_GROUPS else FAMILIES
    family_gates: dict[str, np.ndarray] = {}
    for fam_name, fam_cols in groups_dict.items():
        mask_tr = np.isin(y_sub, fam_cols)
        min_samples = 20 if len(fam_cols) <= 3 else 30
        if mask_tr.sum() >= min_samples and len(np.unique(y_sub[mask_tr])) == len(fam_cols):
            sub_y = np.array([fam_cols.index(int(c)) for c in y_sub[mask_tr]], dtype=int)
            clf_p = LogisticRegression(C=0.10, max_iter=1000, class_weight="balanced", random_state=42)
            clf_p.fit(X_tr[mask_tr], sub_y)
            lp_p = clf_p.predict_log_proba(X_te)

            clf_c = LogisticRegression(C=0.06, max_iter=1000, class_weight="balanced", random_state=42)
            clf_c.fit(X_tr_c[mask_tr], sub_y)
            lp_c = clf_c.predict_log_proba(X_te_c)

            w_p, w_c = (
                (0.35, 0.65)
                if fam_name
                in (
                    "gemini_37_38_pair",
                    "gemini_flash_triad",
                    "glm",
                    "glm_kimi_triad",
                    "gpt_56_triad",
                    "kimi_gpt55_qwen",
                    "frontier_and_fable",
                )
                else (0.65, 0.35)
            )
            family_gates[fam_name] = log_softmax(w_p * lp_p + w_c * lp_c, axis=1)
    return family_gates


In [ ]:
# === MODULE: src/namethatllm/neural_extractor.py ===
"""
Universal Neural Trunk Loader & Feature Extractor
=================================================
Manages multi-GPU sharding (2xT4 GPUs, max 13.5 GiB), layer truncation (keep_layers),
dynamic bfloat16 row-level retry for 14B fp16 overflow rescue, LayerNorm heads,
and bit-exact CPU fp16 snapshot/restore for shared trunks.
"""

import gc
import json
import os
import re
import shutil
import time
from pathlib import Path
from typing import Any

import numpy as np
import torch
from safetensors.torch import load_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer



def infer_adapter_max_layer(ckpt_dir: Path) -> int | None:
    """Returns 1-indexed max transformer layer modified by the LoRA adapter (e.g. 24 or 27)."""
    ckpt_dir = Path(ckpt_dir)
    for rel in (
        "adapter_model.safetensors",
        "adapter/adapter_model.safetensors",
        "adapter_model.bin",
        "adapter/adapter_model.bin",
        "adapter_model.pt",
        "adapter/adapter_model.pt",
    ):
        p = ckpt_dir / rel
        if p.exists():
            try:
                if p.suffix == ".safetensors":
                    sd = load_file(str(p))
                else:
                    sd = torch.load(str(p), map_location="cpu", weights_only=False)
                    if hasattr(sd, "state_dict") and callable(sd.state_dict):
                        sd = sd.state_dict()
                layer_nums = []
                for k in sd.keys():
                    m = re.search(r"layers\.(\d+)\.", str(k))
                    if m:
                        layer_nums.append(int(m.group(1)) + 1)
                if layer_nums:
                    return max(layer_nums)
            except Exception:
                continue
    return None


def in_place_copy(dst: torch.Tensor, src: torch.Tensor):
    """In-place weight copy."""
    dst.data.copy_(src.to(device=dst.device, dtype=dst.dtype))


def _resolve_module_name(k: str, state_dict: dict) -> str | None:
    """Resolves module name across various AutoModel and PEFT naming schemes."""
    raw = k.split(".lora_A")[0]
    candidates = [raw, raw.replace("base_model.model.", "")]
    if "layers." in raw:
        rel = raw[raw.index("layers.") :]
        candidates.extend([rel, f"model.{rel}"])
    for c in candidates:
        if c in state_dict and hasattr(state_dict[c], "weight"):
            return c
    return None


def merge_lora(model: nn.Module, adapter_path: Path, sign: float = +1.0) -> bool:
    """
    Atomically merges or unmerges (+1.0 or -1.0) a LoRA adapter into trunk weights.
    Pre-validates all tensor shapes BEFORE mutating any weights so an incompatible
    architecture (e.g. Llama vs Qwen) never partially corrupts the trunk.
    """
    weights_path = adapter_path / "adapter_model.safetensors"
    if not weights_path.exists():
        weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
    if not weights_path.exists():
        return False

    try:
        sd = load_file(str(weights_path))
    except Exception as e:
        print(f"[merge_lora] Could not load {weights_path}: {e}")
        return False

    cfg_p = adapter_path / "adapter_config.json"
    if not cfg_p.exists():
        cfg_p = adapter_path / "adapter" / "adapter_config.json"
    scale = 1.0
    if cfg_p.exists():
        try:
            with open(cfg_p, "r", encoding="utf-8") as f:
                c = json.load(f)
            scale = float(c.get("lora_alpha", 16)) / float(c.get("r", 16))
        except Exception:
            scale = 1.0

    state_dict = dict(model.named_modules())
    pairs = {}
    for k in sd.keys():
        if "lora_A" in k:
            b_key = k.replace("lora_A", "lora_B")
            if b_key in sd:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None:
                    pairs[m_name] = (sd[k], sd[b_key])

    if not pairs:
        return False

    # Pass 1: Pre-validate all shapes before modifying any weight in `model`
    for m_name, (A, B) in pairs.items():
        mod = state_dict.get(m_name)
        if mod is None or not hasattr(mod, "weight"):
            continue
        expected_shape = (B.shape[0], A.shape[1])
        if tuple(mod.weight.shape) != expected_shape:
            print(
                f"[merge_lora] Architecture mismatch on {m_name} in {adapter_path.name}: "
                f"trunk weight {tuple(mod.weight.shape)} != adapter delta {expected_shape}. Aborting merge."
            )
            return False

    # Pass 2: All shapes verified — apply in-place update
    applied = 0
    with torch.no_grad():
        for m_name, (A, B) in pairs.items():
            mod = state_dict.get(m_name)
            if mod is not None and hasattr(mod, "weight"):
                A_d = A.to(mod.weight.device, dtype=torch.float32)
                B_d = B.to(mod.weight.device, dtype=torch.float32)
                delta = (B_d @ A_d) * (scale * sign)
                mod.weight.data.add_(delta.to(mod.weight.dtype))
                applied += 1
    return applied > 0


@torch.inference_mode()
def forward_and_pool_batch(
    model: nn.Module,
    enc: dict,
    layers: list[int],
    norms: nn.ModuleList,
    dtype: torch.dtype = torch.float16,
    drop_first_token: bool = True,
    final_norm: nn.Module | None = None,
):
    """
    Runs forward pass in `dtype` (float16 on T4).
    CRITICAL: If any target layer overflows fp16 (Inf/NaN), recomputes ONLY the overflowed
    rows in `torch.bfloat16` and pools directly in float32 before restoring `dtype`!
    """
    hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states

    B = enc["input_ids"].shape[0]
    first_norm_dev = norms[0].weight.device

    # Check if any selected layer has Inf/NaN per row
    bad_mask = torch.zeros(B, dtype=torch.bool, device="cpu")
    for l in layers:
        bad_mask |= ~torch.isfinite(hs[l]).all(dim=(1, 2)).cpu()

    n_bad = int(bad_mask.sum().item())
    m = enc["attention_mask"].unsqueeze(-1).float().to(first_norm_dev)
    if drop_first_token:
        m[:, 0, :] = 0.0
    den = m.sum(dim=1).clamp(min=1.0)

    # Pool finite rows first in float32
    pooled_layers = [torch.zeros((B, model.config.hidden_size), dtype=torch.float32, device=first_norm_dev) for _ in layers]
    good_rows = torch.where(~bad_mask)[0].to(first_norm_dev)
    if len(good_rows) > 0:
        for idx_l, l in enumerate(layers):
            h_raw = hs[l][good_rows.to(hs[l].device)]
            if final_norm is not None and idx_l == len(layers) - 1:
                fn_params = list(final_norm.parameters())
                fn_dev = fn_params[0].device if fn_params else h_raw.device
                fn_dtype = fn_params[0].dtype if fn_params else h_raw.dtype
                h_raw = final_norm(h_raw.to(device=fn_dev, dtype=fn_dtype)).to(h_raw.device)
            h_good = h_raw.to(first_norm_dev).float()
            pooled_layers[idx_l][good_rows] = (h_good * m[good_rows]).sum(dim=1) / den[good_rows]

    # Rescue overflowed rows in bfloat16 (exact exponent range as float32, NEVER overflows!)
    if n_bad > 0:
        bad_rows = torch.where(bad_mask)[0]
        del hs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model.to(torch.bfloat16)
        try:
            enc_bad = {k: v[bad_rows] for k, v in enc.items()}
            hs_bad = model(**enc_bad, output_hidden_states=True, use_cache=False).hidden_states
            bad_rows_dev = bad_rows.to(first_norm_dev)
            for idx_l, l in enumerate(layers):
                h_b = hs_bad[l]
                if final_norm is not None and idx_l == len(layers) - 1:
                    fn_params = list(final_norm.parameters())
                    fn_dev = fn_params[0].device if fn_params else h_b.device
                    fn_dtype = fn_params[0].dtype if fn_params else h_b.dtype
                    h_b = final_norm(h_b.to(device=fn_dev, dtype=fn_dtype)).to(h_b.device)
                h_b = h_b.to(first_norm_dev).float()
                h_b = torch.nan_to_num(h_b, nan=0.0, posinf=1e4, neginf=-1e4)
                pooled_layers[idx_l][bad_rows_dev] = (h_b * m[bad_rows_dev]).sum(dim=1) / den[bad_rows_dev]
            del hs_bad
        finally:
            model.to(dtype)

    out = torch.cat([nm(pooled_layers[idx_l]) for idx_l, nm in enumerate(norms)], dim=-1)
    return out.cpu().numpy(), n_bad


def _flatten_state_dict(obj: Any, prefix: str = "", out: dict | None = None) -> dict:
    if out is None:
        out = {}
    if hasattr(obj, "state_dict") and callable(obj.state_dict):
        obj = obj.state_dict()
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_k = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, (dict, list, tuple)) or (hasattr(v, "state_dict") and callable(v.state_dict)):
                _flatten_state_dict(v, new_k, out)
            else:
                out[new_k] = v
                if prefix in ("state_dict", "model", "heads", "head") and str(k) not in out:
                    out[str(k)] = v
    elif isinstance(obj, (list, tuple)):
        for idx, item in enumerate(obj):
            new_k = f"{prefix}.{idx}" if prefix else str(idx)
            _flatten_state_dict(item, new_k, out)
    return out


def load_head_file(path_or_dir: Path) -> dict:
    """Unified loader for classification head state dicts (.safetensors, .pt, .bin, .npz)."""
    p = Path(path_or_dir)
    res = {}
    if p.is_file():
        if p.suffix == ".safetensors":
            res = load_file(str(p))
        elif p.suffix in (".pt", ".bin"):
            raw = torch.load(str(p), map_location="cpu", weights_only=False)
            res = _flatten_state_dict(raw)
        elif p.suffix == ".npz":
            npz = np.load(str(p), allow_pickle=True)
            res = {k: npz[k] for k in npz.files}
        if res:
            print(f"[Head File] {p.name} loaded ({len(res)} keys): {list(res.keys())[:15]}")
        return res
    for fname in ("heads.safetensors", "head.safetensors", "heads.pt", "head.pt", "readouts.npz"):
        cand = p / fname
        if cand.exists():
            return load_head_file(cand)
    return {}


def resolve_head_tensors(
    heads_dict: dict, num_classes: int | None = None
) -> tuple[np.ndarray | None, np.ndarray | None, np.ndarray | None, np.ndarray | None]:
    """Resolves plain and centered head weights and biases matching American/British naming and handles transposed weights."""
    W_p, b_p, W_c, b_c = None, None, None, None
    valid_k = (num_classes,) if num_classes is not None else (K, 5, 4, 2)

    for wk, bk in [
        ("plain.weight", "plain.bias"),
        ("plain_coef", "plain_intercept"),
        ("head.weight", "head.bias"),
        ("classifier.weight", "classifier.bias"),
        ("fc.weight", "fc.bias"),
        ("W_plain", "b_plain"),
        ("weight", "bias"),
        ("W", "b"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_p, b_p = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_p = heads_dict[wk]
            k_rows = (
                W_p.shape[0]
                if (hasattr(W_p, "shape") and len(W_p.shape) > 0 and W_p.shape[0] in valid_k)
                else (W_p.shape[1] if (hasattr(W_p, "shape") and len(W_p.shape) > 1 and W_p.shape[1] in valid_k) else (num_classes or K))
            )
            b_p = np.zeros(k_rows, dtype=np.float32)
            break

    for wk, bk in [
        ("centered.weight", "centered.bias"),
        ("centred.weight", "centred.bias"),
        ("center.weight", "center.bias"),
        ("twin.weight", "twin.bias"),
        ("center_coef", "center_intercept"),
        ("centred_coef", "centred_intercept"),
        ("W_center", "b_center"),
        ("W_twin", "b_twin"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_c, b_c = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_c = heads_dict[wk]
            k_rows = (
                W_c.shape[0]
                if (hasattr(W_c, "shape") and len(W_c.shape) > 0 and W_c.shape[0] in valid_k)
                else (W_c.shape[1] if (hasattr(W_c, "shape") and len(W_c.shape) > 1 and W_c.shape[1] in valid_k) else (num_classes or K))
            )
            b_c = np.zeros(k_rows, dtype=np.float32)
            break

    # Suffix / Keyword Matching fallback if W_p or W_c still None
    if W_p is None or W_c is None:
        for k, v in heads_dict.items():
            k_lower = str(k).lower()
            shape = getattr(v, "shape", None)
            if shape is not None and len(shape) == 2 and (shape[0] in valid_k or shape[1] in valid_k):
                # Check for center / twin
                if any(tag in k_lower for tag in ("center", "centred", "twin")):
                    if W_c is None:
                        W_c = v
                        base_prefix = k.rsplit(".", 1)[0] if "." in k else k.rsplit("_", 1)[0]
                        for bk in [
                            f"{base_prefix}.bias",
                            f"{base_prefix}_bias",
                            f"{base_prefix}.intercept",
                            f"{base_prefix}_intercept",
                            "bias",
                            "centered.bias",
                            "b",
                        ]:
                            if bk in heads_dict and getattr(heads_dict[bk], "shape", None) is not None:
                                b_cand = heads_dict[bk]
                                if getattr(b_cand, "shape", ()) in [(k_item,) for k_item in valid_k]:
                                    b_c = b_cand
                                    break
                        if b_c is None:
                            k_rows = W_c.shape[0] if W_c.shape[0] in valid_k else W_c.shape[1]
                            b_c = np.zeros(k_rows, dtype=np.float32)
                # Check for plain / head / clf / classifier
                elif any(tag in k_lower for tag in ("plain", "head", "clf", "classifier")):
                    if W_p is None:
                        W_p = v
                        base_prefix = k.rsplit(".", 1)[0] if "." in k else k.rsplit("_", 1)[0]
                        for bk in [
                            f"{base_prefix}.bias",
                            f"{base_prefix}_bias",
                            f"{base_prefix}.intercept",
                            f"{base_prefix}_intercept",
                            "bias",
                            "plain.bias",
                            "b",
                        ]:
                            if bk in heads_dict and getattr(heads_dict[bk], "shape", None) is not None:
                                b_cand = heads_dict[bk]
                                if getattr(b_cand, "shape", ()) in [(k_item,) for k_item in valid_k]:
                                    b_p = b_cand
                                    break
                        if b_p is None:
                            k_rows = W_p.shape[0] if W_p.shape[0] in valid_k else W_p.shape[1]
                            b_p = np.zeros(k_rows, dtype=np.float32)

    if W_p is not None and hasattr(W_p, "cpu"):
        W_p = W_p.cpu().numpy()
    if b_p is not None and hasattr(b_p, "cpu"):
        b_p = b_p.cpu().numpy()
    if W_c is not None and hasattr(W_c, "cpu"):
        W_c = W_c.cpu().numpy()
    if b_c is not None and hasattr(b_c, "cpu"):
        b_c = b_c.cpu().numpy()

    # Transpose Guard: if shape is (D, K_cls), transpose to canonical (K_cls, D)
    if W_p is not None and W_p.ndim == 2:
        if W_p.shape[1] in valid_k and W_p.shape[0] not in valid_k:
            W_p = W_p.T
    if W_c is not None and W_c.ndim == 2:
        if W_c.shape[1] in valid_k and W_c.shape[0] not in valid_k:
            W_c = W_c.T

    return W_p, b_p, W_c, b_c


def compute_centered_logits(
    Z_test: np.ndarray,
    groups: np.ndarray,
    W_c: np.ndarray,
    b_c: np.ndarray,
    src_labels: list,
    max_std: float = 4.2,
) -> tuple[np.ndarray, np.ndarray]:
    """Computes prompt-topic centered twin logits with temperature normalization matching plain heads."""
    N = len(Z_test)
    out_lp = np.zeros((N, K), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(safe_g[valid]):
        mask = safe_g == g
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = align_cols(Zc[valid] @ W_c.T + b_c, src_labels)
    std_c = float(np.std(raw_c, axis=1).mean()) if len(raw_c) > 0 else 1.0
    temp_c = max(1.0, std_c / max_std)
    out_lp[valid] = log_softmax(raw_c / temp_c, axis=1)
    return out_lp, valid


def compute_family_centered_logits(
    Z_test: np.ndarray,
    groups: np.ndarray,
    W_c: np.ndarray,
    b_c: np.ndarray,
    src_labels: list,
    target_labels: list,
) -> tuple[np.ndarray, np.ndarray]:
    """Computes within-cluster centered twin logits for a K_fam-class family specialist."""
    N = len(Z_test)
    K_fam = len(target_labels)
    out_lp = np.zeros((N, K_fam), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(safe_g[valid]):
        mask = safe_g == g
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = Zc[valid] @ W_c.T + b_c
    src_list = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    if len(src_list) == K_fam and set(src_list) == set(target_labels):
        perm = [src_list.index(l) for l in target_labels]
        raw_c = raw_c[:, perm]
    out_lp[valid] = log_softmax(raw_c, axis=1)
    return out_lp, valid


class UniversalLoRATrunkLoader:
    def __init__(self, base_dir: Path, keep_layers: int, layers: list[int], max_len: int = 447):
        if base_dir is not None and not (base_dir / "config.json").exists():
            for sub in base_dir.rglob("config.json"):
                base_dir = sub.parent
                break
        self.base_dir = base_dir
        self.keep_layers = keep_layers
        self.layers = layers
        self.max_len = max_len
        self.total_rescued = 0
        self.tok = AutoTokenizer.from_pretrained(str(base_dir), trust_remote_code=True)
        self.tok.padding_side = "right"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token or "<|endoftext|>"

        cfg = AutoConfig.from_pretrained(str(base_dir), trust_remote_code=True)
        cfg.num_hidden_layers = keep_layers
        if getattr(cfg, "layer_types", None) is not None:
            cfg.layer_types = cfg.layer_types[:keep_layers]
        cfg.use_cache = False

        n_gpus = torch.cuda.device_count()
        if n_gpus >= 2:
            max_memory = {0: "13.5GiB", 1: "13.5GiB"}
            self.model = AutoModel.from_pretrained(
                str(base_dir),
                config=cfg,
                device_map="auto",
                max_memory=max_memory,
                torch_dtype=torch.float16,
                trust_remote_code=True,
                low_cpu_mem_usage=True,
            )
        elif n_gpus == 1:
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
            )
        else:
            self.model = AutoModel.from_pretrained(str(base_dir), config=cfg, torch_dtype=torch.float32, low_cpu_mem_usage=True)

        # Preserve original final trunk norm before optionally disabling
        self.orig_final_norm = getattr(self.model, "norm", None) or getattr(getattr(self.model, "model", None), "norm", None)

        # Default: keep HuggingFace model.norm as nn.Identity() at all times
        if hasattr(self.model, "norm"):
            self.model.norm = nn.Identity()
        if hasattr(self.model, "model") and hasattr(self.model.model, "norm"):
            self.model.model.norm = nn.Identity()

        self.final_norm_enabled = False
        self.model.eval()
        self.device_norms = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # nn.LayerNorm with both weight and bias matching heads.safetensors
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in layers]).to(self.device_norms).float()

        self.norms.eval()
        self.first_dev = next(self.model.parameters()).device

    def set_final_norm(self, enabled: bool):
        """Sets final_norm_enabled flag, keeping HuggingFace model.norm as nn.Identity()."""
        self.final_norm_enabled = bool(enabled)

    def snapshot_weights(self, adapter_path: Path) -> dict:
        """Saves CPU fp16 copies of trunk modules modified by adapter."""
        weights_path = adapter_path / "adapter_model.safetensors"
        if not weights_path.exists():
            weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
        if not weights_path.exists():
            return {}
        sd = load_file(str(weights_path))
        state_dict = dict(self.model.named_modules())
        backup = {}
        for k in sd.keys():
            if "lora_A" in k:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None and m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    backup[m_name] = state_dict[m_name].weight.detach().cpu().half().clone()
        return backup

    def restore_weights(self, backup: dict):
        """Restores base trunk weights bit-for-bit without FP16 subtraction drift."""
        state_dict = dict(self.model.named_modules())
        with torch.no_grad():
            for m_name, orig in backup.items():
                if m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    state_dict[m_name].weight.data.copy_(orig.to(state_dict[m_name].weight.device))

    def set_active_layers(self, layers: list[int], max_len: int | None = None):
        """Updates active pooled layers and instantiates LayerNorms matching the checkpoint."""
        self.layers = list(layers)
        if max_len is not None:
            self.max_len = max_len
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in self.layers]).to(self.device_norms).float()
        self.norms.eval()

    def load_norm_weights(self, heads_sd: dict):
        """Loads LayerNorm weights and biases from head state dict after resetting to identity."""
        with torch.no_grad():
            # First reset every ln in self.norms to identity so weights/biases from previous checkpoint never bleed
            for ln in self.norms:
                ln.weight.data.fill_(1.0)
                if hasattr(ln, "bias") and ln.bias is not None:
                    ln.bias.data.zero_()

            # Support all norm key prefixes: norms.{i}, layer_norms.{i}, ln.{i}, norm.{i}
            for i, ln in enumerate(self.norms):
                for pfx in ("norms", "layer_norms", "ln", "norm"):
                    w_k = f"{pfx}.{i}.weight"
                    b_k = f"{pfx}.{i}.bias"
                    if w_k in heads_sd:
                        ln.weight.copy_(heads_sd[w_k].to(ln.weight.device, ln.weight.dtype))
                    elif (
                        f"{pfx}.{i}" in heads_sd
                        and hasattr(heads_sd[f"{pfx}.{i}"], "shape")
                        and heads_sd[f"{pfx}.{i}"].shape == ln.weight.shape
                    ):
                        ln.weight.copy_(heads_sd[f"{pfx}.{i}"].to(ln.weight.device, ln.weight.dtype))

                    if b_k in heads_sd and hasattr(ln, "bias") and ln.bias is not None:
                        ln.bias.copy_(heads_sd[b_k].to(ln.bias.device, ln.bias.dtype))

            # Check for stacked 2D tensor: norms_weight or norms.weight of shape (len(self.layers), H)
            for stk in ("norms_weight", "norms.weight", "layer_norms.weight", "ln.weight"):
                if stk in heads_sd and hasattr(heads_sd[stk], "ndim") and heads_sd[stk].ndim == 2:
                    stk_t = heads_sd[stk]
                    if stk_t.shape[0] == len(self.norms):
                        for i, ln in enumerate(self.norms):
                            ln.weight.copy_(stk_t[i].to(ln.weight.device, ln.weight.dtype))

    def _extract_batch_ids(self, id_list: list[list[int]], batch_size: int = 8) -> np.ndarray:
        order = np.argsort([len(x) for x in id_list], kind="stable")
        total_dim = len(self.layers) * self.model.config.hidden_size
        Z = np.zeros((len(id_list), total_dim), dtype=np.float32)

        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            enc = self.tok.pad({"input_ids": [id_list[i] for i in idx]}, padding=True, return_tensors="pt")
            enc = {k: v.to(self.first_dev) for k, v in enc.items()}
            fn = self.orig_final_norm if getattr(self, "final_norm_enabled", False) else None
            out_np, n_bad = forward_and_pool_batch(
                self.model, enc, self.layers, self.norms, torch.float16, drop_first_token=True, final_norm=fn
            )
            self.total_rescued += n_bad
            Z[idx] = out_np
        return Z

    @torch.inference_mode()
    def extract_exact(self, texts: list[str], batch_size: int = 8, end_id: int | None = 151643) -> np.ndarray:
        t0 = time.time()
        self.total_rescued = 0
        raw_ids = self.tok(list(texts), add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
        if end_id is not None:
            ids = [x + [end_id] for x in raw_ids]
        else:
            ids = raw_ids
        Z = self._extract_batch_ids(ids, batch_size=batch_size)
        print(f"[Extractor] Extracted {len(texts)} samples in {time.time() - t0:.2f}s (Total BF16 Rescued Rows: {self.total_rescued}).")
        return Z

    def free_memory(self):
        del self.model, self.norms
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def read_checkpoint_meta(ckpt_dir: Path, default_keep: int, default_layers: list[int], default_max_len: int = 447) -> dict:
    """Reads keep, layers, max_len, end_id, and labels dynamically from meta.json or infers from adapter weights."""
    ckpt_dir = Path(ckpt_dir)
    meta_p = ckpt_dir / "meta.json"
    dir_str = str(ckpt_dir).lower().replace("\\", "/")
    is_dhruv = any(x in dir_str for x in ("q14v5", "ts_q3b14", "ts_q3b8", "q8v5", "ll8v5", "ts_llama8", "l24x"))

    labels = sorted(LABELS) if is_dhruv else list(LABELS)
    keep = 24 if is_dhruv else default_keep
    layers = [2, 4, 6, 8, 12, 16, 20, 24] if is_dhruv else list(default_layers)
    max_len = 384 if is_dhruv else default_max_len
    end_id = None if is_dhruv else 151643
    no_final_norm = False if is_dhruv else True

    if meta_p.exists():
        try:
            with open(meta_p, "r", encoding="utf-8") as f:
                m = json.load(f)
            labels = m.get("labels", m.get("classes", labels))
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", keep))
            layers = list(cfg.get("layers", layers))
            max_len = int(cfg.get("max_len", max_len))
            if "no_final_norm" in cfg:
                no_final_norm = bool(cfg["no_final_norm"])
            for ek in ("end_token_id", "eos_id", "eos_token_id", "end_token"):
                if ek in cfg:
                    end_id = int(cfg[ek]) if isinstance(cfg[ek], int) else None
                    break
        except Exception:
            pass
    else:
        max_l = infer_adapter_max_layer(ckpt_dir)
        if max_l == 24:
            keep = 24
            layers = [2, 4, 6, 8, 12, 16, 20, 24]
        elif max_l == 27:
            keep = 27
            layers = [2, 4, 6, 9, 13, 17, 22, 27]
        elif max_l == 32:
            keep = 32
            layers = [3, 5, 8, 11, 16, 21, 27, 32]
        elif max_l is not None:
            keep = max_l

    return {"keep": keep, "layers": layers, "max_len": max_len, "end_id": end_id, "no_final_norm": no_final_norm, "labels": labels}


def extract_deberta_checkpoint(
    ckpt_dir: Path,
    base_deb_dir: Path | None,
    texts_test: list[str],
    texts_train: list[str],
    y_train: np.ndarray,
    max_len: int = 512,
    batch_size: int = 8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray | None, np.ndarray | None, np.ndarray | None, np.ndarray | None, list]:
    """
    Dedicated feature extractor for DeBERTa-v3-large sequence classification checkpoints.
    Supports both single-layer encoder (Layer 24 float32 last_hidden_state) and multi-layer heads.
    """
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    heads_sd = load_head_file(ckpt_dir)
    W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

    # Load tokenizer from encoder directory if available, matching exact training setup
    if (ckpt_dir / "encoder").exists():
        tok_dir = str(ckpt_dir / "encoder")
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        tok_dir = str(base_deb_dir)
    else:
        tok_dir = str(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(tok_dir, use_fast=True)

    meta = read_checkpoint_meta(ckpt_dir, default_keep=24, default_layers=[3, 6, 9, 12, 15, 18, 21, 24], default_max_len=max_len)
    ckpt_labels = meta["labels"]
    layers = meta["layers"]
    max_len = meta["max_len"]

    if (ckpt_dir / "encoder" / "config.json").exists():
        model_src = ckpt_dir / "encoder"
    elif (ckpt_dir / "config.json").exists() and ((ckpt_dir / "model.safetensors").exists() or (ckpt_dir / "pytorch_model.bin").exists()):
        model_src = ckpt_dir
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        model_src = base_deb_dir
    else:
        model_src = ckpt_dir

    stage_dir = Path("/kaggle/temp/deberta_p2_stage")
    staged = False
    load_src = model_src
    if Path("/kaggle/temp").exists() and os.access("/kaggle/temp", os.W_OK):
        try:
            if stage_dir.exists():
                shutil.rmtree(stage_dir, ignore_errors=True)
            shutil.copytree(model_src, stage_dir)
            load_src = stage_dir
            staged = True
        except Exception:
            load_src = model_src

    # CRITICAL: Always use torch.float32 for DeBERTa-v3 to avoid FP16 overflow!
    model = None
    try:
        model = AutoModel.from_pretrained(str(load_src), torch_dtype=torch.float32).to(dev)
    finally:
        if staged and stage_dir.exists():
            shutil.rmtree(stage_dir, ignore_errors=True)

    model.eval()

    hidden_dim = model.config.hidden_size
    has_norm_keys = any(k.startswith("norms.") and k.endswith(".weight") for k in heads_sd)
    is_single_last_layer = W_p is not None and W_p.shape[1] == hidden_dim and not has_norm_keys

    if not is_single_last_layer:
        if W_p is not None and W_p.shape[1] > 0:
            expected_layers = W_p.shape[1] // hidden_dim
            if len(layers) != expected_layers:
                norm_keys = [k for k in heads_sd if k.startswith("norms.") and k.endswith(".weight")]
                if len(norm_keys) == expected_layers:
                    layers = layers[-expected_layers:] if len(layers) > expected_layers else list(range(expected_layers))
                elif len(layers) > expected_layers:
                    layers = layers[-expected_layers:]

        norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in layers]).to(dev).float()
        with torch.no_grad():
            for i, ln in enumerate(norms):
                w_k = f"norms.{i}.weight"
                b_k = f"norms.{i}.bias"
                if w_k in heads_sd:
                    ln.weight.copy_(heads_sd[w_k].to(dev, ln.weight.dtype))
                if b_k in heads_sd:
                    ln.bias.copy_(heads_sd[b_k].to(dev, ln.bias.dtype))
        norms.eval()
    else:
        norms = None

    def pool_texts(txt_list: list[str], drop_first: bool = False) -> np.ndarray:
        out_dim = hidden_dim if is_single_last_layer else (len(layers) * hidden_dim)
        if len(txt_list) == 0:
            return np.zeros((0, out_dim), dtype=np.float32)
        order = np.argsort([len(t) for t in txt_list], kind="stable")
        all_vecs = np.zeros((len(txt_list), out_dim), dtype=np.float32)
        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            chunk = [txt_list[i] for i in idx]
            enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            enc = {k: v.to(dev) for k, v in enc.items()}
            with torch.no_grad():
                if is_single_last_layer:
                    out_hs = model(**enc).last_hidden_state
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    if drop_first:
                        m[:, 0, :] = 0.0
                    den = m.sum(dim=1).clamp(min=1.0)
                    pooled = (torch.nan_to_num(out_hs.float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den
                    all_vecs[idx] = pooled.cpu().numpy()
                else:
                    hs = model(**enc, output_hidden_states=True).hidden_states
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    if drop_first:
                        m[:, 0, :] = 0.0
                    den = m.sum(dim=1).clamp(min=1.0)
                    pooled = [((torch.nan_to_num(hs[l].float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den) for l in layers]
                    out = torch.cat([norms[i](pooled[i]) for i in range(len(layers))], dim=-1)
                    all_vecs[idx] = out.cpu().numpy()
        return all_vecs

    # Probe drop_first_token on train.csv[:180]
    probe_n = min(180, len(texts_train))
    best_drop = False if is_single_last_layer else True
    if probe_n > 0 and W_p is not None:
        best_acc = -1.0
        for cand_drop in [False, True] if is_single_last_layer else [True, False]:
            z_pr = pool_texts(texts_train[:probe_n], drop_first=cand_drop)
            preds = align_cols(z_pr @ W_p.T + b_p, ckpt_labels).argmax(axis=1)
            acc = float((preds == y_train[:probe_n]).mean())
            if acc > best_acc:
                best_acc = acc
                best_drop = cand_drop
        print(f"[DeBERTa Extractor] Probed drop_first={best_drop} (accuracy = {best_acc:.4f})")

    out_dim = hidden_dim if is_single_last_layer else (len(layers) * hidden_dim)
    Z_test = pool_texts(texts_test, drop_first=best_drop)
    Z_train = pool_texts(texts_train, drop_first=best_drop) if len(texts_train) > 0 else np.zeros((0, out_dim), dtype=np.float32)

    if model is not None:
        del model
    if norms is not None:
        del norms
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Z_test, Z_train, W_p, b_p, W_c, b_c, ckpt_labels


In [ ]:
# === MODULE: src/namethatllm/postprocess.py ===
"""
Postprocessing, Fusion & Calibration Engine
===========================================
This module handles all post-ensemble refinement stages:
1. Two-tier convex mixture (85% Main Stack + 15% Complementary Stack)
2. Mass-preserving Specialist Neural Gates (Claude head, Gemini LoRA, GLM LoRA, GPT LoRA)
3. Confidence-aware Within-Family Spectral Gates (dynamic margin tie-breakers)
4. Unified Handwriting & Orthographic Shield (discrete Bayesian tokenization rules + continuous density)
5. Cavity Mean-Field Cluster Uniqueness Repulsion (soft at-most-once constraint on prompt clusters)
6. Multiplicative Sinkhorn-Knopp Column Marginal Balancing (global class prior equalization)
7. Hierarchical Beta-Bernoulli TypoBayes Postprocessor
8. Final Kaggle submission generation (submission.parquet and submission.csv)
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# 1. WITHIN-FAMILY CONFIDENCE GATES (SPECTRAL / TIE-BREAKER HEURISTICS)
# =============================================================================


def apply_all_family_confidence_gates(
    P_mix: np.ndarray,
    family_gates: dict[str, np.ndarray] | None,
    max_strength: float = 0.20,
    margin_thr: float = 0.45,
) -> np.ndarray:
    """
    Confidence-aware within-family and cross-family specialist reranker.

    Concept:
    --------
    When an ensemble predicts probabilities across 18 classes, sister models within
    the same family (e.g., Gemini 3.7 vs 3.8, or GLM-5.3 vs GLM-5.3-flash) can have
    extremely close probabilities (low margin). This function uses auxiliary gate
    logits (from spectral/TF-IDF classifiers) to break ties in low-margin cases.

    Key Invariant - Mass Conservation:
    ----------------------------------
    The total probability assigned to the family is strictly preserved!
    Only the internal relative distribution among the sisters is adjusted.
    Probabilities for classes outside the family are completely untouched.

    Dynamic Confidence Gating:
    --------------------------
    - Margin = (Top-1 prob within family) - (Top-2 prob within family).
    - If margin >= margin_thr: Model is already confident; gate strength = 0 (no intervention).
    - If margin < margin_thr: Low confidence; gate strength scales up to max_strength.

    Parameters:
    -----------
    P_mix : np.ndarray of shape (N, 18)
        Current blended prediction probabilities (summing to 1.0 per row).
    family_gates : dict mapping group_name -> log_probabilities of shape (N, K_group)
        Pre-computed log-probabilities from the auxiliary family classifiers.
    max_strength : float
        Maximum interpolation weight for the gate (0.0 to 1.0). If <= 0, gates are bypassed.
    margin_thr : float
        Confidence margin threshold above which the gate does not intervene.

    Returns:
    --------
    P_out : np.ndarray of shape (N, 18)
        Calibrated probabilities with sister-class confusions resolved.
    """
    # Early exit: If gates are disabled or weight is zero, return immediately
    if not family_gates or float(max_strength) <= 0.0:
        return P_mix

    P_out = P_mix.astype(np.float64).copy()

    for fam_name, gate_lp in family_gates.items():
        # Identify the column indices belonging to this family/specialist group
        cols = SPECIALIST_GROUPS.get(fam_name, FAMILIES.get(fam_name))
        if cols is None:
            continue
        if gate_lp is None or gate_lp.shape != (len(P_out), len(cols)):
            continue

        # Determine which samples are eligible for this gate:
        # Cross-family triads only apply when top-1 prediction is in the triad.
        # Single-family gates apply across all rows where that family has mass.
        if fam_name in ("frontier_and_fable", "kimi_gpt55_qwen", "glm_kimi_triad"):
            row_mask = np.isin(P_out.argmax(axis=1), cols)
            if not row_mask.any():
                continue
        elif fam_name == "gemini_37_38_pair":
            row_mask = np.isin(P_out.argmax(axis=1), FAMILIES.get("gemini", [5, 6, 7, 8]))
            if not row_mask.any():
                continue
        else:
            row_mask = np.ones(len(P_out), dtype=bool)

        # Hard-sister groups (e.g. Gemini 3.7 vs 3.8, GLM) use wider margin threshold
        eff_margin_thr = 0.65 if fam_name in ("gemini_37_38_pair", "glm", "glm_kimi_triad") else margin_thr
        eff_max_strength = (
            max(float(max_strength), 0.35) if fam_name in ("gemini_37_38_pair", "glm", "glm_kimi_triad") else float(max_strength)
        )

        # Step 1: Extract sub-probabilities and preserve total marginal mass
        sub_p = np.clip(P_out[np.ix_(row_mask, cols)], 1e-12, 1.0)
        tot_mass = sub_p.sum(axis=1, keepdims=True)
        base_rel_p = sub_p / tot_mass  # Relative distribution within the family (sums to 1.0)

        # Step 2: Compute margin between top-1 and top-2 candidates within family
        sp = np.sort(base_rel_p, axis=1)
        margin = sp[:, -1] - sp[:, -2]

        # Step 3: Compute dynamic intervention strength (higher when margin is small)
        strength = np.clip((eff_margin_thr - margin) / max(eff_margin_thr, 1e-6), 0.0, 1.0)[:, None] * eff_max_strength

        # Step 4: Blend base probabilities with gate probabilities in log-space
        base_lp = np.log(np.clip(base_rel_p, 1e-12, 1.0))
        gate_p = softmax(gate_lp[row_mask], axis=1)
        gate_lp_norm = np.log(np.clip(gate_p, 1e-12, 1.0))
        mixed_rel_p = softmax((1.0 - strength) * base_lp + strength * gate_lp_norm, axis=1)

        # Step 5: Restore original total family mass, leaving external classes unaffected
        P_out[np.ix_(row_mask, cols)] = tot_mass * mixed_rel_p

    # Final safety re-normalization to guarantee row sums == 1.0
    return P_out / P_out.sum(axis=1, keepdims=True).clip(min=1e-12)


# =============================================================================
# 2. PROMPT-CLUSTER UNIQUENESS REPULSION (CAVITY MEAN-FIELD)
# =============================================================================


def apply_cluster_uniqueness_repulsion(
    P_mix: np.ndarray,
    groups: np.ndarray | None,
    strength: float = 0.35,
    iters: int = 3,
) -> np.ndarray:
    """
    Enforces soft at-most-once author uniqueness within test prompt clusters.

    Domain Insight:
    ---------------
    In LLM benchmarking, multiple completions are generated by presenting the
    EXACT SAME PROMPT to different LLM models. Hence, when two or more test passages
    share an identical prompt (forming a 'prompt cluster'), competition rules imply
    that each completion came from a DIFFERENT model (at most 1 completion per model).

    Mathematical Formulation (Cavity Mean-Field):
    ---------------------------------------------
    For each text i in cluster g, the 'cavity field' for class k is the probability
    that no other text j in cluster g (j != i) was generated by class k:
        log P(other != k) = sum_{j in g, j != i} log(1 - q_{j, k})

    We update the unnormalized log-posterior:
        L_{i, k} = log(P0_{i, k}) + strength * sum_{j != i} log(1 - q_{j, k})
    where P0 is the original prediction prior, and q is the current cluster belief.
    Softmax is applied over k, iterating 3 times to reach a self-consistent equilibrium.

    Parameters:
    -----------
    P_mix : np.ndarray of shape (N, 18)
        Input prediction probabilities.
    groups : np.ndarray of shape (N,)
        Cluster IDs assigned to each test sample (e.g. from TF-IDF prompt clustering).
    strength : float
        Repulsion strength factor (typically 0.35).
    iters : int
        Number of mean-field coordinate ascent iterations (typically 3).

    Returns:
    --------
    q : np.ndarray of shape (N, 18)
        Repulsed probabilities penalizing multiple identical predictions in one prompt cluster.
    """
    N = len(P_mix)
    # Require at least 18 rows and valid cluster assignments
    if N < 18 or groups is None or len(groups) != N or strength <= 0.0:
        return P_mix

    # Map cluster identifiers to dense integer labels 1, 2, ...
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]

    # Only operate on multi-sample clusters between 2 and 18 texts (size <= 18 since there are 18 classes)
    valid_g = np.unique(safe_g[(sz >= 2) & (sz <= 18)])
    if len(valid_g) == 0:
        return P_mix

    log_P0 = np.log(np.clip(P_mix.astype(np.float64), 1e-12, 1.0))
    q = P_mix.astype(np.float64).copy()

    # Mean-field relaxation loop
    for _ in range(iters):
        L = log_P0.copy()
        for g in valid_g:
            m = safe_g == g
            q_g = q[m]  # Shape: (cluster_size, 18)

            # lo[j, k] = log(1 - q_{j, k}): log-probability that sample j did NOT pick class k
            lo = np.log(np.clip(1.0 - q_g, 1e-4, 1.0))

            # Cavity sum: sum of lo over all samples in cluster minus sample i's own term
            # Vectorized identity: sum_{j} lo[j] - lo[i] equals sum_{j != i} lo[j]
            cavity_penalty = lo.sum(axis=0, keepdims=True) - lo
            L[m] += float(strength) * cavity_penalty

        # Re-normalize across the 18 classes for next iteration
        q = softmax(L, axis=1)

    return q


# =============================================================================
# 3. TWO-TIER CONVEX ENSEMBLE BLENDING
# =============================================================================


def blend_main_and_complementary(
    LOGIT_MAIN: np.ndarray, COMP_LIST: list[tuple[float, np.ndarray]], main_ratio: float = 0.85, comp_ratio: float = 0.15
) -> tuple[np.ndarray, np.ndarray]:
    """
    Blends the Main Stack and Complementary Stack in probability space.

    Architecture:
    -------------
    - Main Stack (85% weight): High-capacity deep readers (Qwen3-14B Part 2,
      in-house champion LoRA, etc.) providing top semantic representations.
    - Complementary Stack (15% weight): Fast, diverse, non-neural or shallow readers
      (Style SVC 4-view, Lexical 5-view, DeBERTa, 8B patches) providing orthographic
      and stylistic diversity that regularizes neural hallucinations.

    Formula:
    --------
        R_main = softmax(LOGIT_MAIN)
        P_comp = sum(w_i * P_i) / sum(w_i)
        P_mix  = main_ratio * R_main + comp_ratio * P_comp

    Parameters:
    -----------
    LOGIT_MAIN : np.ndarray of shape (N, 18)
        Raw logit sum from the primary deep models.
    COMP_LIST : list of tuples (weight, probability_array)
        List of complementary models and their assigned ensemble weights.
    main_ratio : float
        Convex weight for Main Stack (default 0.85).
    comp_ratio : float
        Convex weight for Complementary Stack (default 0.15).

    Returns:
    --------
    P_mix : np.ndarray of shape (N, 18)
        Blended ensemble probabilities.
    R_main : np.ndarray of shape (N, 18)
        Main stack probabilities alone (used for routing/gating decisions).
    """
    R_main = softmax(LOGIT_MAIN, axis=1)
    if len(COMP_LIST) > 0:
        comp_weight_total = sum(w for w, _ in COMP_LIST)
        P_comp = sum(w * P_c for w, P_c in COMP_LIST) / max(1e-8, comp_weight_total)
        P_mix = main_ratio * R_main + comp_ratio * P_comp
    else:
        P_mix = R_main
    return P_mix, R_main


# =============================================================================
# 4. NEURAL SPECIALIST GATES (CLAUDE HEAD & FAMILY LORAS)
# =============================================================================


def apply_claude_specialist_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    claude_spec_logits: np.ndarray | None,
    claude_classes: list | None = None,
    labels: list = LABELS,
    claude_cols: list = FAMILIES["claude"],
    gate_weight: float = 0.15,
    weight: float | None = None,
) -> np.ndarray:
    """
    Refines passages predicted in the Claude family using a specialist head
    while strictly conserving total Claude family probability mass.
    """
    if claude_spec_logits is None:
        return P_mix
    if weight is not None:
        gate_weight = weight
    if float(gate_weight) <= 0.0:
        return P_mix

    P_out = P_mix.copy()
    P_claude_head = softmax(claude_spec_logits, axis=-1)

    if claude_classes is not None:
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in claude_classes]
        if all(labels[c] in src_c for c in claude_cols):
            c_idx = [src_c.index(labels[c]) for c in claude_cols]
            P_claude_head = P_claude_head[:, c_idx]

    claude_mask = np.isin(R_main.argmax(axis=1), claude_cols)
    if claude_mask.any():
        rows = np.flatnonzero(claude_mask)
        sub_mix = P_out[np.ix_(rows, claude_cols)]
        tot_mass = sub_mix.sum(axis=1, keepdims=True)
        blended = (1.0 - gate_weight) * sub_mix + gate_weight * (tot_mass * P_claude_head[rows])
        P_out[np.ix_(rows, claude_cols)] = blended

    return P_out


def apply_family_specialist_lora_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    spec_logits: np.ndarray | None,
    spec_classes: list | None = None,
    labels: list = LABELS,
    family_cols: list = FAMILIES["gemini"],
    gate_weight: float = 0.65,
) -> np.ndarray:
    """
    Refines intra-family predictions using fine-tuned Qwen3-14B Family-Specialist LoRAs
    while strictly preserving 100% of total marginal family probability mass.
    """
    if spec_logits is None or len(spec_logits) != len(P_mix) or float(gate_weight) <= 0.0:
        return P_mix

    P_out = P_mix.astype(np.float64).copy()
    spec_arr = np.asarray(spec_logits, dtype=np.float64)
    if spec_arr.ndim != 2 or spec_arr.shape[1] != len(family_cols):
        return P_mix

    P_spec = softmax(spec_arr, axis=-1)

    if spec_classes is not None and len(spec_classes) == len(family_cols):
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in spec_classes]
        if all(labels[c] in src_c for c in family_cols):
            c_idx = [src_c.index(labels[c]) for c in family_cols]
            P_spec = P_spec[:, c_idx]

    fam_mass = P_out[:, family_cols].sum(axis=1)
    has_signal = np.ptp(spec_arr, axis=1) > 1e-6
    fam_mask = (np.isin(R_main.argmax(axis=1), family_cols) | (fam_mass >= 0.35)) & has_signal

    if fam_mask.any():
        rows = np.flatnonzero(fam_mask)
        sub_mix = np.clip(P_out[np.ix_(rows, family_cols)], 1e-12, 1.0)
        tot_mass = sub_mix.sum(axis=1, keepdims=True)
        base_rel = sub_mix / tot_mass

        log_rel = (1.0 - float(gate_weight)) * np.log(base_rel) + float(gate_weight) * np.log(np.clip(P_spec[rows], 1e-12, 1.0))
        blended_rel = softmax(log_rel, axis=1)
        P_out[np.ix_(rows, family_cols)] = tot_mass * blended_rel

    return P_out


# =============================================================================
# 5. ORTHOGRAPHIC RULES & HANDWRITING SHIELD
# =============================================================================


def build_orthographic_penalty_matrix(
    curly_quote_log_penalty: float = -1.00,  # Rule 1: Claude & GLM never emit curly quotes
    multi_em_dash_log_penalty: float = 0.0,  # Rule 2: Multiple prose em-dashes (rare in Gemini)
    straight_contr_log_penalty: float = 0.0,  # Rule 3: Straight contractions
    curly_contr_log_penalty: float = 0.0,  # Rule 4: Curly contractions
) -> np.ndarray:
    """
    Builds the (4, 18) log-penalty matrix W_pen mapping discrete orthographic
    rule indicators to the 18 canonical model classes.

    Empirical Foundations:
    ----------------------
    - Rule 1 (Curly Quotes): In the competition training corpus, Claude (0 / 1049)
      and GLM (0 / 300) NEVER produce typographic curly quotes (‘, ’, “, ”) in standard prose.
      They consistently output straight quotes (', "). If a passage contains curly quotes,
      the probability of it being Claude or GLM is astronomically low.
      Log penalty of -1.0 reduces their odds by a factor of e^(-1) ≈ 0.368.
    - Rule 2 (Multiple Em-Dashes): Gemini models rarely use two or more prose em-dashes (—).
    - Rule 3 (Straight Contractions): Distinctive signature for gpt-5.6-sol vs sisters.
    - Rule 4 (Curly Contractions): Reinforces rejection of Claude & GLM.
    """
    W_pen = np.zeros((4, K), dtype=np.float64)
    claude_and_glm_idx = FAMILIES["claude"] + FAMILIES["glm"]
    gemini_idx = FAMILIES["gemini"]
    sol_idx = LAB2I["gpt-5.6-sol"]

    # Rule 1: Any curly quote -> penalize Claude (5 models) and GLM (2 models)
    W_pen[0, claude_and_glm_idx] = float(curly_quote_log_penalty)
    # Rule 2: >= 2 prose em-dashes -> penalize Gemini (4 models)
    W_pen[1, gemini_idx] = float(multi_em_dash_log_penalty)
    # Rule 3: >= 2 prose straight contractions (with 0 curly) -> penalize gpt-5.6-sol
    W_pen[2, sol_idx] = float(straight_contr_log_penalty)
    # Rule 4: >= 1 prose curly contraction -> reinforce penalty on Claude & GLM
    W_pen[3, claude_and_glm_idx] = float(curly_contr_log_penalty)

    return W_pen


def _extract_rules_fallback(texts: list[str]) -> np.ndarray:
    """
    Fallback discrete rules extractor using regex when lexical_engine is unavailable.
    Detects curly quotes, em-dashes, and straight/curly prose contractions.
    """
    re_code = re.compile(r"```[\s\S]*?```|`[^`\n]+`")
    re_str = re.compile(r"[A-Za-z]'[A-Za-z]")
    re_cur = re.compile(r"[A-Za-z]\u2019[A-Za-z]")
    rules = np.zeros((len(texts), 4), dtype=np.float32)

    for i, raw_t in enumerate(texts):
        t = str(raw_t)
        prose = re_code.sub(" ", t)  # Strip inline and block code before checking prose
        raw_curly = t.count("\u2018") + t.count("\u2019") + t.count("\u201c") + t.count("\u201d")
        rules[i, 0] = 1.0 if raw_curly > 0 else 0.0
        rules[i, 1] = 1.0 if prose.count("\u2014") >= 2 else 0.0
        rules[i, 2] = 1.0 if (len(re_str.findall(prose)) >= 2 and (t.count("\u2018") + t.count("\u2019")) == 0) else 0.0
        rules[i, 3] = 1.0 if len(re_cur.findall(prose)) >= 1 else 0.0

    return rules


def apply_unified_handwriting_shield(
    P_mix: np.ndarray,
    test_texts: list[str],
    log_p_handwriting: np.ndarray | None = None,
    rule_flags: np.ndarray | None = None,
    curly_quote_multiplier: float = 0.36787944117144233,  # exp(-1.0)
    multi_em_dash_log_penalty: float = 0.0,
    straight_contr_log_penalty: float = 0.0,
    curly_contr_log_penalty: float = 0.0,
    continuous_hw_weight: float = 0.0,
) -> np.ndarray:
    """
    Single unified point of application for ALL handwriting and orthographic rules:
    1. Computes vectorized discrete log-penalties via matrix product: rule_flags @ W_pen
    2. Adds continuous 26-feature handwriting Gaussian log-likelihoods
    3. Re-normalizes rows to the probability simplex via softmax.
    """
    if rule_flags is None:
        if "extract_handwriting_and_orthographic_features" in globals():
            _, rule_flags = extract_handwriting_and_orthographic_features(test_texts)
        else:
            rule_flags = _extract_rules_fallback(test_texts)

    curly_log_pen = float(np.log(max(curly_quote_multiplier, 1e-12)))
    W_pen = build_orthographic_penalty_matrix(
        curly_quote_log_penalty=curly_log_pen,
        multi_em_dash_log_penalty=multi_em_dash_log_penalty,
        straight_contr_log_penalty=straight_contr_log_penalty,
        curly_contr_log_penalty=curly_contr_log_penalty,
    )

    log_P = np.log(np.clip(P_mix.astype(np.float64), 1e-12, 1.0))

    # 1. Discrete Bayesian rule subtraction (vectorized matrix multiply)
    log_P += rule_flags.astype(np.float64) @ W_pen

    # 2. Continuous 26-feature handwriting log-likelihood contribution
    if log_p_handwriting is not None and continuous_hw_weight > 0.0:
        log_P += float(continuous_hw_weight) * log_p_handwriting.astype(np.float64)

    return softmax(log_P, axis=1)


def apply_curly_quote_shield(P_mix: np.ndarray, test_texts: list[str], multiplier: float = 0.36787944117144233) -> np.ndarray:
    """Backward-compatible wrapper invoking the unified handwriting shield."""
    return apply_unified_handwriting_shield(
        P_mix=P_mix,
        test_texts=test_texts,
        log_p_handwriting=None,
        rule_flags=None,
        curly_quote_multiplier=multiplier,
        multi_em_dash_log_penalty=0.0,
        straight_contr_log_penalty=0.0,
        curly_contr_log_penalty=0.0,
        continuous_hw_weight=0.0,
    )


# =============================================================================
# 6. SINKHORN-KNOPP COLUMN MASS BALANCING
# =============================================================================


def sinkhorn_balancing(
    P: np.ndarray,
    iters: int = 100,
    sinkhorn_strength: float = 0.65,
    target_mix: float = 0.35,
    train_prior: np.ndarray | None = None,
    allow_pure: bool = False,
) -> np.ndarray:
    """
    Soft Multiplicative Sinkhorn-Knopp Column Marginal Balancing.
    =============================================================
    Equalizes global class marginals using Soft Sinkhorn shrinkage.
    Pure Sinkhorn (1.0 / 1.0) is disallowed by default as it destroys test performance
    by hard-forcing uniform marginals on batches with natural class variance.

    Parameters:
    -----------
    P : np.ndarray of shape (N, 18)
        Probability matrix to balance.
    iters : int
        Number of alternating row/column scaling steps (default 100).
    sinkhorn_strength : float
        Soft shrinkage interpolation between original P and balanced Q (default 0.65).
        Capped at 0.75 unless allow_pure=True.
    target_mix : float
        Convex blend between predicted marginals and prior (default 0.35).
        Capped at 0.50 unless allow_pure=True.
    train_prior : np.ndarray of shape (18,), optional
        Empirical training class frequencies (used when target_mix < 1.0).
    allow_pure : bool, optional
        Whether to allow 100% hard pure Sinkhorn (default False).

    Returns:
    --------
    Balanced probability matrix Q of shape (N, 18).
    """
    if len(P) < 18 or sinkhorn_strength <= 0.0:
        return P

    P_work = np.clip(P.astype(np.float64), 1e-12, 1.0)
    K_cols = P_work.shape[1]

    # Target column marginal
    if train_prior is not None:
        p_train = np.clip(train_prior.astype(np.float64), 1e-6, None)
        p_train /= p_train.sum()
    else:
        p_train = np.full(K_cols, 1.0 / K_cols, dtype=np.float64)

    eff_mix = target_mix if allow_pure else min(target_mix, 0.50)
    if eff_mix < 1.0:
        pred_marg = np.clip(P_work.mean(axis=0), 1e-6, None)
        pred_marg /= pred_marg.sum()
        rho = float(np.clip(eff_mix, 0.0, 1.0))
        target_col = (1.0 - rho) * pred_marg + rho * p_train
        target_col = np.clip(target_col, 1e-6, None)
        target_col /= target_col.sum()
    else:
        target_col = p_train

    # Alternating row and column projections
    Q = P_work.copy()
    for _ in range(iters):
        # Row normalization: sum across classes = 1.0
        Q /= Q.sum(axis=1, keepdims=True).clip(min=1e-12)
        # Column scaling: sum across samples for class k = N * target_col[k]
        Q *= (len(Q) * target_col[None, :]) / Q.sum(axis=0, keepdims=True).clip(min=1e-12)

    # Final row normalization
    Q /= Q.sum(axis=1, keepdims=True).clip(min=1e-12)

    if allow_pure and sinkhorn_strength >= 1.0:
        return Q

    # Soft logarithmic mixture between original P and balanced Q (pure Sinkhorn disallowed)
    eff_strength = float(np.clip(sinkhorn_strength, 0.0, 0.75 if not allow_pure else 1.0))
    if eff_strength <= 0.0:
        return P_work

    log_P = (1.0 - eff_strength) * np.log(np.clip(P_work, 1e-12, None)) + eff_strength * np.log(np.clip(Q, 1e-12, None))
    return softmax(log_P, axis=1)


# =============================================================================
# 7. TYPOBAYES POSTPROCESSOR (HIERARCHICAL BETA-BERNOULLI SHRINKAGE)
# =============================================================================


def apply_typobayes_postprocess(
    P_or_logP: np.ndarray,
    train_texts: list[str],
    y_train: np.ndarray,
    test_texts: list[str],
    labels: list = LABELS,
    kappa: float = 24.0,
    evidence_scale: float = 1.2,
    evidence_cap: float = 0.30,
) -> np.ndarray:
    """
    Applies Hierarchical Beta-Bernoulli 37-Marker TypoBayes postprocessing.

    Concept:
    --------
    Extracts 37 orthographic, punctuation, and structural markdown markers
    (e.g., ' — ', '`', '###', 'However,', '\n- ') across training texts and models.
    Uses Beta-Bernoulli empirical Bayes with prior strength kappa to calculate
    smoothed class-conditional emission rates, deriving log-likelihood ratio evidence.
    Applies conservative confidence gating so high-confidence predictions are not overridden.
    """
    if len(train_texts) == 0 or len(test_texts) == 0 or len(y_train) == 0:
        return P_or_logP

    y_tr = np.array([LAB2I[y] if isinstance(y, str) else int(y) for y in y_train], dtype=int)
    marks = [
        " — ",
        "—",
        "–",
        "’",
        "‘",
        "“",
        "”",
        "«",
        "»",
        '"',
        ";",
        ":",
        "(",
        ")",
        "[",
        "]",
        "**",
        "`",
        "…",
        "•",
        "\n",
        "\n\n",
        "e.g.",
        "i.e.",
        "However,",
        "Overall,",
        "In conclusion",
        "Therefore,",
        "Moreover,",
        "\t",
        " - ",
        " = ",
        "\n- ",
        "\n1.",
        "###",
        "```",
        "> ",
    ]
    K_cls = len(labels)

    def _extract_marker_row(t: str) -> list[float]:
        # Mutually exclusive decoupling for overlapping substrings:
        # 1. ' — ' vs unspaced '—'
        has_spaced_em = " — " in t
        t_no_spaced = t.replace(" — ", "")
        has_unspaced_em = "—" in t_no_spaced

        # 2. '\n\n', '\n- ', '\n1.' vs standalone '\n'
        has_double_nl = "\n\n" in t
        has_bullet = "\n- " in t
        has_num_list = "\n1." in t
        t_no_compound_nl = t.replace("\n\n", "").replace("\n- ", "").replace("\n1.", "")
        has_single_nl = "\n" in t_no_compound_nl

        row = []
        for m in marks:
            if m == " — ":
                row.append(float(has_spaced_em))
            elif m == "—":
                row.append(float(has_unspaced_em))
            elif m == "\n\n":
                row.append(float(has_double_nl))
            elif m == "\n- ":
                row.append(float(has_bullet))
            elif m == "\n1.":
                row.append(float(has_num_list))
            elif m == "\n":
                row.append(float(has_single_nl))
            else:
                row.append(float(m in t))
        return row

    Xtr = np.array([_extract_marker_row(t) for t in train_texts], dtype=np.float32)
    Xte = np.array([_extract_marker_row(t) for t in test_texts], dtype=np.float32)
    freq = np.bincount(y_tr, minlength=K_cls).astype(np.float64)
    hits = np.stack([Xtr[y_tr == k].sum(axis=0) for k in range(K_cls)], axis=0).astype(np.float64)

    # Hierarchical Beta-Bernoulli smoothing: class rates shrink toward global rate
    global_rate = (Xtr.sum(axis=0) + 2.0) / (len(Xtr) + 4.0)
    rate = (hits + kappa * global_rate[None, :]) / (freq[:, None] + kappa)
    rate = np.clip(rate, 0.01, 0.99)
    global_rate = np.clip(global_rate, 0.01, 0.99)

    log_rate = np.log(rate)
    log_not_rate = np.log1p(-rate)
    base_log_rate = np.log(global_rate)
    base_log_not_rate = np.log1p(-global_rate)

    # Weight markers by discriminative power across classes
    signal = np.sqrt(np.mean((rate - global_rate[None, :]) ** 2, axis=0))
    med_signal = float(np.median(signal[signal > 0])) if np.any(signal > 0) else 1.0
    fw = signal / max(med_signal, 1e-6)
    fw = np.clip(fw, 0.25, 2.0)

    # Matrix form: weighted Bernoulli log-likelihood ratio for each class
    evidence = (Xte * fw[None, :]) @ log_rate.T + ((1.0 - Xte) * fw[None, :]) @ log_not_rate.T
    evidence -= (((Xte * fw[None, :]) @ base_log_rate) + (((1.0 - Xte) * fw[None, :]) @ base_log_not_rate))[:, None]
    evidence -= evidence.mean(axis=1, keepdims=True)
    evidence = np.clip(evidence_scale * evidence / max(float(fw.sum()), 1e-6), -evidence_cap, evidence_cap)

    # Convert inputs to probabilities and log-probabilities
    if np.all(P_or_logP <= 0.0) and not np.all(P_or_logP == 0.0):
        raw_logP = P_or_logP.copy()
        probs = softmax(raw_logP, axis=1)
    else:
        probs = P_or_logP.copy()
        raw_logP = np.log(np.clip(probs, 1e-12, 1.0))

    # Gating: only apply evidence when model is uncertain, text is long, and markers are present
    sorted_p = np.sort(probs, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    confidence_gate = np.clip((0.60 - margin) / 0.60, 0.0, 1.0)
    word_count = np.array([len(re.findall(r"\b\w+\b", str(t))) for t in test_texts], dtype=np.float32)
    length_gate = np.clip(np.sqrt(np.maximum(word_count, 1.0) / 80.0), 0.40, 1.0)
    active_markers = Xte.sum(axis=1)
    coverage_gate = np.clip(np.sqrt((active_markers + 1.0) / 4.0), 0.45, 1.0)
    gate = confidence_gate * length_gate * coverage_gate

    safe_logits = raw_logP + gate[:, None] * evidence
    return softmax(safe_logits, axis=1)


# =============================================================================
# 8. FINAL SUBMISSION EXPORT
# =============================================================================


def generate_submission(P_final: np.ndarray, test_ids: pd.Series, working_dir: Path) -> tuple[pd.DataFrame, Path, Path]:
    """
    Generates submission.parquet and submission.csv with exact Kaggle competition schema:
    - id: test sample identifier string
    - model: predicted model class name string (argmax over calibrated probabilities)
    """
    preds = [LABELS[idx] for idx in P_final.argmax(axis=1)]
    df_sub = pd.DataFrame({"id": test_ids, "model": preds})
    sub_parquet_path = working_dir / "submission.parquet"
    sub_csv_path = working_dir / "submission.csv"

    df_sub.to_parquet(sub_parquet_path, index=False)
    df_sub.to_csv(sub_csv_path, index=False)
    return df_sub, sub_parquet_path, sub_csv_path


In [ ]:
# === MODULE: src/namethatllm/ablation.py ===
"""
Zero-Leakage Unseen Holdout Benchmark & Forward Stepwise Ablation Engine
========================================================================
Constructs a strictly unseen evaluation dataset from mounted parquet/CSV corpora
by subtracting all passages present in `train.csv` and `pool_v4.parquet`
(the training sets of the pre-trained Stage-1/Stage-2 checkpoints), quarantines
those holdout passages from `lexical_engine.py` and `trainer.py`, and evaluates:
1. Standalone (Solo) Unseen Accuracy of each model.
2. Forward Stepwise Cumulative Accuracy (1 model -> +2nd model -> ... -> Full Ensemble).
"""

from itertools import combinations
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score


# =============================================================================
# HELPER FUNCTIONS: MASS BALANCING & ZERO-LEAKAGE TEXT HASHING
# =============================================================================


def _ablation_sinkhorn(
    P: np.ndarray,
    iters: int = 100,
    eps: float = 1e-12,
    y_true: np.ndarray | None = None,
    allow_pure: bool = False,
) -> np.ndarray:
    """
    Fast vectorized Sinkhorn-Knopp column mass balancing for evaluation.
    Delegates directly to sinkhorn_balancing in namethatllm.postprocess.
    """
    train_prior = np.bincount(y_true, minlength=P.shape[1]).astype(np.float64) if y_true is not None and len(y_true) == len(P) else None
    return _sinkhorn_balancing(
        P,
        iters=iters,
        sinkhorn_strength=1.0 if allow_pure else 0.65,
        target_mix=1.0 if allow_pure else 0.35,
        train_prior=train_prior,
        allow_pure=allow_pure,
    )


def _fast_sinkhorn_bacc(lp_mat: np.ndarray, y_true: np.ndarray, iters: int = 40) -> float:
    """
    Quickly computes balanced accuracy of log-probabilities after 40 Sinkhorn iterations.
    Used inside inner optimization loops (line search, greedy selection) for speed.
    """
    P = softmax(lp_mat, axis=1)
    if len(P) >= 18:
        P = _ablation_sinkhorn(P, iters=iters, y_true=y_true)
    return float(balanced_accuracy_score(y_true, P.argmax(axis=1)))


def build_unseen_holdout_dataset(
    input_dir: Path,
    train_path: Path | None = None,
    samples_per_class: int = 20,
    seed: int = 42,
) -> tuple[pd.DataFrame, set[str]]:
    """
    Builds a balanced 18-class unseen holdout dataset (`df_unseen`) by:
    1. Hashing all texts in `train.csv` and `pool_v4.parquet` (and `pool_v1.parquet`) as `contaminated_hashes`.
    2. Scanning `nlm_matched_clean.parquet`, `nlm_pool_v5.parquet`, and `openrouter-harvest`
       for passages whose text hash is strictly NOT in `contaminated_hashes`.
    3. Sampling up to `samples_per_class` passages per class (e.g., 18 * 20 = 360 unseen passages)
       and returning `(df_unseen, holdout_hashes)` so `lexical_engine.py` and `trainer.py`
       exclude `holdout_hashes` from training!
    """
    input_dir = Path(input_dir) if input_dir else Path(".")
    contaminated_hashes: set[str] = set()

    # 1. Mark all train.csv rows as contaminated (seen by all Stage-2 LoRAs)
    if train_path and Path(train_path).exists():
        try:
            df_tr = pd.read_csv(train_path, usecols=["text"])
            contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
        except Exception:
            try:
                df_tr = pd.read_csv(train_path)
                if "text" in df_tr.columns:
                    contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
            except Exception:
                pass

    # 2. Mark only pool_v4.parquet and pool_v1.parquet as contaminated
    # (seen by Part-2 LoRAs, DeBERTa, inhouse)
    for pat in ["pool_v4.parquet", "pool_v1.parquet"]:
        for p in input_dir.rglob(pat):
            try:
                df_p = pd.read_parquet(p, columns=["text"])
                contaminated_hashes.update(_text_hash(t) for t in df_p["text"].fillna(""))
            except Exception:
                pass

    # 3. Collect candidate passages in canonical trainer quarantine order
    candidate_dfs = []
    seen_paths = set()

    def _process_candidate_file(p: Path) -> pd.DataFrame | None:
        try:
            df_c = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
            if "label" in df_c.columns and "model" not in df_c.columns:
                df_c = df_c.rename(columns={"label": "model"})
            finish_col = next((c for c in ["finish_reason", "finish"] if c in df_c.columns), None)
            if finish_col:
                df_c = df_c[df_c[finish_col].astype(str).str.lower().eq("stop")]
                df_c = df_c[df_c["text"].fillna("").astype(str).str.split().str.len().between(25, 360)]
            if "text" in df_c.columns and "model" in df_c.columns:
                keep_cols = [c for c in ["id", "prompt_id", "text", "model"] if c in df_c.columns]
                df_c = df_c[keep_cols].dropna(subset=["text", "model"]).copy()
                df_c = df_c[df_c["model"].isin(LABELS)]
                df_c["_hash"] = [_text_hash(t) for t in df_c["text"]]
                df_c = df_c[~df_c["_hash"].isin(contaminated_hashes)]
                if len(df_c) > 0:
                    return df_c
        except Exception:
            pass
        return None

    candidate_patterns = [
        "nlm_matched_clean.parquet",
        "nlm_pool_v5.parquet",
        "nlm_pool_v2.parquet",
        "*openrouter*.parquet",
        "*openrouter*.csv",
        "*harvest*.parquet",
        "*harvest*.csv",
    ]
    for pat in candidate_patterns:
        for p in input_dir.rglob(pat):
            if p in seen_paths:
                continue
            p_name = p.name.lower()
            if any(skip in p_name for skip in ["train", "test", "submission", "pool_v4", "pool_v1"]):
                continue
            seen_paths.add(p)
            df_c = _process_candidate_file(p)
            if df_c is not None:
                candidate_dfs.append(df_c)

    # Also scan any directory or file matching openrouter, harvest, passages, generations
    for ext in ["*.parquet", "*.csv"]:
        for p in input_dir.rglob(ext):
            if p in seen_paths:
                continue
            p_str = str(p).lower().replace("\\", "/")
            p_name = p.name.lower()
            if any(skip in p_name for skip in ["train", "test", "submission", "pool_v4", "pool_v1"]):
                continue
            if any(k in p_str for k in ["openrouter", "harvest", "passages", "generations"]):
                seen_paths.add(p)
                df_c = _process_candidate_file(p)
                if df_c is not None:
                    candidate_dfs.append(df_c)

    # Fallback to any remaining non-training parquet/csv if still empty
    if not candidate_dfs:
        for ext in ["*.parquet", "*.csv"]:
            for p in input_dir.rglob(ext):
                if p in seen_paths:
                    continue
                p_name = p.name.lower()
                if any(x in p_name for x in ["train", "test", "submission", "pool_v4", "pool_v1"]):
                    continue
                seen_paths.add(p)
                df_c = _process_candidate_file(p)
                if df_c is not None:
                    candidate_dfs.append(df_c)

    if not candidate_dfs:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_pool_unseen = pd.concat(candidate_dfs, ignore_index=True).drop_duplicates(subset=["_hash"]).reset_index(drop=True)

    v5_hashes: set[str] = set()
    for pat in ["nlm_pool_v5.parquet", "*pool_v5*.parquet"]:
        for p in input_dir.rglob(pat):
            try:
                df_v5 = pd.read_parquet(p, columns=["text"])
                v5_hashes.update(_text_hash(t) for t in df_v5["text"].fillna(""))
            except Exception:
                pass
    strictly_disjoint = df_pool_unseen[~df_pool_unseen["_hash"].isin(v5_hashes)].reset_index(drop=True)
    if strictly_disjoint["model"].nunique() == K and (strictly_disjoint["model"].value_counts() >= samples_per_class).all():
        pool_to_sample = strictly_disjoint
    else:
        pool_to_sample = df_pool_unseen

    has_source_prompt = "prompt_id" in pool_to_sample.columns and pool_to_sample["prompt_id"].notna().sum() >= 36
    if has_source_prompt:
        prompt_richness = pool_to_sample.groupby("prompt_id")["model"].nunique()

    # Stratified balanced sampling across all 18 classes from a single consistent pool
    rng = np.random.RandomState(seed)
    sampled_parts = []
    for m in LABELS:
        sub = pool_to_sample[pool_to_sample["model"] == m].reset_index(drop=True)
        if len(sub) > 0:
            n_take = min(samples_per_class, len(sub))
            if has_source_prompt:
                sub_prompts = sub["prompt_id"].map(prompt_richness).fillna(1).values
                jitter = rng.rand(len(sub))
                sort_keys = np.lexsort((jitter, -(sub_prompts >= 2).astype(int), -sub_prompts))
                sub_sorted = sub.iloc[sort_keys].reset_index(drop=True)
                sampled_parts.append(sub_sorted.iloc[:n_take])
            else:
                idx = rng.choice(len(sub), size=n_take, replace=False)
                sampled_parts.append(sub.iloc[idx])

    if not sampled_parts:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_unseen = pd.concat(sampled_parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    if "id" not in df_unseen.columns:
        df_unseen["id"] = [f"unseen_{i}" for i in range(len(df_unseen))]

    # Compute valid prompt clusters
    clusters = None
    if len(df_unseen) >= 2:
        try:
            tfidf = TfidfVectorizer(stop_words="english", max_features=10000, sublinear_tf=True)
            X_tfidf = tfidf.fit_transform(df_unseen["text"].fillna("").tolist())
            D = 1.0 - (X_tfidf @ X_tfidf.T).toarray().clip(0.0, 1.0)
            np.fill_diagonal(D, 0.0)

            use_source_prompt = False
            if "prompt_id" in df_unseen.columns and df_unseen["prompt_id"].notna().sum() == len(df_unseen):
                codes = pd.factorize(df_unseen["prompt_id"])[0]
                if len(codes) > 0:
                    counts = np.bincount(codes)
                    if (counts[codes] >= 2).mean() >= 0.40:
                        clusters = split_oversized_clusters_dist(D, codes + 1, max_cluster_size=14)
                        use_source_prompt = True

            if not use_source_prompt:
                cond_d = squareform(D, checks=False)
                raw_clusters = fcluster(linkage(cond_d, method="average"), t=0.90, criterion="distance")
                clusters = split_oversized_clusters_dist(D, raw_clusters, max_cluster_size=14)
        except Exception:
            clusters = np.arange(1, len(df_unseen) + 1, dtype=int)
    else:
        clusters = np.ones(len(df_unseen), dtype=int)

    df_unseen["cluster_id"] = clusters
    df_unseen["prompt_id"] = clusters

    holdout_hashes = set(df_unseen["_hash"].tolist())
    print(
        f"[Unseen Benchmark] Built strictly unseen holdout set: {len(df_unseen)} rows "
        f"across {df_unseen['model'].nunique()} classes (0% overlap with train.csv, pool_v4.parquet & pool_v5.parquet)."
    )
    return df_unseen, holdout_hashes


class UnseenStepwiseAuditor:
    """
    Evaluates every model on the strictly unseen holdout dataset (`df_unseen`),
    reporting:
    1. Table 1 — Standalone (Solo) Unseen Balanced Accuracy (sorted descending).
    2. Table 2 — Greedy Forward Stepwise Selection (optimal combinations, stopping when lift < 0.0).
    3. Table 3 — Exhaustive All-2^M Subset Search & Leave-One-Out (LOO) Attribution.
    """

    def __init__(self, y_unseen: np.ndarray, groups_unseen: np.ndarray, N_test: int):
        self.y_unseen = np.asarray(y_unseen, dtype=int) if len(y_unseen) > 0 else np.array([], dtype=int)
        if groups_unseen is not None and len(groups_unseen) > 0:
            try:
                self.groups_unseen = np.asarray(groups_unseen, dtype=int)
            except (ValueError, TypeError):
                codes, _ = pd.factorize(groups_unseen)
                self.groups_unseen = (codes + 1).astype(int)
        else:
            self.groups_unseen = np.array([], dtype=int)
        self.N_test = int(N_test)
        self.models: list[dict[str, Any]] = []

    def add_model(
        self,
        name: str,
        tier: str,
        weight: float,
        unseen_log_probs: np.ndarray,
        test_log_probs: np.ndarray,
    ):
        u_lp = (
            calibrate_model_lp(unseen_log_probs, model_name=name)
            if len(self.y_unseen) > 0 and len(unseen_log_probs) == len(self.y_unseen)
            else np.zeros((0, K), dtype=np.float64)
        )
        t_lp = (
            calibrate_model_lp(test_log_probs, model_name=name)
            if len(test_log_probs) == self.N_test
            else np.zeros((self.N_test, K), dtype=np.float64)
        )
        solo_raw = (
            float(balanced_accuracy_score(self.y_unseen, u_lp.argmax(axis=1)))
            if len(self.y_unseen) > 0 and len(u_lp) == len(self.y_unseen)
            else 0.0
        )
        solo_sink = (
            _fast_sinkhorn_bacc(u_lp, self.y_unseen, iters=200)
            if len(self.y_unseen) >= 18 and len(u_lp) == len(self.y_unseen)
            else solo_raw
        )
        self.models.append(
            {
                "name": name,
                "tier": tier,
                "weight": float(weight),
                "prior_weight": float(weight),
                "unseen_lp": u_lp,
                "test_lp": t_lp,
                "solo_unseen_bacc": solo_raw,
                "solo_unseen_sinkhorn_bacc": solo_sink,
            }
        )

    def optimize_weights_and_blend(
        self,
        working_dir: Path,
        extra_test_log_probs: list[tuple[str, float, np.ndarray]] | None = None,
        optimize_on_unseen: bool = True,
        lock_prior_weights: bool = False,
    ) -> tuple[np.ndarray, dict[str, float], pd.DataFrame]:
        """
        1. Runs Greedy Forward Stepwise Selection + Coordinate Line Search on `df_unseen`
           to determine each model's optimal individual weight `w_star` (>= 0.0).
        2. Any model that fails to improve unseen balanced accuracy gets `w_star = 0.0` (PRUNED).
        3. Combines all active models + extra_test_log_probs (wpca_dp, spectral_bank) directly
           in flat log-probability space:
               LOGIT_FLAT = sum_m (w_star[m] * test_lp[m])
        """
        if len(self.models) == 0:
            return np.ones((self.N_test, K), dtype=np.float64) / float(K), {}, pd.DataFrame()

        if not optimize_on_unseen or len(self.y_unseen) < 36:
            weights_map = {m["name"]: m["weight"] for m in self.models}
            logit_flat = np.zeros((self.N_test, K), dtype=np.float64)
            for m in self.models:
                logit_flat += m["weight"] * m["test_lp"]
            if extra_test_log_probs:
                for ex_name, w_ex, lp_ex in extra_test_log_probs:
                    if len(lp_ex) == self.N_test and w_ex > 0.0:
                        logit_flat += float(w_ex) * log_softmax(lp_ex, axis=1)
                        weights_map[ex_name] = float(w_ex)
            P_mix = softmax(logit_flat, axis=1)
            df_summary = pd.DataFrame(
                [
                    {
                        "model": m["name"],
                        "tier": m["tier"],
                        "prior_weight": m.get("prior_weight", m["weight"]),
                        "learned_weight": m["weight"],
                    }
                    for m in self.models
                ]
            )
            return P_mix, weights_map, df_summary

        # Composite objective function: Balanced Accuracy with mild Negative Log-Likelihood regularization.
        # Why: Hard 0-1 accuracy has flat plateaus where many weight values produce identical discrete predictions.
        # Adding -0.015 * NLL provides smooth, well-conditioned gradients favoring well-calibrated confidence.
        def _composite_obj(lp_mat: np.ndarray) -> tuple[float, float]:
            P = softmax(lp_mat, axis=1)
            P_s = _ablation_sinkhorn(P, iters=60, y_true=self.y_unseen) if len(P) >= 18 else P
            bacc = float(balanced_accuracy_score(self.y_unseen, P_s.argmax(axis=1)))
            nll = float(-np.mean(np.log(np.clip(P_s[np.arange(len(self.y_unseen)), self.y_unseen], 1e-9, 1.0))))
            return bacc - 0.015 * nll, bacc

        M = len(self.models)

        def _effective_bacc(m: dict[str, Any]) -> float:
            return float(max(m.get("solo_unseen_bacc", 0.0), m.get("solo_unseen_sinkhorn_bacc", 0.0)))

        # Sort candidate models by solo performance on unseen holdout descending
        order = sorted(range(M), key=lambda i: _effective_bacc(self.models[i]), reverse=True)

        if lock_prior_weights:
            # Conservative mode: keep fixed prior weights from configuration, pruning sub-threshold models
            active_weights = np.zeros(M, dtype=np.float64)
            for idx, m in enumerate(self.models):
                prior_w = float(m.get("prior_weight", m["weight"]))
                eff_b = _effective_bacc(m)
                if eff_b < 0.22 or prior_w <= 0.0:
                    active_weights[idx] = 0.0
                else:
                    active_weights[idx] = prior_w
        else:
            # Adaptive mode: Greedy forward stepwise selection + coordinate scale search
            active_weights = np.zeros(M, dtype=np.float64)
            valid_candidates = [i for i in order if _effective_bacc(self.models[i]) >= 0.22 and self.models[i]["weight"] > 0.0]
            if not valid_candidates:
                for idx, m in enumerate(self.models):
                    active_weights[idx] = m["weight"]
            else:
                # Step 1: Initialize ensemble with the single best solo performer
                first_idx = valid_candidates[0]
                m_first = self.models[first_idx]
                mults_first = (0.75, 1.00, 1.25) if _effective_bacc(m_first) >= 0.34 else (0.25, 0.50, 0.75, 1.00)
                best_first_w = m_first["weight"]
                best_first_obj = -1e9
                best_first_bacc = 0.0
                for mult in mults_first:
                    cand_w = m_first["weight"] * mult
                    obj_val, bacc_val = _composite_obj(cand_w * m_first["unseen_lp"])
                    if obj_val > best_first_obj:
                        best_first_obj = obj_val
                        best_first_bacc = bacc_val
                        best_first_w = cand_w
                active_weights[first_idx] = best_first_w
                cum_u_lp = active_weights[first_idx] * m_first["unseen_lp"]
                best_obj = best_first_obj
                best_bacc = best_first_bacc

                remaining = list(valid_candidates[1:])
                while remaining:
                    step_best_idx = None
                    step_best_w = 0.0
                    step_best_obj = best_obj
                    step_best_bacc = best_bacc

                    for idx in remaining:
                        m_info = self.models[idx]
                        prior_w = m_info["weight"]
                        u_lp = m_info["unseen_lp"]
                        eff_b = _effective_bacc(m_info)

                        if eff_b < 0.22 or prior_w <= 0.0:
                            continue

                        mults = (0.75, 1.00, 1.25) if eff_b >= 0.34 else (0.25, 0.50, 0.75, 1.00)
                        for mult in mults:
                            cand_w = prior_w * mult
                            trial_lp = cum_u_lp + cand_w * u_lp
                            obj_val, bacc_val = _composite_obj(trial_lp)
                            if obj_val > step_best_obj + 1e-6:
                                step_best_obj = obj_val
                                step_best_bacc = bacc_val
                                step_best_idx = idx
                                step_best_w = cand_w

                    if step_best_idx is not None and step_best_w > 0.0:
                        active_weights[step_best_idx] = step_best_w
                        cum_u_lp = cum_u_lp + step_best_w * self.models[step_best_idx]["unseen_lp"]
                        best_obj = step_best_obj
                        best_bacc = step_best_bacc
                        remaining.remove(step_best_idx)
                    else:
                        break

        if not lock_prior_weights:
            # Post-loop enforcement:
            # 1. Enforce 0.75 * prior_weight floor on all validated models (eff_bacc >= 0.34 and weight > 0)
            #    so nlm_pool_v5 holdout leakage on qwen3_14b_v5 never zeroes out Part-2, in-house, DeBERTa, or LinearSVC!
            # 2. Prune any model with eff_bacc < 0.30 or weight <= 0 to strictly 0.0.
            for idx, m_info in enumerate(self.models):
                eff_b = _effective_bacc(m_info)
                if eff_b < 0.22 or m_info["weight"] <= 0.0:
                    active_weights[idx] = 0.0
                elif eff_b >= 0.34 and m_info["weight"] > 0.0:
                    active_weights[idx] = max(active_weights[idx], 0.75 * m_info["weight"])

        weights_map: dict[str, float] = {}
        logit_flat = np.zeros((self.N_test, K), dtype=np.float64)

        for idx, m in enumerate(self.models):
            m["prior_weight"] = m.get("prior_weight", m["weight"])
            m["learned_weight"] = float(active_weights[idx])
            m["weight"] = float(active_weights[idx])
            weights_map[m["name"]] = float(active_weights[idx])
            if active_weights[idx] > 0.0:
                logit_flat += active_weights[idx] * m["test_lp"]

        if extra_test_log_probs:
            for ex_name, w_ex, lp_ex in extra_test_log_probs:
                if len(lp_ex) == self.N_test and w_ex > 0.0:
                    logit_flat += float(w_ex) * log_softmax(lp_ex, axis=1)
                    weights_map[ex_name] = float(w_ex)

        P_mix = softmax(logit_flat, axis=1)
        df_summary = pd.DataFrame(
            [
                {"model": m["name"], "tier": m["tier"], "prior_weight": m.get("prior_weight", m["weight"]), "learned_weight": m["weight"]}
                for m in self.models
            ]
        )
        return P_mix, weights_map, df_summary

    def evaluate_marginal_lift(
        self,
        candidate_unseen_lp: np.ndarray,
        weight: float = 1.0,
    ) -> float:
        """
        Multi-scale marginal lift search: tests adding `candidate_unseen_lp` across
        multipliers [0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75, 1.00] * weight
        and returns the maximum Delta BAcc achieved at any positive scale.
        """
        if len(self.y_unseen) == 0 or len(candidate_unseen_lp) != len(self.y_unseen) or len(self.models) == 0:
            return 0.0

        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        for m in self.models:
            w_m = m["weight"] if "learned_weight" in m else (m["weight"] if m["weight"] > 0 else m.get("prior_weight", 0.15))
            if w_m <= 0.0:
                continue
            cum_unseen_lp += w_m * m["unseen_lp"]

        cur_bacc = _fast_sinkhorn_bacc(cum_unseen_lp, self.y_unseen, iters=40)
        cand_lp = log_softmax(np.asarray(candidate_unseen_lp, dtype=np.float64), axis=1)

        best_lift = -1.0
        full_weight_lift = 0.0
        for mult in (0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75, 1.00):
            cand_w = float(weight) * mult
            new_unseen_lp = cum_unseen_lp + cand_w * cand_lp
            new_bacc = _fast_sinkhorn_bacc(new_unseen_lp, self.y_unseen, iters=40)
            lift = new_bacc - cur_bacc
            if mult == 1.00:
                full_weight_lift = lift
            best_lift = max(best_lift, lift)

        if best_lift <= 0.0:
            return float(full_weight_lift)
        return float(best_lift)

    def print_and_save_report(self, working_dir: Path, final_test_preds: np.ndarray | None = None) -> pd.DataFrame:
        if not self.models or len(self.y_unseen) == 0:
            print("[Ablation] No models or unseen holdout samples available; skipping report.")
            return pd.DataFrame()

        # Part 1: Standalone (Solo) Accuracy, sorted descending
        print("\n" + "=" * 105)
        print(f" PART 1: STANDALONE (SOLO) ACCURACY ON STRICTLY UNSEEN HOLDOUT ({len(self.y_unseen)} Disjoint Passages)")
        print("=" * 105)
        print(
            f"{'Model / Checkpoint':<28} | {'Tier':<6} | {'Unseen Raw BAcc':<16} | {'Unseen+Sinkhorn BAcc':<20} | {'Agrees w/ Final Test'}"
        )
        print("-" * 105)
        sorted_solo = sorted(self.models, key=lambda m: m["solo_unseen_sinkhorn_bacc"], reverse=True)
        for m in sorted_solo:
            agree_str = "N/A"
            if final_test_preds is not None and len(final_test_preds) == self.N_test:
                pct = 100.0 * float((m["test_lp"].argmax(axis=1) == final_test_preds).mean())
                agree_str = f"{pct:5.1f}%"
            print(
                f"{m['name']:<28} | {m['tier']:<6} | {m['solo_unseen_bacc']:<16.4f} | {m['solo_unseen_sinkhorn_bacc']:<20.4f} | {agree_str}"
            )

        # Part 2: Greedy Forward Stepwise Selection (starts with strongest solo model, tests all candidates at each step)
        print("\n" + "=" * 105)
        print(" PART 2: GREEDY FORWARD STEPWISE SELECTION ON STRICTLY UNSEEN HOLDOUT (Optimal Combinations)")
        print("=" * 105)
        print(
            f"{'Step':<5} | {'Added Model':<28} | {'Wt':<5} | {'Cumul Raw':<10} | {'Cumul+Sink':<11} | {'Marginal Lift':<13} | {'Test Rows Flipped'}"
        )
        print("-" * 105)

        M = len(self.models)
        remaining = list(range(M))
        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        cum_test_lp = np.zeros((self.N_test, K), dtype=np.float64)
        prev_bacc = 0.0
        prev_test_preds = None
        records = []

        step_idx = 1
        while remaining:
            best_cand_idx = None
            best_cand_bacc = -1.0
            best_cand_raw = 0.0

            if step_idx == 1:
                # Pick model with highest solo unseen sinkhorn bacc
                for idx in remaining:
                    cand = self.models[idx]
                    cand_w = (
                        cand["weight"]
                        if "learned_weight" in cand
                        else (cand["weight"] if cand["weight"] > 0 else cand.get("prior_weight", 0.15))
                    )
                    if cand_w <= 0.0:
                        continue
                    cand_lp = cand_w * cand["unseen_lp"]
                    cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                    if cand_bacc > best_cand_bacc:
                        best_cand_bacc = cand_bacc
                        best_cand_idx = idx
                        best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))
            else:
                # Test all remaining candidate models
                for idx in remaining:
                    cand = self.models[idx]
                    cand_w = (
                        cand["weight"]
                        if "learned_weight" in cand
                        else (cand["weight"] if cand["weight"] > 0 else cand.get("prior_weight", 0.15))
                    )
                    if cand_w <= 0.0:
                        continue
                    cand_lp = cum_unseen_lp + cand_w * cand["unseen_lp"]
                    cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                    if cand_bacc > best_cand_bacc:
                        best_cand_bacc = cand_bacc
                        best_cand_idx = idx
                        best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))

            if best_cand_idx is None:
                break

            lift = best_cand_bacc - prev_bacc if step_idx > 1 else 0.0
            if step_idx > 1 and lift < 0.0:
                print(
                    f"[Greedy Forward Selection] Best remaining candidate '{self.models[best_cand_idx]['name']}' gives negative lift ({lift:+.4f}); stopping forward selection."
                )
                break

            m = self.models[best_cand_idx]
            remaining.remove(best_cand_idx)

            w = m["weight"] if "learned_weight" in m else (m["weight"] if m["weight"] > 0 else m.get("prior_weight", 0.15))
            cum_unseen_lp = cum_unseen_lp + w * m["unseen_lp"]
            cum_test_lp = cum_test_lp + w * m["test_lp"]

            lift_str = f"{lift:+.4f}" if step_idx > 1 else "BASELINE"
            cur_test_preds = _ablation_sinkhorn(softmax(cum_test_lp, axis=1), iters=40).argmax(axis=1) if self.N_test > 0 else np.array([])
            if prev_test_preds is None:
                flips = self.N_test
                flip_str = f"{flips} (Init)"
            else:
                flips = int((cur_test_preds != prev_test_preds).sum()) if len(cur_test_preds) == len(prev_test_preds) else 0
                flip_pct = 100.0 * flips / max(1, self.N_test)
                flip_str = f"{flips:4d} ({flip_pct:4.1f}%)"

            print(
                f"{step_idx:<5} | + {m['name']:<26} | {w:<5.2f} | "
                f"{best_cand_raw:<10.4f} | {best_cand_bacc:<11.4f} | {lift_str:<13} | {flip_str}"
            )
            prev_bacc = best_cand_bacc
            prev_test_preds = cur_test_preds
            records.append(
                {
                    "step": step_idx,
                    "model": m["name"],
                    "added_model": m["name"],
                    "tier": m["tier"],
                    "weight": w,
                    "solo_unseen_bacc": m["solo_unseen_bacc"],
                    "solo_unseen_sinkhorn_bacc": m["solo_unseen_sinkhorn_bacc"],
                    "cumul_unseen_raw_bacc": best_cand_raw,
                    "cumul_unseen_sinkhorn_bacc": best_cand_bacc,
                    "marginal_lift": lift,
                    "test_rows_flipped": flips,
                }
            )
            step_idx += 1

        print("=" * 105 + "\n")

        # Part 3: Exhaustive All 2^M Subset Search & LOO Attribution (for M <= 12)
        if 2 <= M <= 12:
            print("=" * 105)
            print(f" PART 3: GLOBALLY OPTIMAL SUBSET & LEAVE-ONE-OUT (LOO) ATTRIBUTION (All {2**M - 1} Non-Empty Subsets)")
            print("=" * 105)

            best_subset = None
            best_subset_bacc = -1.0

            for r in range(1, M + 1):
                for sub in combinations(range(M), r):
                    sub_unseen_lp = sum(
                        (self.models[i]["weight"] if self.models[i]["weight"] > 0 else self.models[i].get("prior_weight", 0.15))
                        * self.models[i]["unseen_lp"]
                        for i in sub
                    )
                    bacc = _fast_sinkhorn_bacc(sub_unseen_lp, self.y_unseen, iters=30)
                    if bacc > best_subset_bacc:
                        best_subset_bacc = bacc
                        best_subset = sub

            opt_names = [self.models[i]["name"] for i in best_subset]
            print(f"Globally Optimal Ensemble BAcc: {best_subset_bacc:.4f} (Uses {len(best_subset)} / {M} models)")
            print(f"Optimal Set: {', '.join(opt_names)}")
            print("-" * 105)
            print(f"{'Model / Checkpoint':<28} | {'In Optimal?':<12} | {'Weight':<7} | {'Delta_LOO (BAcc Drop If Excluded)'}")
            print("-" * 105)

            opt_set = set(best_subset)
            for i, m in enumerate(self.models):
                in_opt = i in opt_set
                in_str = "YES" if in_opt else "NO (Pruned)"
                if in_opt:
                    if len(best_subset) == 1:
                        delta_loo = best_subset_bacc
                    else:
                        loo_sub = [j for j in best_subset if j != i]
                        loo_lp = sum(
                            (self.models[j]["weight"] if self.models[j]["weight"] > 0 else self.models[j].get("prior_weight", 0.15))
                            * self.models[j]["unseen_lp"]
                            for j in loo_sub
                        )
                        loo_bacc = _fast_sinkhorn_bacc(loo_lp, self.y_unseen, iters=30)
                        delta_loo = best_subset_bacc - loo_bacc
                    loo_str = f"-{delta_loo:.4f}" if delta_loo >= 0 else f"+{abs(delta_loo):.4f}"
                else:
                    with_m_sub = list(best_subset) + [i]
                    with_m_lp = sum(
                        (self.models[j]["weight"] if self.models[j]["weight"] > 0 else self.models[j].get("prior_weight", 0.15))
                        * self.models[j]["unseen_lp"]
                        for j in with_m_sub
                    )
                    with_m_bacc = _fast_sinkhorn_bacc(with_m_lp, self.y_unseen, iters=30)
                    delta_add = with_m_bacc - best_subset_bacc
                    loo_str = f"{delta_add:+.4f} (Degrades if added)"

                print(f"{m['name']:<28} | {in_str:<12} | {m['weight']:<7.2f} | {loo_str}")

                for r in records:
                    if r["model"] == m["name"]:
                        r["in_optimal_subset"] = in_opt
                        r["delta_loo"] = delta_loo if in_opt else 0.0

            print("=" * 105 + "\n")

        df_rep = pd.DataFrame(records)
        working_dir = Path(working_dir)
        working_dir.mkdir(parents=True, exist_ok=True)
        df_rep.to_csv(working_dir / "unseen_stepwise_ablation.csv", index=False)
        return df_rep


In [ ]:
# === MODULE: src/namethatllm/pipeline.py ===
r"""
Top-Level Pipeline Orchestrator
===============================
This module orchestrates the complete end-to-end inference and evaluation pipeline:

Architectural Dataflow:
-----------------------
  [ Frozen Neural Readers ]              [ Fit Live in Pipeline ]
  • Qwen3-14B Part 2                     • Style SVC (4 views)
  • Qwen3-14B Part 1                     • Lexical Model (5 views)
  • Qwen2.5-14B Part 2                   • Exact Prompt Group Search (TF-IDF linkage)
  • DeBERTa-v3-Large                     • Spectral Bank (joint multi-view features)
  • 8B Patches (Claude, In-Style, LoRA24)• wPCA-DP Posterior (shared Gaussian DP)
                    \                           /
                     \                         /
                      ▼                       ▼
            ┌───────────────────────────────────────────────┐
            │ Stage 1: Quick Probe and Gate                 │
            │ Evaluates solo balanced accuracy on strictly   │
            │ unseen 360-row holdout set. Drops any reader  │
            │ scoring below min threshold (0.40 BAcc).      │
            └───────────────────────┬───────────────────────┘
                                    │
                                    ▼
            ┌───────────────────────────────────────────────┐
            │ Stage 2: Flat Ensemble / Greedy Stepwise      │
            │ Computes optimal combination weights using    │
            │ Leave-One-Out (LOO) attribution. Non-helpers  │
            │ receive weight 0.0 (pruned).                  │
            └───────────────────────┬───────────────────────┘
                                    │
                                    ▼
            ┌───────────────────────────────────────────────┐
            │ Stage 3: Neural Specialist Gates Layer        │
            │ Mass-preserving neural gating inside families: │
            │ • Claude Head     (Model 3 head, weight 0.15) │
            │ • Gemini LoRA     (Qwen3-14B, weight 0.65)    │
            │ • GLM LoRA        (Qwen3-14B, weight 0.70)    │
            │ • GPT LoRA        (Qwen3-14B, weight 0.55)    │
            └───────────────────────┬───────────────────────┘
                                    │
                                    ▼
            ┌───────────────────────────────────────────────┐
            │ Stage 4: Orthographic Punctuation Shield      │
            │ Bayesian tokenization penalties: curly quotes │
            │ get -1.0 log penalty for Claude and GLM.      │
            └───────────────────────┬───────────────────────┘
                                    │
                                    ▼
            ┌───────────────────────────────────────────────┐
            │ Stage 5: Cluster Uniqueness Repulsion         │
            │ Cavity mean-field repulsion: at-most-one text │
            │ per model within identical test prompt group. │
            └───────────────────────┬───────────────────────┘
                                    │
                                    ▼
            ┌───────────────────────────────────────────────┐
            │ Stage 6: Multiplicative Sinkhorn Balancing    │
            │ 300 alternating row/column normalization steps│
            │ matching uniform 1/18 global class priors.    │
            └───────────────────────┬───────────────────────┘
                                    │
                                    ▼
            ┌───────────────────────────────────────────────┐
            │ Output: submission.parquet & submission.csv   │
            └───────────────────────────────────────────────┘
"""

import gc
import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import balanced_accuracy_score



def verify_checkpoint_gate(
    Z_tr: np.ndarray,
    W_p: np.ndarray | None,
    b_p: np.ndarray | None,
    ckpt_labels: list,
    y_sub: np.ndarray,
    name: str,
    min_bacc: float = 0.50,
) -> bool:
    """
    Verifies that a checkpoint's plain classification head achieves >= min_bacc
    on the prompt-heldout validation partition (train_sub) before adding its logits.
    Guarantees that corrupt or misaligned heads cannot degrade Main or Comp stack.
    """
    if W_p is None or b_p is None or len(Z_tr) == 0:
        print(f"[Gate] {name}: No plain head or empty Z_tr; skipping head logits.")
        return False
    if W_p.shape[1] != Z_tr.shape[1] or W_p.shape[0] != K:
        print(f"[Gate] {name}: Shape mismatch W_p={W_p.shape} vs Z_tr={Z_tr.shape}; skipping head logits.")
        return False
    logits_tr = align_cols(Z_tr @ W_p.T + b_p, ckpt_labels)
    preds_tr = logits_tr.argmax(axis=1)
    acc = float((preds_tr == y_sub).mean())
    passed = acc >= min_bacc
    print(f"[Gate] {name}: train_sub accuracy = {acc:.4f} (threshold={min_bacc:.2f}) -> {'PASSED' if passed else 'REJECTED'}")
    return passed


def auto_align_and_verify_head(
    Z_tr: np.ndarray,
    W_p: np.ndarray | None,
    b_p: np.ndarray | None,
    W_c: np.ndarray | None,
    b_c: np.ndarray | None,
    ckpt_labels: list,
    y_sub: np.ndarray,
    prompts_sub: np.ndarray,
    name: str,
    min_bacc: float = 0.50,
) -> tuple[bool, np.ndarray | None, np.ndarray | None, np.ndarray | None, np.ndarray | None, list, float]:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import GroupKFold

    if len(Z_tr) == 0:
        return False, W_p, b_p, W_c, b_c, ckpt_labels, 0.0

    if W_p is not None and b_p is not None and W_p.shape == (K, Z_tr.shape[1]):
        best_order = ckpt_labels
        best_acc = -1.0
        for _, cand_order in [("canonical", ckpt_labels), ("alphabetical", sorted(LABELS))]:
            try:
                logits_tr = align_cols(Z_tr @ W_p.T + b_p, cand_order)
                acc = float(balanced_accuracy_score(y_sub, logits_tr.argmax(axis=1)))
                if acc > best_acc:
                    best_acc = acc
                    best_order = cand_order
            except Exception:
                continue
        if best_acc >= min_bacc:
            print(f"[Gate] {name}: Pre-trained head PASSED with BAcc = {best_acc:.4f} (threshold={min_bacc:.2f})")
            return True, W_p, b_p, W_c, b_c, best_order, best_acc
        print(f"[Gate] {name}: Pre-trained head BAcc = {best_acc:.4f} < {min_bacc:.2f}; probing 5-fold OOF LogReg readout...")

    if len(y_sub) >= 90 and len(np.unique(prompts_sub)) >= 5:
        scale = float(np.sqrt(Z_tr.shape[1])) / 8.0
        Z_scaled = Z_tr / scale
        gkf = GroupKFold(n_splits=5)
        oof_preds = np.zeros(len(y_sub), dtype=int)
        for tr_i, val_i in gkf.split(Z_scaled, y_sub, groups=prompts_sub):
            clf = LogisticRegression(C=0.20, max_iter=1000, random_state=42)
            clf.fit(Z_scaled[tr_i], y_sub[tr_i])
            oof_preds[val_i] = clf.predict(Z_scaled[val_i])
        oof_bacc = float(balanced_accuracy_score(y_sub, oof_preds))
        if oof_bacc >= min_bacc:
            clf_full = LogisticRegression(C=0.20, max_iter=1000, random_state=42)
            clf_full.fit(Z_scaled, y_sub)
            W_p_new = (clf_full.coef_ / scale).astype(np.float32)
            b_p_new = clf_full.intercept_.astype(np.float32)

            Z_c = Z_scaled.copy()
            for p in np.unique(prompts_sub):
                m = prompts_sub == p
                if m.sum() > 1:
                    Z_c[m] -= Z_c[m].mean(axis=0, keepdims=True)
            clf_c = LogisticRegression(C=0.12, max_iter=1000, random_state=42)
            clf_c.fit(Z_c, y_sub)
            W_c_new = (clf_c.coef_ / scale).astype(np.float32)
            b_c_new = clf_c.intercept_.astype(np.float32)
            fit_labels = [LABELS[c] for c in clf_full.classes_]
            print(f"[Gate] {name}: 5-Fold OOF Readout PASSED with BAcc = {oof_bacc:.4f}!")
            return True, W_p_new, b_p_new, W_c_new, b_c_new, fit_labels, oof_bacc

    print(f"[Gate] {name}: REJECTED (failed both pre-trained and 5-fold OOF gates).")
    return False, None, None, None, None, ckpt_labels, 0.0


def _combine_plain_and_twin_lp(
    lp_plain: np.ndarray,
    lp_twin: np.ndarray | None,
    w_plain: float,
    w_twin: float,
) -> np.ndarray:
    """
    Combines plain and prompt-centered twin log-probabilities without compressing
    logit temperature on singleton rows (where lp_twin == 0):
        log_softmax(lp_plain + (w_twin / w_plain) * lp_twin)
    """
    if lp_twin is None or w_twin <= 0.0 or len(lp_twin) != len(lp_plain):
        return log_softmax(lp_plain, axis=1)
    ratio = float(w_twin) / max(float(w_plain), 1e-6)
    return log_softmax(lp_plain + ratio * lp_twin, axis=1)


def extract_family_specialist_logits(
    trunk1: UniversalLoRATrunkLoader,
    spec_dir: Path,
    family_name: str,
    test_texts: list[str],
    unseen_texts: list[str],
    LOGIT_MAIN: np.ndarray,
    auditor: Any,
    groups_test_multi: tuple[np.ndarray, np.ndarray, np.ndarray],
    groups_unseen: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, list]:
    """
    Selectively runs inference for a Qwen3-14B family specialist adapter
    (GPT 5-class, Gemini 4-class, or GLM 2-class) on candidate test and unseen passages.

    Performance & Resource Optimization:
    ------------------------------------
    Rather than running the massive 14B model over ALL test samples, this function
    applies selective family masking:
    1. Only passages where the ensemble indicates membership in this family
       (either top-1 prediction is in family OR cumulative family probability >= 0.15)
       are selected for specialist inference.
    2. Dynamically hot-swaps LoRA adapter weights on the already loaded Qwen3-14B trunk
       without re-loading the base model from disk (saving ~2 minutes per specialist).
    3. Restores original trunk weights immediately upon completion.
    """
    fam_cols = FAMILIES[family_name]
    target_labels = [LABELS[i] for i in fam_cols]
    K_fam = len(target_labels)
    N_test = len(test_texts)
    N_unseen = len(unseen_texts)

    spec_logits_te = np.zeros((N_test, K_fam), dtype=np.float32)
    spec_logits_u = np.zeros((N_unseen, K_fam), dtype=np.float32)

    # 1. Selective Family Masking for Test set: filter to candidate rows
    if N_test <= 60:
        te_mask = np.ones(N_test, dtype=bool)
    else:
        P_cur = softmax(LOGIT_MAIN, axis=1)
        te_mask = np.isin(P_cur.argmax(axis=1), fam_cols) | (P_cur[:, fam_cols].sum(axis=1) >= 0.15)

    # 2. Selective Family Masking for Unseen holdout set
    if N_unseen <= 60:
        u_mask = np.ones(N_unseen, dtype=bool) if N_unseen > 0 else np.zeros(0, dtype=bool)
    else:
        u_mask = np.zeros(N_unseen, dtype=bool)
        if hasattr(auditor, "models") and auditor.models:
            for m in auditor.models:
                u_lp = m.get("unseen_lp")
                if u_lp is not None and len(u_lp) == N_unseen:
                    p_u = softmax(u_lp, axis=-1)
                    u_mask |= np.isin(p_u.argmax(axis=1), fam_cols) | (p_u[:, fam_cols].sum(axis=1) >= 0.15)
        if not u_mask.any() and N_unseen > 0:
            u_mask[:] = True

    sel_te_idx = np.flatnonzero(te_mask)
    sel_u_idx = np.flatnonzero(u_mask)
    n_sel_te = len(sel_te_idx)
    n_sel_u = len(sel_u_idx)

    print(f"[Specialist {family_name}] Masked inference: {n_sel_te}/{N_test} test rows, {n_sel_u}/{N_unseen} unseen rows.")
    if n_sel_te == 0 and n_sel_u == 0:
        return spec_logits_te, spec_logits_u, target_labels

    meta_spec = read_checkpoint_meta(spec_dir, default_keep=27, default_layers=[2, 4, 6, 9, 13, 17, 22, 27], default_max_len=447)
    # Step 1: Configure active layers and take a snapshot of base model weights
    trunk1.set_active_layers(meta_spec["layers"], meta_spec["max_len"])
    backup_spec = trunk1.snapshot_weights(spec_dir)

    # Step 2: Merge specialist LoRA weights directly into Qwen3-14B attention and MLP matrices
    merged = merge_lora(trunk1.model, spec_dir, sign=+1.0)
    heads_sd = load_head_file(spec_dir)
    trunk1.load_norm_weights(heads_sd)

    # Step 3: Resolve linear heads (plain head W_p, b_p and centered twin head W_c, b_c)
    W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd, num_classes=K_fam)
    src_labels = meta_spec.get("labels", target_labels)
    src_list = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    perm = None
    if len(src_list) == K_fam and set(src_list) == set(target_labels):
        perm = [src_list.index(l) for l in target_labels]

    # Step 4: Extract multi-layer representations for selected test & holdout passages
    sel_texts = [test_texts[i] for i in sel_te_idx] + [unseen_texts[j] for j in sel_u_idx]
    Z_sel = trunk1.extract_exact(sel_texts, batch_size=8, end_id=meta_spec["end_id"])

    Z_sel_te = Z_sel[:n_sel_te]
    Z_sel_u = Z_sel[n_sel_te:]

    g85, g72, g65 = groups_test_multi

    # Step 5: Compute temperature-scaled plain logits for candidate test passages
    # (Do NOT prompt-center across family-masked subsets, as full-cluster mean is unavailable on masked rows)
    if n_sel_te > 0 and W_p is not None:
        raw_p_te = Z_sel_te @ W_p.T + b_p
        if perm is not None:
            raw_p_te = raw_p_te[:, perm]
        std_te = float(np.std(raw_p_te, axis=1).mean()) if len(raw_p_te) > 0 else 1.0
        temp_te = max(1.0, std_te / 4.2)
        spec_logits_te[sel_te_idx] = log_softmax(raw_p_te / temp_te, axis=1).astype(np.float32)

    # Step 6: Compute plain logits for candidate unseen holdout passages & validate within-family BAcc
    if n_sel_u > 0 and W_p is not None:
        raw_p_u = Z_sel_u @ W_p.T + b_p
        if perm is not None:
            raw_p_u = raw_p_u[:, perm]
        std_u = float(np.std(raw_p_u, axis=1).mean()) if len(raw_p_u) > 0 else 1.0
        temp_u = max(1.0, std_u / 4.2)
        spec_logits_u[sel_u_idx] = log_softmax(raw_p_u / temp_u, axis=1).astype(np.float32)

        if hasattr(auditor, "y_unseen") and len(auditor.y_unseen) == N_unseen:
            fam_true_mask = np.isin(auditor.y_unseen, fam_cols) & u_mask
            if fam_true_mask.sum() >= len(fam_cols) * 2:
                y_fam_true = np.array([fam_cols.index(int(y)) for y in auditor.y_unseen[fam_true_mask]])
                fam_preds = spec_logits_u[fam_true_mask].argmax(axis=1)
                fam_bacc = float(balanced_accuracy_score(y_fam_true, fam_preds))
                print(f"[Specialist {family_name}] Unseen within-family BAcc = {fam_bacc:.4f}")
                if fam_bacc < (1.0 / len(fam_cols)) + 0.10:
                    print(f"[Specialist {family_name}] REJECTED (within-family BAcc {fam_bacc:.4f} too low); zeroing specialist logits.")
                    spec_logits_te.fill(0.0)
                    spec_logits_u.fill(0.0)

    # Step 7: Cleanly restore original base weights and free temporary tensors
    trunk1.restore_weights(backup_spec)
    trunk1.set_final_norm(False)
    del backup_spec, heads_sd, Z_sel
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return spec_logits_te, spec_logits_u, target_labels


def run_submission_pipeline(cfg: dict[str, Any] | None = None) -> tuple[pd.DataFrame, Path]:
    """
    Executes the complete production or smoke-test submission pipeline.
    Returns (submission_dataframe, submission_parquet_path).
    """
    if cfg is None:
        cfg = {}

    start_time = time.time()
    is_smoke = os.environ.get("NTL_SMOKE") == "1"

    # 1. Environment & Paths
    if "working_dir" in cfg:
        working_dir = Path(cfg["working_dir"])
    else:
        working_dir = Path("/kaggle/working")
        if not working_dir.exists():
            working_dir = Path("local_data")
    working_dir.mkdir(parents=True, exist_ok=True)

    input_dir = Path(cfg.get("input_dir", "/kaggle/input"))
    if not input_dir.exists():
        input_dir = Path(".")

    if "train_path" in cfg and "test_path" in cfg:
        train_path = Path(cfg["train_path"])
        test_path = Path(cfg["test_path"])
        pool_file = Path(cfg["pool_file"]) if "pool_file" in cfg else None
        all_pool_files = [pool_file] if pool_file else resolve_all_pool_files(input_dir)
        if pool_file is None and all_pool_files:
            pool_file = all_pool_files[0]
    else:
        train_path, test_path, pool_file = resolve_dataset_paths(input_dir)
        all_pool_files = resolve_all_pool_files(input_dir)

    enable_ablation_probe = bool(cfg.get("enable_ablation_probe", True)) if cfg else True
    probe_gate_min_bacc = float(cfg.get("probe_gate_min_bacc", 0.40)) if cfg else 0.40
    print(
        f"[Pipeline] Startup: is_smoke={is_smoke}, enable_ablation_probe={enable_ablation_probe}, probe_gate_min_bacc={probe_gate_min_bacc:.2f}"
    )
    print(f"[Pipeline] Paths: train={train_path}, test={test_path}, pool={pool_file}, all_pools={len(all_pool_files)}")

    df_test = pd.read_csv(test_path)
    test_ids = df_test["id"]
    test_texts = df_test["text"].fillna("").astype(str).tolist()
    N_test = len(test_texts)
    print(f"[Pipeline] Loaded test set: {N_test} rows")

    # Build Zero-Leakage Unseen Holdout Benchmark if enable_ablation_probe is True
    if enable_ablation_probe and not is_smoke and not (N_test > 100 and bool(cfg.get("lock_prior_weights", True))):
        ablation_spc = int(cfg.get("ablation_samples_per_class", 20))
        df_unseen, holdout_hashes = build_unseen_holdout_dataset(input_dir, train_path, samples_per_class=ablation_spc)
        unseen_parquet_path = working_dir / "unseen_holdout.parquet"
        df_unseen.to_parquet(unseen_parquet_path, index=False)
        unseen_texts = df_unseen["text"].fillna("").astype(str).tolist() if len(df_unseen) > 0 else []
        N_unseen = len(unseen_texts)
        y_unseen = np.array([LAB2I[m] for m in df_unseen["model"].values], dtype=int) if N_unseen > 0 else np.array([], dtype=int)
        groups_unseen = df_unseen["prompt_id"].values if N_unseen > 0 and "prompt_id" in df_unseen.columns else np.ones(N_unseen, dtype=int)
    else:
        if N_test > 100 and bool(cfg.get("lock_prior_weights", True)):
            print(
                "[Pipeline] Hidden Leaderboard Rerun (N_test > 100 with locked priors): Skipping unseen holdout extraction to maximize GPU throughput."
            )
        elif not enable_ablation_probe:
            print("[Pipeline] Frozen Production Mode (enable_ablation_probe=False): Skipping unseen holdout extraction to save GPU time.")
        df_unseen = pd.DataFrame()
        unseen_texts = []
        N_unseen = 0
        y_unseen = np.array([], dtype=int)
        groups_unseen = np.array([], dtype=int)

    auditor = UnseenStepwiseAuditor(y_unseen=y_unseen, groups_unseen=groups_unseen, N_test=N_test)

    # Discover fine-tuned checkpoints early for pre-flight validation
    ckpts = find_checkpoints(input_dir)
    print(f"[Pipeline] Discovered checkpoints ({len(ckpts)}): {list(ckpts.keys())}")

    req_assets = cfg.get("required_assets", {})
    cfg_model_weights = cfg.get("model_weights", {})
    DEFAULT_ALLOWED_CKPTS = {
        "qwen3_14b_p2",
        "qwen3_14b_inhouse",
        "deberta_p2",
        "deberta_inhouse",
        "qwen3_8b_claude",
        "qwen3_14b_gpt",
        "qwen3_14b_gemini",
        "qwen3_14b_glm",
    }
    if "required_checkpoints" in req_assets or "optional_checkpoints" in req_assets:
        allowed_ckpts = set(req_assets.get("required_checkpoints", [])) | set(req_assets.get("optional_checkpoints", []))
    elif cfg_model_weights:
        allowed_ckpts = set(cfg_model_weights.keys())
    else:
        allowed_ckpts = DEFAULT_ALLOWED_CKPTS

    for k in list(ckpts.keys()):
        if k not in allowed_ckpts:
            print(f"[Pipeline] Pruning non-whitelisted checkpoint '{k}' at {ckpts[k]}")
            ckpts.pop(k, None)
    print(f"[Pipeline] Active whitelisted checkpoints ({len(ckpts)}): {list(ckpts.keys())}")

    if not is_smoke:
        verified_bases = validate_production_assets(
            input_dir=input_dir,
            train_path=train_path,
            test_path=test_path,
            pool_files=all_pool_files or ([pool_file] if pool_file else []),
            ckpts=ckpts,
            cfg=cfg,
        )
    else:
        verified_bases = {}

    family_spec_logits_te: dict[str, np.ndarray] = {}
    family_spec_logits_unseen: dict[str, np.ndarray] = {}
    family_spec_classes: dict[str, list] = {}

    # 2. MODULE 1: Subprocess Dual-Output Lexical Engine
    run_lex_script = working_dir / "run_lexical.py"
    if run_lex_script.exists() and not is_smoke:
        print("[Pipeline] Module 1: Launching subprocess run_lexical.py...")
        subprocess.run([sys.executable, str(run_lex_script)], check=True)
        log_p_svc_main = np.load(working_dir / "log_p_svc_main.npy")
        p_lex_comp = np.load(working_dir / "p_lex_comp.npy")
        D_tfidf = np.load(working_dir / "D_tfidf.npy")
        groups_085 = np.load(working_dir / "groups_085.npy")
        log_p_handwriting = np.load(working_dir / "log_p_handwriting.npy") if (working_dir / "log_p_handwriting.npy").exists() else None
        rule_flags_test = np.load(working_dir / "rule_flags_test.npy") if (working_dir / "rule_flags_test.npy").exists() else None
        unseen_log_p_svc_main = (
            np.load(working_dir / "unseen_log_p_svc_main.npy")
            if (working_dir / "unseen_log_p_svc_main.npy").exists()
            else np.zeros((0, K), dtype=np.float32)
        )
        unseen_p_lex_comp = (
            np.load(working_dir / "unseen_p_lex_comp.npy")
            if (working_dir / "unseen_p_lex_comp.npy").exists()
            else np.zeros((0, K), dtype=np.float32)
        )
    else:
        print("[Pipeline] Module 1: Executing run_lexical_engine directly...")
        tok_paths = list(input_dir.rglob("vocab.json"))
        qwen_tok_dir = str(tok_paths[0].parent) if tok_paths else ""
        lex_out = run_lexical_engine(train_path, test_path, all_pool_files or pool_file, qwen_tok_dir, working_dir)
        log_p_svc_main = lex_out["log_p_svc_main"]
        p_lex_comp = lex_out["p_lex_comp"]
        D_tfidf = lex_out["D_tfidf"]
        groups_085 = lex_out["groups_085"]
        log_p_handwriting = lex_out.get("log_p_handwriting")
        rule_flags_test = lex_out.get("rule_flags_test")
        unseen_log_p_svc_main = lex_out.get("unseen_log_p_svc_main", np.zeros((0, K), dtype=np.float32))
        unseen_p_lex_comp = lex_out.get("unseen_p_lex_comp", np.zeros((0, K), dtype=np.float32))

    print(f"[Pipeline] Module 1 complete: log_p_svc_main shape = {log_p_svc_main.shape}")

    # Read configuration hierarchies from configs/production_ensemble.json
    main_cfg = cfg.get("main_stack", {})
    comp_cfg = cfg.get("complementary_stack", {})
    fusion_cfg = cfg.get("fusion", {})
    post_cfg = cfg.get("postprocess", {})
    model_weights = cfg.get("model_weights", {})

    svc_weight = model_weights.get("svc_4view", {}).get("plain_weight", main_cfg.get("linear_svc_weight", 1.00))
    main_members = main_cfg.get("members", {})
    comp_streams = comp_cfg.get("stream_weights", {})
    comp_members = comp_cfg.get("lora_members", {})
    lex_comp_weight = model_weights.get("lexical_5v", {}).get("plain_weight", comp_streams.get("lexical", 0.20))
    se_floors = comp_cfg.get(
        "se_floors",
        {"qwen3_14b_p2": 0.10, "qwen25_14b_p2": 0.10, "deberta_p2": 0.10, "qwen3_14b_p1": 0.35, "instyle": 0.35, "qwen3_8b_claude": 0.35},
    )
    prior_scales = comp_cfg.get("prior_scales", PRIOR_SCALES)
    dp_weights = comp_cfg.get("dp_weights", {"q3_14b": 0.38, "q25_14b": 0.24, "deberta": 0.20, "instyle": 0.10, "part1_14b": 0.08})
    twin_res = comp_cfg.get("twin_resolutions", {"t85": 0.50, "t72": 0.30, "t60": 0.20})
    w85 = float(twin_res.get("t85", 0.50))
    w72 = float(twin_res.get("t72", 0.30))
    w60 = float(twin_res.get("t60", 0.20))
    lock_priors = bool(cfg.get("lock_prior_weights", True))
    max_cl = post_cfg.get("max_cluster_size", 14)

    LOGIT_MAIN = log_p_svc_main * svc_weight
    COMP_LIST: list[tuple[float, np.ndarray]] = [(lex_comp_weight, p_lex_comp)]
    u_lp_svc = (
        unseen_log_p_svc_main if (N_unseen > 0 and len(unseen_log_p_svc_main) == N_unseen) else np.zeros((N_unseen, K), dtype=np.float32)
    )
    if svc_weight > 0.0 or not lock_priors:
        auditor.add_model(
            name="svc_4view",
            tier="Main",
            weight=svc_weight,
            unseen_log_probs=u_lp_svc,
            test_log_probs=log_p_svc_main,
        )
    u_p_lex = (
        unseen_p_lex_comp
        if (N_unseen > 0 and len(unseen_p_lex_comp) == N_unseen)
        else (np.ones((N_unseen, K), dtype=np.float32) / float(K))
    )
    if lex_comp_weight > 0.0 or not lock_priors:
        auditor.add_model(
            name="lexical_5v",
            tier="Comp",
            weight=lex_comp_weight,
            unseen_log_probs=np.log(np.clip(u_p_lex, 1e-9, 1.0)),
            test_log_probs=np.log(np.clip(p_lex_comp, 1e-9, 1.0)),
        )
    claude_spec_logits = None
    claude_classes = None

    D_neural_sum = np.zeros((N_test, N_test), dtype=np.float64)
    D_neural_weight_total = 0.0
    D_hybrid = D_tfidf.copy()

    if N_test >= 2:
        Z_link_init = linkage(squareform(D_tfidf, checks=False), method="average")
        groups_085 = split_oversized_clusters_dist(D_tfidf, fcluster(Z_link_init, t=0.85, criterion="distance"), max_cluster_size=max_cl)
        groups_072 = split_oversized_clusters_dist(D_tfidf, fcluster(Z_link_init, t=0.72, criterion="distance"), max_cluster_size=max_cl)
        groups_065 = split_oversized_clusters_dist(D_tfidf, fcluster(Z_link_init, t=0.65, criterion="distance"), max_cluster_size=max_cl)
        groups_060 = split_oversized_clusters_dist(D_tfidf, fcluster(Z_link_init, t=0.60, criterion="distance"), max_cluster_size=max_cl)
    else:
        groups_085 = np.ones(N_test, dtype=int)
        groups_072 = np.ones(N_test, dtype=int)
        groups_065 = np.ones(N_test, dtype=int)
        groups_060 = np.ones(N_test, dtype=int)

    def update_hybrid_groups(Z_te_mat: np.ndarray, ckpt_scale: float):
        nonlocal D_neural_sum, D_neural_weight_total, D_hybrid, groups_085, groups_072, groups_065, groups_060
        if N_test >= 2:
            Z_c = Z_te_mat - Z_te_mat.mean(axis=0, keepdims=True)
            Z_norm = Z_c / np.linalg.norm(Z_c, axis=1, keepdims=True).clip(min=1e-6)
            D_cos = np.clip(1.0 - (Z_norm @ Z_norm.T), 0.0, None)
            np.fill_diagonal(D_cos, 0.0)
            w_d = float(ckpt_scale)
            D_neural_sum += w_d * D_cos
            D_neural_weight_total += w_d
            D_neural_avg = D_neural_sum / max(1e-6, D_neural_weight_total)
            D_hybrid = 0.55 * D_tfidf + 0.45 * D_neural_avg
            Z_link = linkage(squareform(D_hybrid, checks=False), method="average")
            groups_085 = split_oversized_clusters_dist(D_hybrid, fcluster(Z_link, t=0.85, criterion="distance"), max_cluster_size=max_cl)
            groups_072 = split_oversized_clusters_dist(D_hybrid, fcluster(Z_link, t=0.72, criterion="distance"), max_cluster_size=max_cl)
            groups_065 = split_oversized_clusters_dist(D_hybrid, fcluster(Z_link, t=0.65, criterion="distance"), max_cluster_size=max_cl)
            groups_060 = split_oversized_clusters_dist(D_hybrid, fcluster(Z_link, t=0.60, criterion="distance"), max_cluster_size=max_cl)

    has_neural = len(ckpts) > 0 and not is_smoke
    spectral_blocks_tr: list[np.ndarray] = []
    spectral_blocks_te: list[np.ndarray] = []
    spectral_blocks_unseen: list[np.ndarray] = []
    dp_posteriors: list[tuple[float, np.ndarray]] = []
    dp_posteriors_unseen: list[tuple[float, np.ndarray]] = []
    comp_lora_posteriors: list[tuple[float, np.ndarray]] = []
    family_spec_logits_te: dict[str, np.ndarray] = {}
    family_spec_logits_unseen: dict[str, np.ndarray] = {}
    family_spec_classes: dict[str, list] = {}
    claude_spec_logits_unseen: np.ndarray | None = None
    P_spectral = None
    P_spectral_unseen = None

    w_dp_config = float(model_weights.get("wpca_dp", {}).get("plain_weight", comp_streams.get("dp_posteriors", 0.0)))
    w_sb_config = float(model_weights.get("spectral_bank", {}).get("plain_weight", comp_streams.get("spectral_bank", 0.0)))
    need_sub_train = w_dp_config > 0.0 or w_sb_config > 0.0

    df_train = pd.read_csv(train_path)
    train_sub_texts: list[str] = []
    y_sub: np.ndarray = (
        np.array([LAB2I[m] for m in df_train["model"].values], dtype=int) if "model" in df_train.columns else np.array([], dtype=int)
    )
    prompts_sub: np.ndarray = df_train["prompt_id"].values if "prompt_id" in df_train.columns else np.array([])
    N_tr_sub: int = 0

    if has_neural:
        if need_sub_train:
            rng = np.random.RandomState(42)
            u_p = df_train["prompt_id"].unique()
            max_wpca_prompts = int(cfg.get("wpca_max_prompts", len(u_p)))
            sel_prompts = u_p if len(u_p) <= max_wpca_prompts else rng.choice(u_p, size=max_wpca_prompts, replace=False)
            sub_idx = df_train[df_train["prompt_id"].isin(sel_prompts)].index.values
            train_sub = df_train.loc[sub_idx].reset_index(drop=True)
            train_sub_texts = train_sub["text"].fillna("").astype(str).tolist()
            y_sub = np.array([LAB2I[m] for m in train_sub["model"]])
            prompts_sub = train_sub["prompt_id"].values
            N_tr_sub = len(train_sub)
            print(f"[Pipeline] Selected {N_tr_sub} training rows ({len(sel_prompts)} prompts) for wPCA")
        else:
            print("[Pipeline] Zero-weight fast-path: w_dp and w_sb are 0.0; skipping training passage extraction.")

        # ==============================================================================
        # TRUNK 1: Qwen3-14B (40 layers, hidden=5120)
        # ==============================================================================
        base_q3_14b = verified_bases.get("qwen3_14b") or find_base_model(input_dir, "qwen3", 5120, 40)
        if base_q3_14b and any(
            k in ckpts for k in ("qwen3_14b_p2", "qwen3_14b_v5", "qwen3_14b_inhouse", "qwen3_14b_gpt", "qwen3_14b_gemini", "qwen3_14b_glm")
        ):
            print(f"[Pipeline] Loading Trunk 1: Qwen3-14B from {base_q3_14b}...")
            trunk1 = UniversalLoRATrunkLoader(base_q3_14b, keep_layers=27, layers=[2, 4, 6, 9, 13, 17, 22, 27], max_len=447)

            # Checkpoint 1A: Part 2 (lora_qwen3_14b)
            w_main_p = model_weights.get("qwen3_14b_p2", {}).get(
                "plain_weight", main_members.get("lora_qwen3_14b", {}).get("plain_weight", 0.42)
            )
            w_main_c = model_weights.get("qwen3_14b_p2", {}).get(
                "twin_weight", main_members.get("lora_qwen3_14b", {}).get("twin_weight", 0.12)
            )
            is_p2_req = "qwen3_14b_p2" in req_assets.get("required_checkpoints", [])
            if "qwen3_14b_p2" in ckpts and ((w_main_p + w_main_c) <= 0.0 and not is_p2_req):
                print("[Pipeline] qwen3_14b_p2 weight is 0.0 and not required; skipping extraction.")
            elif "qwen3_14b_p2" in ckpts:
                p2_dir = ckpts["qwen3_14b_p2"]
                meta_p2 = read_checkpoint_meta(p2_dir, default_keep=27, default_layers=[2, 4, 6, 9, 13, 17, 22, 27], default_max_len=447)
                trunk1.set_active_layers(meta_p2["layers"], meta_p2["max_len"])
                backup_t1 = trunk1.snapshot_weights(p2_dir)
                merged = merge_lora(trunk1.model, p2_dir, sign=+1.0)
                if not merged and not is_smoke:
                    raise RuntimeError(f"Failed to merge required LoRA adapter at {p2_dir}")
                heads_sd = load_head_file(p2_dir)
                trunk1.load_norm_weights(heads_sd)
                W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)
                if W_p is None and not is_smoke:
                    raise RuntimeError(f"Required classification head W_p is None for {p2_dir}")
                p2_labels = meta_p2["labels"]

                Z_all = trunk1.extract_exact(test_texts + unseen_texts + train_sub_texts, batch_size=8, end_id=meta_p2["end_id"])
                Z_te = Z_all[:N_test]
                Z_unseen = Z_all[N_test : N_test + N_unseen]
                Z_tr = Z_all[N_test + N_unseen :]
                update_hybrid_groups(Z_te, prior_scales.get("qwen3_14b_p2", 1.00))

                # Main Stack contributions
                w_main_p = model_weights.get("qwen3_14b_p2", {}).get(
                    "plain_weight", main_members.get("lora_qwen3_14b", {}).get("plain_weight", 0.42)
                )
                w_main_c = model_weights.get("qwen3_14b_p2", {}).get(
                    "twin_weight", main_members.get("lora_qwen3_14b", {}).get("twin_weight", 0.12)
                )
                lp_u_p = np.zeros((0, K), dtype=np.float32)
                lp_u_c = np.zeros((0, K), dtype=np.float32)
                lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                if W_p is not None:
                    lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, p2_labels))
                    LOGIT_MAIN += lp_p * w_main_p
                    w_comp_p = comp_members.get("lora_qwen3_14b", {}).get("plain_weight", 0.30)
                    comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                    if N_unseen > 0:
                        lp_u_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, p2_labels))

                if W_c is not None:
                    lp_c85, _ = compute_centered_logits(Z_te, groups_085, W_c, b_c, p2_labels)
                    lp_c72, _ = compute_centered_logits(Z_te, groups_072, W_c, b_c, p2_labels)
                    lp_c60, _ = compute_centered_logits(Z_te, groups_060, W_c, b_c, p2_labels)
                    lp_c_blend = w85 * lp_c85 + w72 * lp_c72 + w60 * lp_c60
                    LOGIT_MAIN += lp_c_blend * w_main_c
                    w_comp_c = comp_members.get("lora_qwen3_14b", {}).get("twin_weight", 0.10)
                    comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))
                    if N_unseen > 0:
                        lp_u_c, _ = compute_centered_logits(Z_unseen, groups_unseen, W_c, b_c, p2_labels)

                if W_p is not None:
                    w_tot_14b = w_main_p + (w_main_c if W_c is not None else 0.0)
                    if w_tot_14b > 0.0 or not lock_priors:
                        lp_model_u = (
                            _combine_plain_and_twin_lp(lp_u_p, lp_u_c if W_c is not None else None, w_main_p, w_main_c)
                            if N_unseen > 0
                            else np.zeros((0, K), dtype=np.float32)
                        )
                        lp_model_te = _combine_plain_and_twin_lp(lp_p, lp_c_blend if W_c is not None else None, w_main_p, w_main_c)
                        auditor.add_model(
                            "qwen3_14b_p2", tier="Main", weight=w_tot_14b, unseen_log_probs=lp_model_u, test_log_probs=lp_model_te
                        )

                # wPCA + DP Posterior + Spectral Bank
                if need_sub_train and len(Z_tr) > 0:
                    res_p2 = fit_shared_gaussian_lowram(
                        Z_tr, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get("qwen3_14b_p2", 0.10)
                    )
                    U_te = transform_lowram(Z_te, res_p2)
                    lp_dp = scores_exact_dp(U_te, groups_065, res_p2, D_hybrid)
                    dp_posteriors.append((dp_weights.get("q3_14b", 0.38), softmax(lp_dp, axis=1)))

                    S_tr, S_te = make_spectral_blocks(Z_tr, Z_te, res_p2, prior_scales.get("qwen3_14b_p2", 1.00))
                    spectral_blocks_tr.append(S_tr)
                    spectral_blocks_te.append(S_te)
                    if N_unseen > 0 and "Z_unseen" in locals() and Z_unseen is not None:
                        U_u = transform_lowram(Z_unseen, res_p2)
                        lp_dp_u = scores_exact_dp(U_u, groups_unseen, res_p2, None)
                        dp_posteriors_unseen.append((dp_weights.get("q3_14b", 0.38), softmax(lp_dp_u, axis=1)))
                        _, S_u = make_spectral_blocks(Z_tr, Z_unseen, res_p2, prior_scales.get("qwen3_14b_p2", 1.00))
                        spectral_blocks_unseen.append(S_u)
                        if "U_u" in locals():
                            del U_u
                    if "S_u" in locals():
                        del S_u
                if "res_p2" in locals():
                    del res_p2
                if "U_te" in locals():
                    del U_te

                trunk1.restore_weights(backup_t1)
                del backup_t1, Z_all, Z_tr, Z_te, heads_sd
                if "Z_unseen" in locals():
                    del Z_unseen
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # Checkpoint 1B / Optional Whitelisted Checkpoints on Trunk 1
            for opt_14b_key in ("qwen3_14b_inhouse",):
                if opt_14b_key in ckpts:
                    primary_ts14_dir = ckpts[opt_14b_key]
                    prefix_14b = "q14v5" if "v5" in opt_14b_key else "ts_q3b14" if "ts" in opt_14b_key else opt_14b_key
                    found_14b_folds = find_fold_adapters(input_dir, prefix_14b)

                    def _is_fold_14b(p: Path) -> bool:
                        nm = p.name.lower()
                        return any(f"_f{i}" in nm or f"-f{i}" in nm or f"_fold{i}" in nm for i in range(10))

                    folds_only_14b = sorted([p for p in found_14b_folds if _is_fold_14b(p)], key=lambda x: str(x).lower())
                    full_only_14b = [p for p in found_14b_folds if not _is_fold_14b(p)]
                    if not full_only_14b:
                        full_only_14b = [primary_ts14_dir]
                    fold_dirs_14b = folds_only_14b + full_only_14b
                    ts14_dir = fold_dirs_14b[-1]

                    def_keep_14b = 27 if opt_14b_key == "qwen3_14b_inhouse" else 24
                    def_layers_14b = [2, 4, 6, 9, 13, 17, 22, 27] if opt_14b_key == "qwen3_14b_inhouse" else [2, 4, 6, 8, 12, 16, 20, 24]
                    meta_ts14 = read_checkpoint_meta(
                        ts14_dir, default_keep=def_keep_14b, default_layers=def_layers_14b, default_max_len=447
                    )
                    if opt_14b_key == "qwen3_14b_inhouse" and meta_ts14.get("keep", 24) < 27:
                        print(f"[Pipeline] Skipping qwen3_14b_inhouse because keep={meta_ts14.get('keep', 24)} < 27 (old 24L checkpoint)")
                        continue

                    trunk1.set_active_layers(meta_ts14["layers"], meta_ts14["max_len"])
                    trunk1.set_final_norm(not meta_ts14["no_final_norm"])
                    backup_t1_ts = trunk1.snapshot_weights(ts14_dir)
                    merged = merge_lora(trunk1.model, ts14_dir, sign=+1.0)
                    if not merged:
                        print(f"[Pipeline] Optional checkpoint {opt_14b_key} failed to merge LoRA; skipping.")
                        trunk1.restore_weights(backup_t1_ts)
                        trunk1.set_final_norm(False)
                    else:
                        heads_sd = load_head_file(ts14_dir)
                        trunk1.load_norm_weights(heads_sd)
                        W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)
                        ts14_labels = meta_ts14["labels"]
                        p_comp_def = 0.12 if opt_14b_key == "qwen3_14b_p1" else 0.22
                        c_comp_def = 0.04 if opt_14b_key == "qwen3_14b_p1" else 0.08
                        comp_alias = "lora14b_v1" if opt_14b_key == "qwen3_14b_p1" else opt_14b_key
                        w_comp_p = model_weights.get(opt_14b_key, {}).get(
                            "plain_weight", comp_members.get(comp_alias, {}).get("plain_weight", p_comp_def)
                        )
                        w_comp_c = model_weights.get(opt_14b_key, {}).get(
                            "twin_weight", comp_members.get(comp_alias, {}).get("twin_weight", c_comp_def)
                        )
                        w_comp_tot = w_comp_p + w_comp_c

                        # Stage A: Fast 360-row probe on df_unseen
                        if N_unseen >= 36 and W_p is not None:
                            Z_unseen = trunk1.extract_exact(unseen_texts, batch_size=8, end_id=meta_ts14["end_id"])

                            # Auto-align label order: test canonical, alphabetical, and meta labels
                            cand_orders = [ts14_labels]
                            if sorted(LABELS) not in cand_orders:
                                cand_orders.append(sorted(LABELS))
                            if list(LABELS) not in cand_orders:
                                cand_orders.append(list(LABELS))

                            best_align_order = ts14_labels
                            best_solo_bacc = -1.0
                            best_lp_u = None
                            for cand_o in cand_orders:
                                try:
                                    trial_lp_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, cand_o))
                                    if W_c is not None:
                                        trial_lp_c, _ = compute_centered_logits(Z_unseen, groups_unseen, W_c, b_c, cand_o)
                                        trial_lp = _combine_plain_and_twin_lp(trial_lp_p, trial_lp_c, w_comp_p, w_comp_c)
                                    else:
                                        trial_lp = trial_lp_p
                                    trial_acc = float(balanced_accuracy_score(y_unseen, trial_lp.argmax(axis=1)))
                                    if trial_acc > best_solo_bacc:
                                        best_solo_bacc = trial_acc
                                        best_align_order = cand_o
                                        best_lp_u = trial_lp
                                except Exception:
                                    continue

                            solo_bacc = best_solo_bacc if best_solo_bacc >= 0 else 0.0
                            lp_u = best_lp_u if best_lp_u is not None else log_softmax(align_cols(Z_unseen @ W_p.T + b_p, ts14_labels))
                            ts14_labels = best_align_order
                            align_name = (
                                "canonical"
                                if best_align_order == list(LABELS)
                                else "alphabetical"
                                if best_align_order == sorted(LABELS)
                                else "custom"
                            )
                            print(f"[Fast Probe] {opt_14b_key}: solo unseen BAcc = {solo_bacc:.4f} (alignment: {align_name})")

                            is_required = opt_14b_key in req_assets.get("required_checkpoints", [])
                            lock_priors = bool(cfg.get("lock_prior_weights", True))
                            if solo_bacc < probe_gate_min_bacc and not is_required:
                                if opt_14b_key in ("qwen3_14b_ts", "qwen3_14b_p1"):
                                    print(
                                        f"[Fast Probe] {opt_14b_key}: solo BAcc {solo_bacc:.4f} < {probe_gate_min_bacc:.2f} -> deferring to auto_align_and_verify_head on Z_tr..."
                                    )
                                    W_p = None
                                    W_c = None
                                else:
                                    print(
                                        f"[Fast Probe Pruning] {opt_14b_key}: solo BAcc {solo_bacc:.4f} < {probe_gate_min_bacc:.2f} -> SKIPPING 5,620-row test/train extraction!"
                                    )
                                    trunk1.restore_weights(backup_t1_ts)
                                    trunk1.set_final_norm(False)
                                    del backup_t1_ts, heads_sd, Z_unseen
                                    gc.collect()
                                    if torch.cuda.is_available():
                                        torch.cuda.empty_cache()
                                    continue
                            elif len(auditor.models) > 0 and not is_required and not lock_priors:
                                lift = auditor.evaluate_marginal_lift(candidate_unseen_lp=lp_u, weight=w_comp_tot)
                                if lift < -0.002:
                                    print(
                                        f"[Fast Probe Pruning] {opt_14b_key}: marginal lift {lift:+.4f} < -0.002 -> SKIPPING 5,620-row test/train extraction!"
                                    )
                                    trunk1.restore_weights(backup_t1_ts)
                                    trunk1.set_final_norm(False)
                                    del backup_t1_ts, heads_sd, Z_unseen
                                    gc.collect()
                                    if torch.cuda.is_available():
                                        torch.cuda.empty_cache()
                                    continue
                                print(
                                    f"[Fast Probe] {opt_14b_key}: passed probe (solo={solo_bacc:.4f}, lift={lift:+.4f}) -> extracting test/train..."
                                )
                            else:
                                print(f"[Fast Probe] {opt_14b_key}: passed probe (solo={solo_bacc:.4f}) -> extracting test/train...")

                        # Stage B: Extract test and train (with multi-fold averaging if multiple folds discovered)
                        num_14b_folds = len(fold_dirs_14b)
                        if num_14b_folds > 1:
                            trunk1.restore_weights(backup_t1_ts)
                            del backup_t1_ts

                            lp_p_acc = np.zeros((N_test, K), dtype=np.float64)
                            lp_c_acc = np.zeros((N_test, K), dtype=np.float64)
                            lp_u_p_acc = np.zeros((N_unseen, K), dtype=np.float64) if N_unseen > 0 else None
                            lp_u_c_acc = np.zeros((N_unseen, K), dtype=np.float64) if N_unseen > 0 else None
                            has_any_c = False
                            Z_te = None
                            Z_tr = None
                            print(f"[Pipeline] {opt_14b_key}: extracting & averaging across {num_14b_folds} fold adapters in log-space...")

                            for f_idx, f_dir in enumerate(fold_dirs_14b):
                                is_last = f_idx == num_14b_folds - 1
                                f_meta = read_checkpoint_meta(
                                    f_dir, default_keep=def_keep_14b, default_layers=def_layers_14b, default_max_len=447
                                )
                                trunk1.set_active_layers(f_meta["layers"], f_meta["max_len"])
                                trunk1.set_final_norm(not f_meta["no_final_norm"])
                                f_backup = trunk1.snapshot_weights(f_dir)
                                f_merged = merge_lora(trunk1.model, f_dir, sign=+1.0)
                                if not f_merged:
                                    trunk1.restore_weights(f_backup)
                                    continue
                                f_heads_sd = load_head_file(f_dir)
                                trunk1.load_norm_weights(f_heads_sd)
                                f_W_p, f_b_p, f_W_c, f_b_c = resolve_head_tensors(f_heads_sd)
                                f_labels = f_meta["labels"]

                                if is_last:
                                    if "Z_unseen" in locals() and Z_unseen is not None and len(Z_unseen) == N_unseen and N_unseen > 0:
                                        extract_batch = test_texts + train_sub_texts
                                        Z_te_tr = trunk1.extract_exact(extract_batch, batch_size=8, end_id=f_meta["end_id"])
                                        f_Z_te = Z_te_tr[:N_test]
                                        f_Z_u = Z_unseen
                                        Z_tr = Z_te_tr[N_test:]
                                        Z_te = f_Z_te
                                        del Z_te_tr
                                    else:
                                        extract_batch = test_texts + (unseen_texts if N_unseen > 0 else []) + train_sub_texts
                                        Z_te_u_tr = trunk1.extract_exact(extract_batch, batch_size=8, end_id=f_meta["end_id"])
                                        f_Z_te = Z_te_u_tr[:N_test]
                                        f_Z_u = Z_te_u_tr[N_test : N_test + N_unseen] if N_unseen > 0 else None
                                        Z_tr = Z_te_u_tr[N_test + N_unseen :]
                                        Z_te = f_Z_te
                                        del Z_te_u_tr
                                else:
                                    extract_batch = test_texts + (unseen_texts if N_unseen > 0 else [])
                                    Z_te_u = trunk1.extract_exact(extract_batch, batch_size=8, end_id=f_meta["end_id"])
                                    f_Z_te = Z_te_u[:N_test]
                                    f_Z_u = Z_te_u[N_test : N_test + N_unseen] if N_unseen > 0 else None
                                    del Z_te_u

                                if f_W_p is not None:
                                    raw_p_f = align_cols(f_Z_te @ f_W_p.T + f_b_p, f_labels)
                                    std_f = float(np.std(raw_p_f, axis=1).mean())
                                    temp_f = max(1.0, std_f / 4.2)
                                    lp_p_acc += log_softmax(raw_p_f / temp_f, axis=1) / num_14b_folds
                                    if N_unseen > 0 and f_Z_u is not None:
                                        raw_u_f = align_cols(f_Z_u @ f_W_p.T + f_b_p, f_labels)
                                        lp_u_p_acc += log_softmax(raw_u_f / temp_f, axis=1) / num_14b_folds
                                if f_W_c is not None:
                                    has_any_c = True
                                    lp_c85_f, _ = compute_centered_logits(f_Z_te, groups_085, f_W_c, f_b_c, f_labels)
                                    lp_c72_f, _ = compute_centered_logits(f_Z_te, groups_072, f_W_c, f_b_c, f_labels)
                                    lp_c60_f, _ = compute_centered_logits(f_Z_te, groups_060, f_W_c, f_b_c, f_labels)
                                    c_blend_f = w85 * lp_c85_f + w72 * lp_c72_f + w60 * lp_c60_f
                                    lp_c_acc += c_blend_f / num_14b_folds
                                    if N_unseen > 0 and f_Z_u is not None:
                                        lp_u_c_f, _ = compute_centered_logits(f_Z_u, groups_unseen, f_W_c, f_b_c, f_labels)
                                        lp_u_c_acc += lp_u_c_f / num_14b_folds

                                trunk1.restore_weights(f_backup)
                                del f_backup, f_heads_sd, f_Z_te
                                if "f_Z_u" in locals() and f_Z_u is not None:
                                    del f_Z_u
                                gc.collect()
                                if torch.cuda.is_available():
                                    torch.cuda.empty_cache()

                            trunk1.set_final_norm(False)
                            lp_p = lp_p_acc
                            lp_c_blend = lp_c_acc if has_any_c else np.zeros((N_test, K), dtype=np.float32)
                            if N_unseen > 0 and lp_u_p_acc is not None:
                                lp_u = _combine_plain_and_twin_lp(lp_u_p_acc, lp_u_c_acc if has_any_c else None, w_comp_p, w_comp_c)
                            gate_passed = True
                            backup_t1_ts = None
                        else:
                            if N_unseen >= 36 and W_p is not None:
                                Z_te_tr = trunk1.extract_exact(test_texts + train_sub_texts, batch_size=8, end_id=meta_ts14["end_id"])
                                Z_te = Z_te_tr[:N_test]
                                Z_tr = Z_te_tr[N_test:]
                                del Z_te_tr
                            else:
                                Z_all = trunk1.extract_exact(
                                    test_texts + unseen_texts + train_sub_texts, batch_size=8, end_id=meta_ts14["end_id"]
                                )
                                Z_te = Z_all[:N_test]
                                Z_unseen = Z_all[N_test : N_test + N_unseen]
                                Z_tr = Z_all[N_test + N_unseen :]
                                del Z_all

                            gate_passed = True
                            if not is_smoke and len(y_sub) >= 180:
                                gate_passed, W_p, b_p, W_c, b_c, ts14_labels, _ = auto_align_and_verify_head(
                                    Z_tr=Z_tr,
                                    W_p=W_p,
                                    b_p=b_p,
                                    W_c=W_c,
                                    b_c=b_c,
                                    ckpt_labels=ts14_labels,
                                    y_sub=y_sub,
                                    prompts_sub=prompts_sub,
                                    name=opt_14b_key,
                                    min_bacc=probe_gate_min_bacc,
                                )

                            if gate_passed and N_unseen > 0 and W_p is not None and "Z_unseen" in locals():
                                raw_u_p = align_cols(Z_unseen @ W_p.T + b_p, ts14_labels)
                                std_u = float(np.std(raw_u_p, axis=1).mean())
                                temp_u = max(1.0, std_u / 4.2)
                                lp_u_p_fit = log_softmax(raw_u_p / temp_u, axis=1)
                                if W_c is not None:
                                    lp_u_c_fit, _ = compute_centered_logits(Z_unseen, groups_unseen, W_c, b_c, ts14_labels)
                                    lp_u_fitted = _combine_plain_and_twin_lp(lp_u_p_fit, lp_u_c_fit, w_comp_p, w_comp_c)
                                else:
                                    lp_u_fitted = lp_u_p_fit
                                lp_u = lp_u_fitted
                                fitted_bacc = float(balanced_accuracy_score(y_unseen, lp_u_fitted.argmax(axis=1)))
                                print(f"[Post-Fit Probe] {opt_14b_key}: unseen BAcc = {fitted_bacc:.4f}")
                                if fitted_bacc < probe_gate_min_bacc:
                                    print(
                                        f"[Post-Fit Pruning] {opt_14b_key}: fitted unseen BAcc {fitted_bacc:.4f} < {probe_gate_min_bacc:.2f} -> rejecting checkpoint!"
                                    )
                                    gate_passed = False
                                elif len(auditor.models) > 0:
                                    fitted_lift = auditor.evaluate_marginal_lift(candidate_unseen_lp=lp_u_fitted, weight=w_comp_tot)
                                    if fitted_lift < -0.002:
                                        print(
                                            f"[Post-Fit Pruning] {opt_14b_key}: fitted marginal lift {fitted_lift:+.4f} < -0.002 -> rejecting checkpoint!"
                                        )
                                        gate_passed = False

                            if gate_passed:
                                if W_p is not None:
                                    raw_te = align_cols(Z_te @ W_p.T + b_p, ts14_labels)
                                    std_te = float(np.std(raw_te, axis=1).mean())
                                    temp_te = max(1.0, std_te / 4.2)
                                    lp_p = log_softmax(raw_te / temp_te, axis=1)
                                else:
                                    lp_p = None
                                if W_c is not None:
                                    lp_c85, _ = compute_centered_logits(Z_te, groups_085, W_c, b_c, ts14_labels)
                                    lp_c72, _ = compute_centered_logits(Z_te, groups_072, W_c, b_c, ts14_labels)
                                    lp_c60, _ = compute_centered_logits(Z_te, groups_060, W_c, b_c, ts14_labels)
                                    lp_c_blend = w85 * lp_c85 + w72 * lp_c72 + w60 * lp_c60
                                else:
                                    lp_c_blend = np.zeros((N_test, K), dtype=np.float32)

                        if gate_passed:
                            pr_sc = prior_scales.get(opt_14b_key, 0.50)
                            update_hybrid_groups(Z_te, pr_sc)
                            if num_14b_folds > 1:
                                comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                                if has_any_c:
                                    comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))
                            else:
                                if W_p is not None and lp_p is not None:
                                    comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                                if W_c is not None:
                                    comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))

                            if need_sub_train and len(Z_tr) > 0:
                                def_dp_14b = 0.08 if opt_14b_key == "qwen3_14b_p1" else 0.10
                                def_se_14b = 0.35 if opt_14b_key == "qwen3_14b_p1" else 0.25
                                res_ts14 = fit_shared_gaussian_lowram(
                                    Z_tr, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get(opt_14b_key, def_se_14b)
                                )
                                U_te = transform_lowram(Z_te, res_ts14)
                                lp_dp = scores_exact_dp(U_te, groups_065, res_ts14, D_hybrid)
                                dp_posteriors.append(
                                    (
                                        dp_weights.get(
                                            opt_14b_key,
                                            dp_weights.get("part1_14b" if opt_14b_key == "qwen3_14b_p1" else "q3_14b_ts", def_dp_14b),
                                        ),
                                        softmax(lp_dp, axis=1),
                                    )
                                )

                                S_tr, S_te = make_spectral_blocks(Z_tr, Z_te, res_ts14, pr_sc)
                                spectral_blocks_tr.append(S_tr)
                                spectral_blocks_te.append(S_te)
                                if N_unseen > 0 and "Z_unseen" in locals() and Z_unseen is not None:
                                    U_u = transform_lowram(Z_unseen, res_ts14)
                                    lp_dp_u = scores_exact_dp(U_u, groups_unseen, res_ts14, None)
                                    dp_w_ts14 = dp_weights.get(
                                        opt_14b_key,
                                        dp_weights.get("part1_14b" if opt_14b_key == "qwen3_14b_p1" else "q3_14b_ts", def_dp_14b),
                                    )
                                    dp_posteriors_unseen.append((dp_w_ts14, softmax(lp_dp_u, axis=1)))
                                    _, S_u = make_spectral_blocks(Z_tr, Z_unseen, res_ts14, pr_sc)
                                    spectral_blocks_unseen.append(S_u)
                                    del U_u, S_u
                                del res_ts14, U_te

                            if lp_p is not None:
                                w_tot_ts14 = w_comp_p + (w_comp_c if (has_any_c if num_14b_folds > 1 else W_c is not None) else 0.0)
                                if w_tot_ts14 > 0.0 or not lock_priors:
                                    lp_m_u = lp_u if N_unseen > 0 and "lp_u" in locals() else np.zeros((0, K), dtype=np.float32)
                                    lp_m_te = _combine_plain_and_twin_lp(
                                        lp_p,
                                        lp_c_blend if (has_any_c if num_14b_folds > 1 else W_c is not None) else None,
                                        w_comp_p,
                                        w_comp_c,
                                    )
                                    aud_name_14b = opt_14b_key
                                    auditor.add_model(
                                        name=aud_name_14b, tier="Comp", weight=w_tot_ts14, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te
                                    )
                        else:
                            print(f"[Pipeline] Optional checkpoint {opt_14b_key} rejected by gate; excluded from logits and spectral bank.")

                        if backup_t1_ts is not None:
                            trunk1.restore_weights(backup_t1_ts)
                            del backup_t1_ts
                        trunk1.set_final_norm(False)
                        del Z_tr, Z_te, heads_sd
                        if "Z_unseen" in locals():
                            del Z_unseen
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

            # Execute Family Specialists on Trunk 1 (Qwen3-14B)
            spec_candidates = [
                ("qwen3_14b_gpt", "gpt"),
                ("qwen3_14b_gemini", "gemini"),
                ("qwen3_14b_glm", "glm"),
            ]
            for ckpt_key, fam_name in spec_candidates:
                if ckpt_key in ckpts:
                    spec_p = ckpts[ckpt_key]
                    print(f"[Pipeline] Running Trunk 1 Family Specialist: {ckpt_key} ({fam_name}) from {spec_p}...")
                    te_l, u_l, cls_l = extract_family_specialist_logits(
                        trunk1=trunk1,
                        spec_dir=spec_p,
                        family_name=fam_name,
                        test_texts=test_texts,
                        unseen_texts=unseen_texts if N_unseen > 0 else [],
                        LOGIT_MAIN=LOGIT_MAIN,
                        auditor=auditor,
                        groups_test_multi=(groups_085, groups_072, groups_065),
                        groups_unseen=groups_unseen,
                    )
                    family_spec_logits_te[fam_name] = te_l
                    family_spec_logits_unseen[fam_name] = u_l
                    family_spec_classes[fam_name] = cls_l

            trunk1.free_memory()
            del trunk1
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ==============================================================================
        # TRUNK 3: Qwen3-8B (36 layers, hidden=4096)
        # ==============================================================================
        base_q3_8b = verified_bases.get("qwen3_8b") or find_base_model(input_dir, "qwen3", 4096, 36)
        if base_q3_8b and "qwen3_8b_claude" in ckpts:
            print(f"[Pipeline] Loading Trunk 3: Qwen3-8B from {base_q3_8b}...")
            trunk3 = UniversalLoRATrunkLoader(base_q3_8b, keep_layers=24, layers=[2, 4, 6, 8, 12, 16, 20, 24], max_len=447)

            # Claude Head LoRA
            if "qwen3_8b_claude" in ckpts:
                claude_dir = ckpts["qwen3_8b_claude"]
                meta_cl = read_checkpoint_meta(
                    claude_dir, default_keep=24, default_layers=[2, 4, 6, 8, 12, 16, 20, 24], default_max_len=447
                )
                trunk3.set_active_layers(meta_cl["layers"], meta_cl["max_len"])
                backup_t3 = trunk3.snapshot_weights(claude_dir)
                merged = merge_lora(trunk3.model, claude_dir, sign=+1.0)
                if not merged and not is_smoke:
                    raise RuntimeError(f"Failed to merge required LoRA adapter at {claude_dir}")

                try:
                    heads_sd_cl = load_head_file(claude_dir)
                    if heads_sd_cl:
                        trunk3.load_norm_weights(heads_sd_cl)

                    W_p, b_p, W_c, b_c = None, None, None, None
                    W_spec, b_spec = None, None
                    ro_labels = None
                    readouts_file = claude_dir / "readouts.npz"
                    if readouts_file.exists():
                        with np.load(str(readouts_file), allow_pickle=True) as rnpz:
                            if "classes" in rnpz:
                                ro_labels = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in rnpz["classes"]]
                            elif "labels" in rnpz:
                                ro_labels = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in rnpz["labels"]]
                            for wk, bk in [("plain_coef", "plain_intercept"), ("plain.weight", "plain.bias")]:
                                if wk in rnpz and bk in rnpz and rnpz[wk].shape[0] == 18:
                                    W_p, b_p = np.array(rnpz[wk]), np.array(rnpz[bk])
                                    break
                            for wk, bk in [("center_coef", "center_intercept"), ("center.weight", "center.bias")]:
                                if wk in rnpz and bk in rnpz and rnpz[wk].shape[0] == 18:
                                    W_c, b_c = np.array(rnpz[wk]), np.array(rnpz[bk])
                                    break
                            if "claude_coef" in rnpz and "claude_intercept" in rnpz:
                                if rnpz["claude_coef"].shape[0] == 5:
                                    W_spec, b_spec = np.array(rnpz["claude_coef"]), np.array(rnpz["claude_intercept"])
                                    if "claude_classes" in rnpz:
                                        claude_classes = list(rnpz["claude_classes"])
                    if W_p is None and heads_sd_cl:
                        cand_W, cand_b, cand_Wc, cand_bc = resolve_head_tensors(heads_sd_cl)
                        if cand_W is not None and cand_W.shape[0] == 18:
                            W_p, b_p = cand_W, cand_b
                            if cand_Wc is not None and cand_Wc.shape[0] == 18:
                                W_c, b_c = cand_Wc, cand_bc

                    # Ensure ro_labels is a valid 18-class label list (defaults to sorted(LABELS) alphabetical order)
                    if ro_labels is None or len(ro_labels) != 18 or set(ro_labels) != set(LABELS):
                        cand_meta = meta_cl.get("labels", [])
                        if len(cand_meta) == 18 and set(cand_meta) == set(LABELS):
                            ro_labels = list(cand_meta)
                        else:
                            ro_labels = sorted(LABELS)

                    if W_spec is None:
                        spec_file = claude_dir / "claude_head.safetensors"
                        if spec_file.exists():
                            ch_sd = load_head_file(spec_file)
                            for wk, bk in [("weight", "bias"), ("plain.weight", "plain.bias")]:
                                if wk in ch_sd and ch_sd[wk].shape[0] == 5:
                                    W_spec = ch_sd[wk].cpu().numpy()
                                    b_spec = ch_sd[bk].cpu().numpy() if bk in ch_sd else np.zeros(5, dtype=np.float32)
                                    break
                    if W_spec is None and not is_smoke:
                        raise RuntimeError(f"Required specialist head W_spec is None for {claude_dir}")
                    if claude_classes is None:
                        claude_classes = [LABELS[i] for i in FAMILIES["claude"]]

                    Z_all = trunk3.extract_exact(test_texts + unseen_texts + train_sub_texts, batch_size=8, end_id=meta_cl["end_id"])
                    Z_te = Z_all[:N_test]
                    Z_unseen = Z_all[N_test : N_test + N_unseen]
                    Z_tr = Z_all[N_test + N_unseen :]
                    update_hybrid_groups(Z_te, prior_scales.get("qwen3_8b_claude", 0.30))

                    if W_spec is not None and W_spec.shape[0] == 5:
                        claude_spec_logits = Z_te @ W_spec.T + b_spec
                        if N_unseen > 0 and "Z_unseen" in locals() and Z_unseen is not None:
                            claude_spec_logits_unseen = Z_unseen @ W_spec.T + b_spec

                    if W_p is not None and W_p.shape[0] == 18:
                        w_comp_p = model_weights.get("qwen3_8b_claude", {}).get(
                            "plain_weight", comp_members.get("ntl_claude_head", {}).get("plain_weight", 0.0)
                        )
                        w_comp_c = model_weights.get("qwen3_8b_claude", {}).get(
                            "twin_weight", comp_members.get("ntl_claude_head", {}).get("twin_weight", 0.0)
                        )
                        lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, ro_labels))
                        if w_comp_p > 0.0:
                            comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                        if N_unseen > 0:
                            lp_u_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, ro_labels))
                        else:
                            lp_u_p = np.zeros((0, K), dtype=np.float32)

                        if W_c is not None and W_c.shape[0] == 18:
                            lp_c85, _ = compute_centered_logits(Z_te, groups_085, W_c, b_c, ro_labels)
                            lp_c72, _ = compute_centered_logits(Z_te, groups_072, W_c, b_c, ro_labels)
                            lp_c60, _ = compute_centered_logits(Z_te, groups_060, W_c, b_c, ro_labels)
                            lp_c_blend = w85 * lp_c85 + w72 * lp_c72 + w60 * lp_c60
                            if w_comp_c > 0.0:
                                comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))
                            if N_unseen > 0:
                                lp_u_c, _ = compute_centered_logits(Z_unseen, groups_unseen, W_c, b_c, ro_labels)
                            else:
                                lp_u_c = np.zeros((0, K), dtype=np.float32)
                        else:
                            lp_c85 = np.zeros((N_test, K), dtype=np.float32)
                            lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                            lp_u_c = np.zeros((0, K), dtype=np.float32)

                        w_tot_cl = w_comp_p + (w_comp_c if (W_c is not None and W_c.shape[0] == 18) else 0.0)
                        if w_tot_cl > 0.0:
                            twin_c_u = lp_u_c if (W_c is not None and W_c.shape[0] == 18) else None
                            twin_c_te = lp_c_blend if (W_c is not None and W_c.shape[0] == 18) else None
                            lp_m_u = (
                                _combine_plain_and_twin_lp(lp_u_p, twin_c_u, w_comp_p, w_comp_c)
                                if N_unseen > 0
                                else np.zeros((0, K), dtype=np.float32)
                            )
                            lp_m_te = _combine_plain_and_twin_lp(lp_p, twin_c_te, w_comp_p, w_comp_c)
                            auditor.add_model(
                                "qwen3_8b_claude", tier="Comp", weight=w_tot_cl, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te
                            )

                    if need_sub_train and len(Z_tr) > 0:
                        res_cl = fit_shared_gaussian_lowram(
                            Z_tr, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get("qwen3_8b_claude", 0.35)
                        )
                        U_te = transform_lowram(Z_te, res_cl)
                        lp_dp = scores_exact_dp(U_te, groups_065, res_cl, D_hybrid)
                        dp_posteriors.append((dp_weights.get("claude_8b", 0.04), softmax(lp_dp, axis=1)))

                        S_tr, S_te = make_spectral_blocks(Z_tr, Z_te, res_cl, prior_scales.get("qwen3_8b_claude", 0.30))
                        spectral_blocks_tr.append(S_tr)
                        spectral_blocks_te.append(S_te)
                        if N_unseen > 0 and "Z_unseen" in locals() and Z_unseen is not None:
                            U_u = transform_lowram(Z_unseen, res_cl)
                            lp_dp_u = scores_exact_dp(U_u, groups_unseen, res_cl, None)
                            dp_posteriors_unseen.append((dp_weights.get("claude_8b", 0.04), softmax(lp_dp_u, axis=1)))
                            _, S_u = make_spectral_blocks(Z_tr, Z_unseen, res_cl, prior_scales.get("qwen3_8b_claude", 0.30))
                            spectral_blocks_unseen.append(S_u)
                            del U_u, S_u
                        del res_cl, U_te
                    del Z_all, Z_tr, Z_te
                    if "Z_unseen" in locals():
                        del Z_unseen
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                finally:
                    trunk3.restore_weights(backup_t3)

            trunk3.free_memory()
            del trunk3
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ==============================================================================
        # TRUNK 4: DeBERTa-v3-Large
        # ==============================================================================
        for deb_key in ("deberta_p2",):
            if deb_key not in ckpts:
                continue
            deb_dir = ckpts[deb_key]
            w_main_deb = model_weights.get(deb_key, {}).get(
                "plain_weight", main_members.get("deberta_v3_large", {}).get("plain_weight", 0.25)
            )
            w_comp_deb = comp_members.get("deberta_v3_large", {}).get("plain_weight", 0.18)
            w_comp_c = model_weights.get(deb_key, {}).get("twin_weight", comp_members.get("deberta_v3_large", {}).get("twin_weight", 0.06))
            if (w_main_deb + w_comp_deb + w_comp_c) <= 0.0 and deb_key not in req_assets.get("required_checkpoints", []):
                print(f"[Pipeline] {deb_key} has zero weight and not required; skipping extraction.")
                continue

            base_deb_dir = verified_bases.get("deberta_v3_large") or find_base_model(input_dir, "deberta", 1024, 24)
            print(f"[Pipeline] Loading Trunk 4: DeBERTa-v3 ({deb_key}) from {deb_dir}...")
            Z_test_all, Z_tr_deb, W_p_deb, b_p_deb, W_c_deb, b_c_deb, deb_labels = extract_deberta_checkpoint(
                deb_dir, base_deb_dir, test_texts + unseen_texts, train_sub_texts if need_sub_train else [], y_sub
            )
            Z_te_deb = Z_test_all[:N_test]
            Z_unseen_deb = Z_test_all[N_test : N_test + N_unseen]

            if W_p_deb is None and not is_smoke:
                raise RuntimeError(f"Required classification head W_p is None for {deb_dir}")
            update_hybrid_groups(Z_te_deb, prior_scales.get(deb_key, 0.80))

            if W_c_deb is None and len(Z_tr_deb) >= 90 and len(np.unique(prompts_sub)) >= 5:
                from sklearn.linear_model import LogisticRegression

                Z_c_deb = Z_tr_deb.astype(np.float64).copy()
                for p in np.unique(prompts_sub):
                    m_p = prompts_sub == p
                    if m_p.sum() > 1:
                        Z_c_deb[m_p] -= Z_c_deb[m_p].mean(axis=0, keepdims=True)
                sd_c = Z_c_deb.std(axis=0) + 1e-6
                clf_deb_c = LogisticRegression(C=0.05, max_iter=1000, random_state=42)
                clf_deb_c.fit(Z_c_deb / sd_c, y_sub)
                if len(clf_deb_c.classes_) == K:
                    W_c_canon = (clf_deb_c.coef_ / sd_c[None, :]).astype(np.float32)
                    b_c_canon = clf_deb_c.intercept_.astype(np.float32)
                    inv_perm = [LABELS.index(l) for l in deb_labels]
                    W_c_deb = W_c_canon[inv_perm]
                    b_c_deb = b_c_canon[inv_perm]
                    print(f"[DeBERTa] Fitted prompt-centered twin readout on {len(Z_tr_deb)} rows (twin_weight={w_comp_c:.2f})")
            lp_p_deb = None
            lp_u_deb = np.zeros((0, K), dtype=np.float32)
            lp_c85_deb = np.zeros((N_test, K), dtype=np.float32)
            lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
            lp_u_c_deb = np.zeros((0, K), dtype=np.float32)

            if W_p_deb is not None:
                lp_p_deb = log_softmax(align_cols(Z_te_deb @ W_p_deb.T + b_p_deb, deb_labels))
                LOGIT_MAIN += lp_p_deb * w_main_deb
                comp_lora_posteriors.append((w_comp_deb, softmax(lp_p_deb, axis=1)))
                if N_unseen > 0:
                    lp_u_deb = log_softmax(align_cols(Z_unseen_deb @ W_p_deb.T + b_p_deb, deb_labels))

            if W_c_deb is not None:
                lp_c85_deb, _ = compute_centered_logits(Z_te_deb, groups_085, W_c_deb, b_c_deb, deb_labels)
                lp_c72, _ = compute_centered_logits(Z_te_deb, groups_072, W_c_deb, b_c_deb, deb_labels)
                lp_c60, _ = compute_centered_logits(Z_te_deb, groups_060, W_c_deb, b_c_deb, deb_labels)
                lp_c_blend = w85 * lp_c85_deb + w72 * lp_c72 + w60 * lp_c60
                comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))
                if N_unseen > 0:
                    lp_u_c_deb, _ = compute_centered_logits(Z_unseen_deb, groups_unseen, W_c_deb, b_c_deb, deb_labels)

            if W_p_deb is not None and lp_p_deb is not None:
                w_tot_deb = w_main_deb + (w_comp_c if W_c_deb is not None else 0.0)
                if w_tot_deb > 0.0 or not lock_priors:
                    lp_m_u = (
                        _combine_plain_and_twin_lp(lp_u_deb, lp_u_c_deb if W_c_deb is not None else None, w_main_deb, w_comp_c)
                        if N_unseen > 0
                        else np.zeros((0, K), dtype=np.float32)
                    )
                    lp_m_te = _combine_plain_and_twin_lp(lp_p_deb, lp_c_blend if W_c_deb is not None else None, w_main_deb, w_comp_c)
                    aud_name = deb_key
                    auditor.add_model(aud_name, tier="Main", weight=w_tot_deb, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te)

            if need_sub_train and len(Z_tr_deb) > 0:
                res_deb = fit_shared_gaussian_lowram(Z_tr_deb, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get(deb_key, 0.10))
                U_te = transform_lowram(Z_te_deb, res_deb)
                lp_dp = scores_exact_dp(U_te, groups_065, res_deb, D_hybrid)
                dp_posteriors.append((dp_weights.get("deberta", 0.20), softmax(lp_dp, axis=1)))

                S_tr, S_te = make_spectral_blocks(Z_tr_deb, Z_te_deb, res_deb, prior_scales.get(deb_key, 0.80))
                spectral_blocks_tr.append(S_tr)
                spectral_blocks_te.append(S_te)
                if N_unseen > 0 and "Z_unseen_deb" in locals() and Z_unseen_deb is not None:
                    U_u = transform_lowram(Z_unseen_deb, res_deb)
                    lp_dp_u = scores_exact_dp(U_u, groups_unseen, res_deb, None)
                    dp_posteriors_unseen.append((dp_weights.get("deberta", 0.20), softmax(lp_dp_u, axis=1)))
                    _, S_u = make_spectral_blocks(Z_tr_deb, Z_unseen_deb, res_deb, prior_scales.get(deb_key, 0.80))
                    spectral_blocks_unseen.append(S_u)
                    del U_u, S_u
                del res_deb, U_te

            del Z_tr_deb, Z_te_deb, Z_test_all
            if "Z_unseen_deb" in locals():
                del Z_unseen_deb
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Add Secondary LoRAs to COMP_LIST
        if comp_lora_posteriors:
            tot_w = sum(w for w, _ in comp_lora_posteriors)
            P_lora_comb = sum(w * P for w, P in comp_lora_posteriors) / max(1e-8, tot_w)
            COMP_LIST.append((comp_streams.get("lora_and_twins", 0.35), P_lora_comb))

        # Add DP Posteriors to COMP_LIST
        if dp_posteriors:
            tot_dp_w = sum(w for w, _ in dp_posteriors)
            P_dp_comb = sum(w * P_dp for w, P_dp in dp_posteriors) / max(1e-8, tot_dp_w)
            COMP_LIST.append((comp_streams.get("dp_posteriors", 0.30), P_dp_comb))

        # Fit Joint Spectral Bank
        if spectral_blocks_tr and spectral_blocks_te:
            sb_cfg = comp_cfg.get("spectral_bank", {})
            plain_C = sb_cfg.get("plain_C", 0.04)
            center_C = sb_cfg.get("center_C", 0.03)
            center_blend = sb_cfg.get("center_blend", 0.35)

            P_spectral, oof_score = fit_joint_spectral_bank(
                spectral_tr_blocks=spectral_blocks_tr,
                spectral_te_blocks=spectral_blocks_te,
                y_sub=y_sub,
                prompts_sub=prompts_sub,
                D_tfidf=D_tfidf,
                D_neural_sum=D_neural_sum,
                D_neural_weight_total=D_neural_weight_total,
                N_test=N_test,
                plain_C=plain_C,
                center_C=center_C,
                center_blend=center_blend,
                max_cluster_size=max_cl,
            )
            print(f"[Pipeline] Joint Spectral Bank fitted (OOF BAcc = {oof_score:.4f})")
            COMP_LIST.append((comp_streams.get("spectral_bank", 0.15), P_spectral))
            if len(spectral_blocks_unseen) > 0 and N_unseen > 0:
                P_spectral_unseen, _ = fit_joint_spectral_bank(
                    spectral_blocks_tr,
                    spectral_blocks_unseen,
                    y_sub,
                    prompts_sub,
                    D_tfidf=None,
                    D_neural_sum=None,
                    D_neural_weight_total=0.0,
                    N_test=N_unseen,
                    plain_C=plain_C,
                    center_C=center_C,
                    center_blend=center_blend,
                    max_cluster_size=max_cl,
                    groups_override=groups_unseen,
                    compute_oof=False,
                )

    elif is_smoke:
        print("[Pipeline] Smoke mode active: synthesizing high-fidelity mock representations...")
        np.random.seed(42)
        mock_p_lora = softmax(np.random.randn(N_test, K), axis=1)
        mock_p_dp = softmax(np.random.randn(N_test, K), axis=1)
        mock_p_spec = softmax(np.random.randn(N_test, K), axis=1)
        COMP_LIST.append((comp_streams.get("lora_and_twins", 0.35), mock_p_lora))
        COMP_LIST.append((comp_streams.get("dp_posteriors", 0.30), mock_p_dp))
        COMP_LIST.append((comp_streams.get("spectral_bank", 0.15), mock_p_spec))
        dp_posteriors.append((comp_streams.get("dp_posteriors", 0.30), mock_p_dp))
        P_spectral = mock_p_spec

    # 4. MODULE 4: Flat Per-Model Log-Space Ensemble & Dynamic Unseen Calibration
    print("[Pipeline] Module 4: Flat Per-Model Log-Space Ensemble & Dynamic Unseen Calibration...")
    extra_streams: list[tuple[str, float, np.ndarray]] = []
    extra_streams_unseen: list[tuple[str, float, np.ndarray]] = []
    w_dp = model_weights.get("wpca_dp", {}).get("plain_weight", 0.35)
    if dp_posteriors:
        tot_dp_w = sum(w for w, _ in dp_posteriors)
        P_dp_comb = sum(w * P_dp for w, P_dp in dp_posteriors) / max(1e-8, tot_dp_w)
        extra_streams.append(("wpca_dp", w_dp, np.log(np.clip(P_dp_comb, 1e-9, 1.0))))
    elif is_smoke:
        extra_streams.append(("wpca_dp", w_dp, np.log(np.clip(mock_p_dp, 1e-9, 1.0))))

    if dp_posteriors_unseen:
        tot_dp_u_w = sum(w for w, _ in dp_posteriors_unseen)
        P_dp_u_comb = sum(w * P_dp for w, P_dp in dp_posteriors_unseen) / max(1e-8, tot_dp_u_w)
        extra_streams_unseen.append(("wpca_dp", w_dp, np.log(np.clip(P_dp_u_comb, 1e-9, 1.0))))

    w_sb = model_weights.get("spectral_bank", {}).get("plain_weight", 0.20)
    if "P_spectral" in locals() and P_spectral is not None:
        extra_streams.append(("spectral_bank", w_sb, np.log(np.clip(P_spectral, 1e-9, 1.0))))
    elif is_smoke:
        extra_streams.append(("spectral_bank", w_sb, np.log(np.clip(mock_p_spec, 1e-9, 1.0))))

    if "P_spectral_unseen" in locals() and P_spectral_unseen is not None:
        extra_streams_unseen.append(("spectral_bank", w_sb, np.log(np.clip(P_spectral_unseen, 1e-9, 1.0))))

    if len(auditor.models) > 0:
        optimize_on_unseen = N_unseen >= 36 and not is_smoke
        lock_priors = bool(cfg.get("lock_prior_weights", True))
        P_mix, active_weights_map, df_ablation = auditor.optimize_weights_and_blend(
            working_dir=working_dir,
            extra_test_log_probs=extra_streams,
            optimize_on_unseen=optimize_on_unseen,
            lock_prior_weights=lock_priors,
        )
        print(f"[Pipeline] Flat log-space ensemble active weights: {active_weights_map}")
        main_active_lp = np.zeros((N_test, K), dtype=np.float64)
        main_weight_sum = 0.0
        for m in auditor.models:
            if m.get("tier") == "Main":
                w = float(active_weights_map.get(m["name"], m["weight"]))
                if w > 0.0 and len(m.get("test_lp", [])) == N_test:
                    main_active_lp += w * m["test_lp"]
                    main_weight_sum += w
        if main_weight_sum > 0.0:
            R_main = softmax(main_active_lp, axis=1)
        else:
            R_main = P_mix
    else:
        main_ratio = fusion_cfg.get("main_ratio", 0.85)
        comp_ratio = fusion_cfg.get("comp_ratio", 0.15)
        P_mix, R_main = blend_main_and_complementary(LOGIT_MAIN, COMP_LIST, main_ratio=main_ratio, comp_ratio=comp_ratio)

    # 4-Family Within-Family Spectral Gates (Vectorized)
    family_gates = {}
    family_gates_unseen = {}
    if post_cfg.get("enable_within_family_confidence_gates", False) and float(post_cfg.get("family_gate_strength", 0.0)) > 0.0:
        if spectral_blocks_tr and spectral_blocks_te:
            family_gates = fit_within_family_spectral_gates(
                spectral_tr_blocks=spectral_blocks_tr,
                spectral_te_blocks=spectral_blocks_te,
                y_sub=y_sub,
                prompts_sub=prompts_sub,
                D_tfidf=D_tfidf,
                D_neural_sum=D_neural_sum,
                D_neural_weight_total=D_neural_weight_total,
                N_test=N_test,
                max_cluster_size=max_cl,
            )
            P_mix = apply_all_family_confidence_gates(
                P_mix=P_mix,
                family_gates=family_gates,
                max_strength=float(post_cfg.get("family_gate_strength", 0.20)),
                margin_thr=float(post_cfg.get("family_gate_margin_thr", 0.45)),
            )
            print("[Pipeline] Applied within-family confidence gates.")

    # -------------------------------------------------------------------------
    # STAGE 3A: Claude Specialist Gate (Vectorized Head)
    # -------------------------------------------------------------------------
    gate_weight = post_cfg.get("claude_specialist_weight", 0.15)
    P_mix = apply_claude_specialist_gate(
        P_mix=P_mix,
        R_main=R_main,
        claude_spec_logits=claude_spec_logits,
        claude_classes=claude_classes,
        labels=LABELS,
        claude_cols=FAMILIES["claude"],
        gate_weight=gate_weight,
    )

    # -------------------------------------------------------------------------
    # STAGE 3B: Deep Neural Family-Specialist LoRA Gates (GPT, Gemini, GLM)
    # -------------------------------------------------------------------------
    for fam_k, default_w in [("gpt", 0.55), ("gemini", 0.65), ("glm", 0.70)]:
        spec_te_logits = family_spec_logits_te.get(fam_k)
        if spec_te_logits is not None:
            spec_cls = family_spec_classes.get(fam_k)
            w_gate = float(post_cfg.get(f"{fam_k}_specialist_weight", default_w))
            P_mix = apply_family_specialist_lora_gate(
                P_mix=P_mix,
                R_main=R_main,
                spec_logits=spec_te_logits,
                spec_classes=spec_cls,
                labels=LABELS,
                family_cols=FAMILIES[fam_k],
                gate_weight=w_gate,
            )

    # -------------------------------------------------------------------------
    # STAGE 4: Unified Handwriting & Orthographic Shield
    # -------------------------------------------------------------------------
    default_curly_multiplier = float(np.exp(-1.0))
    multiplier = post_cfg.get("curly_quote_multiplier", default_curly_multiplier)
    multi_em_dash_log_pen = post_cfg.get("multi_em_dash_log_penalty", 0.0)
    straight_contr_log_pen = post_cfg.get("straight_contr_log_penalty", 0.0)
    curly_contr_log_pen = post_cfg.get("curly_contr_log_penalty", 0.0)
    continuous_hw_weight = post_cfg.get("continuous_hw_weight", 0.0)

    P_mix = apply_unified_handwriting_shield(
        P_mix=P_mix,
        test_texts=test_texts,
        log_p_handwriting=log_p_handwriting,
        rule_flags=rule_flags_test,
        curly_quote_multiplier=multiplier,
        multi_em_dash_log_penalty=multi_em_dash_log_pen,
        straight_contr_log_penalty=straight_contr_log_pen,
        curly_contr_log_penalty=curly_contr_log_pen,
        continuous_hw_weight=continuous_hw_weight,
    )

    # -------------------------------------------------------------------------
    # STAGE 5: Cavity Cluster Uniqueness Repulsion
    # -------------------------------------------------------------------------
    P_mix = apply_cluster_uniqueness_repulsion(
        P_mix=P_mix, groups=groups_065, strength=post_cfg.get("cluster_repulsion_strength", 0.35), iters=3
    )

    # -------------------------------------------------------------------------
    # STAGE 6: Multiplicative Sinkhorn Column Mass Balancing
    # -------------------------------------------------------------------------
    sinkhorn_iters = post_cfg.get("sinkhorn_iterations", 100)
    sinkhorn_strength = float(post_cfg.get("sinkhorn_strength", 0.65))
    sinkhorn_target_mix = float(post_cfg.get("sinkhorn_target_mix", 0.35))
    train_prior = np.bincount(y_sub, minlength=18).astype(np.float64) + 1.0 if len(y_sub) > 0 else None
    P_final = sinkhorn_balancing(
        P_mix, iters=sinkhorn_iters, sinkhorn_strength=sinkhorn_strength, target_mix=sinkhorn_target_mix, train_prior=train_prior
    )

    # -------------------------------------------------------------------------
    # STAGE 7: TypoBayes 37-Marker Hierarchical Beta-Bernoulli Postprocessing
    # -------------------------------------------------------------------------
    if post_cfg.get("enable_typobayes", True):
        P_final = apply_typobayes_postprocess(
            P_final,
            train_texts=df_train["text"].fillna("").astype(str).tolist(),
            y_train=np.array([LAB2I[m] for m in df_train["model"].values], dtype=int),
            test_texts=test_texts,
            labels=LABELS,
            kappa=float(post_cfg.get("typobayes_kappa", 24.0)),
            evidence_scale=float(post_cfg.get("typobayes_evidence_scale", 1.2)),
            evidence_cap=float(post_cfg.get("typobayes_evidence_cap", 0.30)),
        )

    # Auditing and forward stepwise ablation report
    test_final_preds = P_final.argmax(axis=1) if len(P_final) > 0 else np.array([])
    auditor.print_and_save_report(working_dir, final_test_preds=test_final_preds)

    # Generate Submission Artifacts
    df_sub, out_parquet, out_csv = generate_submission(P_final, test_ids, working_dir)
    elapsed = time.time() - start_time
    print(f"[Pipeline] Submission generated successfully in {elapsed:.2f}s: {out_parquet} ({len(df_sub)} rows)")

    # Export compact local evaluation cache for sub-second offline local tuning
    try:
        if len(groups_unseen) > 0:
            try:
                g_u = np.asarray(groups_unseen, dtype=np.int32)
            except (ValueError, TypeError):
                g_u = (pd.factorize(groups_unseen)[0] + 1).astype(np.int32)
        else:
            g_u = np.array([], dtype=np.int32)

        cache_dict = {
            "N_test": np.array([N_test], dtype=np.int32),
            "N_unseen": np.array([N_unseen], dtype=np.int32),
            "y_unseen": y_unseen.astype(np.int32) if len(y_unseen) > 0 else np.array([], dtype=np.int32),
            "groups_unseen": g_u,
            "D_tfidf": D_tfidf.astype(np.float32)
            if ("D_tfidf" in locals() and D_tfidf is not None)
            else np.zeros((0, 0), dtype=np.float32),
            "D_hybrid": D_hybrid.astype(np.float32)
            if ("D_hybrid" in locals() and D_hybrid is not None)
            else np.zeros((0, 0), dtype=np.float32),
            "groups_085": groups_085.astype(np.int32)
            if ("groups_085" in locals() and groups_085 is not None)
            else np.zeros(0, dtype=np.int32),
            "groups_065": groups_065.astype(np.int32)
            if ("groups_065" in locals() and groups_065 is not None)
            else np.zeros(0, dtype=np.int32),
            "model_names": np.array([m["name"] for m in auditor.models], dtype=object),
            "model_tiers": np.array([m["tier"] for m in auditor.models], dtype=object),
            "model_prior_weights": np.array([m.get("prior_weight", m["weight"]) for m in auditor.models], dtype=np.float64),
            "model_learned_weights": np.array([m["weight"] for m in auditor.models], dtype=np.float64),
            "P_final": P_final.astype(np.float32),
            "test_texts": np.array(test_texts, dtype=object),
        }
        for idx, m in enumerate(auditor.models):
            cache_dict[f"m_{idx}_unseen_lp"] = m["unseen_lp"].astype(np.float32)
            cache_dict[f"m_{idx}_test_lp"] = m["test_lp"].astype(np.float32)
        if "rule_flags_test" in locals() and rule_flags_test is not None:
            cache_dict["rule_flags_test"] = rule_flags_test.astype(np.float32)
        if "log_p_handwriting" in locals() and log_p_handwriting is not None:
            cache_dict["log_p_handwriting"] = log_p_handwriting.astype(np.float32)
        if "claude_spec_logits" in locals() and claude_spec_logits is not None:
            cache_dict["claude_spec_logits"] = claude_spec_logits.astype(np.float32)
        if "claude_classes" in locals() and claude_classes is not None:
            cache_dict["claude_classes"] = np.array(claude_classes, dtype=object)
        if "family_gates" in locals() and family_gates:
            for fam_k, fam_lp in family_gates.items():
                cache_dict[f"gate_{fam_k}_lp"] = fam_lp.astype(np.float32)
        for ex_idx, (ex_name, ex_w, ex_lp) in enumerate(extra_streams):
            cache_dict[f"extra_{ex_idx}_name"] = np.array([ex_name], dtype=object)
            cache_dict[f"extra_{ex_idx}_weight"] = np.array([ex_w], dtype=np.float64)
            cache_dict[f"extra_{ex_idx}_lp"] = ex_lp.astype(np.float32)
            if ex_idx < len(extra_streams_unseen):
                cache_dict[f"extra_{ex_idx}_unseen_lp"] = extra_streams_unseen[ex_idx][2].astype(np.float32)
        cache_dict["num_extra"] = np.array([len(extra_streams)], dtype=np.int32)
        if "claude_spec_logits_unseen" in locals() and claude_spec_logits_unseen is not None:
            cache_dict["claude_spec_logits_unseen"] = claude_spec_logits_unseen.astype(np.float32)
        for fam_k in ("gpt", "gemini", "glm"):
            if "family_spec_logits_te" in locals() and fam_k in family_spec_logits_te and family_spec_logits_te[fam_k] is not None:
                cache_dict[f"{fam_k}_spec_logits"] = family_spec_logits_te[fam_k].astype(np.float32)
            if (
                "family_spec_logits_unseen" in locals()
                and fam_k in family_spec_logits_unseen
                and family_spec_logits_unseen[fam_k] is not None
            ):
                cache_dict[f"{fam_k}_spec_logits_unseen"] = family_spec_logits_unseen[fam_k].astype(np.float32)
            if "family_spec_classes" in locals() and fam_k in family_spec_classes and family_spec_classes[fam_k] is not None:
                cache_dict[f"{fam_k}_classes"] = np.array(family_spec_classes[fam_k], dtype=object)
        if "family_gates_unseen" in locals() and family_gates_unseen:
            for fam_k, fam_lp in family_gates_unseen.items():
                cache_dict[f"gate_{fam_k}_unseen_lp"] = fam_lp.astype(np.float32)
        if "unseen_texts" in locals() and unseen_texts:
            cache_dict["unseen_texts"] = np.array(unseen_texts, dtype=object)
        if "train_sub_texts" in locals() and train_sub_texts:
            cache_dict["train_texts"] = np.array(train_sub_texts, dtype=object)
        if "y_sub" in locals() and y_sub is not None:
            cache_dict["y_train"] = np.asarray(y_sub, dtype=np.int32)

        cache_file_path = working_dir / "local_eval_cache.npz"
        np.savez_compressed(cache_file_path, **cache_dict)
        print(f"[Pipeline] Exported compact local evaluation cache: {cache_file_path} ({cache_file_path.stat().st_size} bytes)")
    except Exception as e:
        print(f"[Pipeline] Note: Could not write local_eval_cache.npz: {e}")

    return df_sub, out_parquet


In [ ]:
if __name__ == '__main__':
    run_submission_pipeline(ENSEMBLE_CONFIG)
